In [1]:
import os
import json
import re
import sqlite3
from pathlib import Path
from collections import OrderedDict
import xml.etree.ElementTree as ET
try:
    import lxml.etree as LET
    _LXML = True
except ImportError:
    _LXML = False

# ── Target Configuration ───────────────────────────────────────
WORKSPACE_DIR = Path("/Users/gcrane/github/persverscomp") 
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)

# Build-only monolith DB. It lives OUTSIDE the repo (in a temp dir) so it can
# never be staged or pushed to GitHub. It is a regenerable build SOURCE only:
# Cell 2 shards it into site/data/** and Cell 3 reads it to build index.html.
# Delete it any time with the cleanup cell — the deployed site/ never needs it.
BUILD_DIR = Path("/tmp/persvers_build")          # any non-repo dir works
BUILD_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = BUILD_DIR / "corpus_alignment_grid.db"
NS = {'tei': 'http://www.tei-c.org/ns/1.0'}

WORK_REGISTRY = {
    "tlg0003.tlg001": {
        "textgroup": "tlg0003",
        "work": "tlg001",
        "editions": {
            "perseus-grc2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.perseus-grc2.xml", "label": "Greek (H. S. Jones, 1942)", "class": "greek-text"}
        },
        "appcrits": {},
        "translations": {
            "1st1K-eng1": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.1st1K-eng1.xml", "label": "English (C. F. Smith, 1919)", "class": "english-text"},
            "1st1K-eng2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.1st1K-eng2.xml", "label": "English (Henry Dale, 1851)", "class": "english-text"},
            "perseus-eng4": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.perseus-eng4.xml", "label": "English (Thomas Hobbes, 1843)", "class": "english-text"},
            "perseus-eng6": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.perseus-eng6.xml", "label": "English (R. Crawley, 1914)", "class": "english-text"},
            "1st1k-fre1": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.1st1k-fre1.xml", "label": "French (E. Bétant, 1863)", "class": "french-text"},
            "1st1K-fre2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.1st1K-fre2.xml", "label": "French (M. C. Zévort, 1852)", "class": "french-text"},
            "1st1K-ger1": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.1st1K-ger1.xml", "label": "German (H. Gürsching, 1856)", "class": "german-text"},
            "1st1K-ger2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.1st1K-ger2.xml", "label": "German (A. Wahrmund, 1864)", "class": "german-text"},
            "1st1K-ger3": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.1st1K-ger3.xml", "label": "German (Theodor Braun, 1917)", "class": "german-text"},
            "1st1K-ger4": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.1st1K-ger4.xml", "label": "German (Rudolf G. Binding, 1937)", "class": "german-text"},
            "1st1K-ita1": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.1st1K-ita1.xml", "label": "Italian (Anonymous, 1835)", "class": "italian-text"},
            "1st1k-lat2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.1st1k-lat2.xml", "label": "Latin (Fr. Haase, 1869)", "class": "latin-text"}
        },
        "commentaries": {},
        "treebanks": {
            "glaux-grc1": {
                "path": "/Users/gcrane/github/GRC_misc/data/tlg0003/tlg001/tlg0003.tlg001.perseus-grc2.glossed.xml",
                "label": "Treebank (Glaux)",
                "class": "treebank-text",
                "parse_mode": "agdt_xml",
                "source_repo": "Glaux"
            }
        }
    },
    "tlg0012.tlg001": {
        "textgroup": "tlg0012",
        "work": "tlg001",
        "editions": {
            "perseus-grc2": {
                        "path": "/Users/gcrane/github/canonical-greekLit/data/tlg0012/tlg001/tlg0012.tlg001.perseus-grc2.xml", \
                        "label": "Greek (David B Monro, 1908)", \
                        "class": "greek-text",\
                        "parse_mode": "poetry_cards"\
            }        
        },
        "appcrits": {},
        "translations": {
            "perseus-eng3": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0012/tlg001/tlg0012.tlg001.perseus-eng3.xml", "label": "English (A.T. Murray, 1924)", "class": "english-text", "parse_mode": "poetry_cards"},
            "butlernagy2020-eng2": {
                "path": "/Users/gcrane/github/AntigonesPublic/data/tlg0012/tlg001/tlg0012.tlg001.butlernagy2020-eng2.xml",
                "label": "English (Butler, rev. Nagy et al., 2020)",
                "class": "english-text",
                "parse_mode": "card_prose"
            },
            "bryant1870-eng2": {
                "path": "/Users/gcrane/github/Homerica/data/tlg0012/tlg001/tlg0012.tlg001.bryant1870-eng2.xml",
                "label": "English (William Cullen Bryant, 1870)",
                "class": "english-text",
                "parse_mode": "poetry_cards"
            },
            "pope1720watson1857-eng2": {
                "path": "/Users/gcrane/github/AntigonesPublic/data/tlg0012/tlg001/tlg0012.tlg001.popewatson1857-eng2.xml",
                "label": "English (Alexander Pope, 1720, ed. Watson, 1857)",
                "class": "english-text",
                "parse_mode": "poetry_cards"
            },
            "hobbes1667molesworth-eng2": {
                "path": "/Users/gcrane/github/AntigonesPublic/data/tlg0012/tlg001/tlg0012.tlg001.hobbes1667molesworth-eng2.xml",
                "label": "English (Thomas Hobbes, 1667, ed. Molesworth 1843)",
                "class": "english-text",
                "parse_mode": "poetry_cards"
            },
            "chapman1611hooper-eng2": {
                "path": "/Users/gcrane/github/AntigonesPublic/data/tlg0012/tlg001/tlg0012.tlg001.chapman1611hooper-eng2.xml",
                "label": "English (George Chapman, 1611, ed. Hooper 1857)",
                "class": "english-text",
                "parse_mode": "poetry_cards"
            },
            "cowper-eng2": {
                "path": "/Users/gcrane/github/AntigonesPublic/data/tlg0012/tlg001/tlg0012.tlg001.cowper-eng2.xml",
                "label": "English (William Cowper, 1791, ed. Southey)",
                "class": "english-text",
                "parse_mode": "poetry_cards"
            },
            "newman1856-eng2": {
                "path": "/Users/gcrane/github/AntigonesPublic/data/tlg0012/tlg001/tlg0012.tlg001.newman1856-eng2.xml",
                "label": "English (F. W. Newman, 1856)",
                "class": "english-text",
                "parse_mode": "poetry_cards"
            },
        },
        "commentaries": {},
        "treebanks": {
            "daphne_perstb-grc1": {
                "path": [
                    "/Users/gcrane/github/Daphne/data/annotation/latest/tlg0012/tlg001/grc_new/tlg0012_tlg001_daphne_perstbgllt-grc1_1-6.conllu",
                    "/Users/gcrane/github/Daphne/data/annotation/latest/tlg0012/tlg001/grc_new/tlg0012.tlg001.daphne_perstbgllt-grc1.7-12.conllu",
                    "/Users/gcrane/github/Daphne/data/annotation/latest/tlg0012/tlg001/grc_new/tlg0012.tlg001.daphne_perstbgllt-grc1.13-18.conllu",
                    "/Users/gcrane/github/Daphne/data/annotation/latest/tlg0012/tlg001/grc_new/tlg0012.tlg001.daphne_perstbgllt-grc1.19-24.conllu"
                ],
                "label": "Treebank (Perseus AGDT)",
                "class": "treebank-text",
                "parse_mode": "conllu",
                "source_repo": "gregorycrane/gAGDT"
            }
        },
        "metrics": {
            "perstb-meter-grc1": {
                "path": "/Users/gcrane/github/Homerica/data/tlg0012/tlg001/tlg0012.tlg001.meter.tsv",
                "label": "Metrical Analysis (Perseus)",
                "class": "metrical-text"
            }
        }
    },
       "tlg0012.tlg002": {
        "textgroup": "tlg0012",
        "work": "tlg002",
        "editions": {
            "perseus-grc2": {
                        "path": "/Users/gcrane/github/canonical-greekLit/data/tlg0012/tlg002/tlg0012.tlg002.perseus-grc2.xml", \
                        "label": "Greek (David B Monro, 1908)", \
                        "class": "greek-text",\
                        "parse_mode": "poetry_cards"\
            }        
        },
        "appcrits": {},
        "translations": {
            "perseus-eng3": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0012/tlg002/tlg0012.tlg002.perseus-eng3.xml", "label": "English (A.T. Murray, 1924)", "class": "english-text", "parse_mode": "poetry_cards"},
            "butlernagy2020-eng2": {
                "path": "/Users/gcrane/github/AntigonesPublic/data/tlg0012/tlg002/tlg0012.tlg002.butlernagy2020-eng2.xml",
                "label": "English (Butler, rev. Nagy et al., 2020)",
                "class": "english-text",
                "parse_mode": "card_prose"
            },       
            "bryant1873-eng2": {
                "path": "/Users/gcrane/github/Homerica/data/tlg0012/tlg002/tlg0012.tlg002.bryant1873-eng2.xml",
                "label": "English (William Cullen Bryant, 1873)",
                "class": "english-text",
                "parse_mode": "poetry_cards"
            },
            "pope1726watson1858-eng2": {
                "path": "/Users/gcrane/github/AntigonesPublic/data/tlg0012/tlg002/tlg0012.tlg002.popewatson1858-eng2.xml",
                "label": "English (Alexander Pope, 1726, ed. Watson, 1858)",
                "class": "english-text",
                "parse_mode": "poetry_cards"
            },
            "hobbes1667molesworth-eng2": {
                "path": "/Users/gcrane/github/AntigonesPublic/data/tlg0012/tlg002/tlg0012.tlg002.hobbes1667molesworth-eng2.xml",
                "label": "English (Thomas Hobbes, 1667, ed. Molesworth 1843)",
                "class": "english-text",
                "parse_mode": "poetry_cards"
            },
            "chapman1615hooper-eng2": {
                "path": "/Users/gcrane/github/AntigonesPublic/data/tlg0012/tlg002/tlg0012.tlg002.chapman1615hooper-eng2.xml",
                "label": "English (George Chapman, 1667, ed. Hooper 1857)",
                "class": "english-text",
                "parse_mode": "poetry_cards"
            },
            "cowper-eng2": {
                "path": "/Users/gcrane/github/AntigonesPublic/data/tlg0012/tlg002/tlg0012.tlg002.cowper-eng2.xml",
                "label": "English (William Cowper, 1791, ed. Southey)",
                "class": "english-text",
                "parse_mode": "poetry_cards"
            },
            "cotterill1911-eng2": {
                "path": "/Users/gcrane/github/AntigonesPublic/data/tlg0012/tlg002/tlg0012.tlg002.cotterill1911-eng2.xml",
                "label": "English (H. B. Cotterill, 1911)",
                "class": "english-text",
                "parse_mode": "poetry_cards"
            },
        },
        "commentaries": {},
        "treebanks": {
            "daphne_perstb-grc1": {
                "path": [
                    "/Users/gcrane/github/Daphne/data/annotation/latest/tlg0012/tlg002/grc_new/tlg0012.tlg002.daphne_tbglosslt-grc1.1-6.conllu",
                    "/Users/gcrane/github/Daphne/data/annotation/latest/tlg0012/tlg002/grc_new/tlg0012.tlg002.daphne_tbglosslt-grc1.7-12.conllu",
                    "/Users/gcrane/github/Daphne/data/annotation/latest/tlg0012/tlg002/grc_new/tlg0012.tlg002.daphne_tbglosslt-grc1.13-18.conllu",
                    "/Users/gcrane/github/Daphne/data/annotation/latest/tlg0012/tlg002/grc_new/tlg0012.tlg002.daphne_tbglosslt-grc1.19-24.conllu"
                ],
                "label": "Treebank (Perseus AGDT)",
                "class": "treebank-text",
                "parse_mode": "conllu",
                "source_repo": "gregorycrane/gAGDT"
            }
        },
        "metrics": {
            "perstb-meter-grc1": {
                "path": "/Users/gcrane/github/Homerica/data/tlg0012/tlg002/tlg0012.tlg002.meter.tsv",
                "label": "Metrical Analysis (Perseus)",
                "class": "metrical-text"
            }
        }
    },  "tlg0086.tlg034": {
        "textgroup": "tlg0086",
        "work": "tlg034",
        "editions": {
            "perseus-grc2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0086/tlg034/tlg0086.tlg034.perseus-grc2.xml", "label": "Greek (Kassel, 1965)", "class": "greek-text"},
            "digicorpus-grc2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0086/tlg034/tlg0086.tlg034.digicorpus-grc2.xml", "label": "Greek (Digital Corpus Variant)", "class": "greek-text"},
            "bywater1909-grc1": {"path": "/Users/gcrane/github/Poetics2.0/data/tlg0086/tlg034/tlg0086.tlg034.bywater1909-grc1.xml", "label": "Greek (Ingram Bywater, 1909)", "class": "greek-text"}
        },
        "appcrits": {},
        "translations": {
            "bishrmatta-ara1": {"path": "/Users/gcrane/github/Poetics2.0/data/tlg0086/tlg034/tlg0086.tlg034.tkatsch-ara1.xml", "label": "Arabic (Abū Bishr Mattā, c. 900)", "class": "arabic-text"},
            "moerbeke1277-lat1": {"path": "/Users/gcrane/github/Poetics2.0/data/tlg0086/tlg034/tlg0086.tlg034.moerbeke1277-lat1.xml", "label": "Latin (Moerbeke, 1277)", "class": "latin-text"},
            "perseus-eng2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0086/tlg034/tlg0086.tlg034.perseus-eng2.xml", "label": "English (W.H. Fyfe, 1927)", "class": "english-text"},
            "butcher1911-eng2": {"path": "/Users/gcrane/github/Poetics2.0/data/tlg0086/tlg034/tlg0086.tlg034.butcher1911-eng2.xml", "label": "English (S.H. Butcher, 1911)", "class": "english-text"},
            "riccobono1587-lat1": {"path": "/Users/gcrane/github/Poetics2.0/data/tlg0086/tlg034/tlg0086.tlg034.riccobono1587-lat1.xml", "label": "Latin (A. Riccobono, 1587)", "class": "latin-text"},
            "bywater1909-eng1": {"path": "/Users/gcrane/github/Poetics2.0/data/tlg0086/tlg034/tlg0086.tlg034.bywater1909-eng1.xml", "label": "English (Ingram Bywater, 1909)", "class": "english-text"},
            "twining1789-eng1": {"path": "/Users/gcrane/github/Poetics2.0/data/tlg0086/tlg034/tlg0086.tlg034.twining1789-eng1.xml", "label": "English (Thomas Twining, 1789)", "class": "english-text", "parse_mode": "milestones"},
            "dacier1692-fra2": {"path": "/Users/gcrane/github/Poetics2.0/data/tlg0086/tlg034/tlg0086.tlg034.dacier1692-fra2.xml", "label": "French (André Dacier, 1692)", "class": "french-text", "parse_mode": "milestones"},
            "segni-ita1-aligned": {
    "path": "/Users/gcrane/github/Poetics2.0/data/tlg0086/tlg034/tlg0086.tlg034.segni-ita1-aligned.xml",
    "label": "Italian (Bernardo Segni, 1549)",
    "class": "italian-text"
},
        },
        "commentaries": {
            "bywater1909-com1": {
                "path": "/Users/gcrane/github/Poetics2.0/data/tlg0086/tlg034/tlg0086.tlg034.bywater1909-commentary-eng1.xml", \
                "label": "Commentary (Bywater, 1909)", \
                "class": "commentary-text"\
            },
            "dacier1692-com-fra1": {
                "path": "/Users/gcrane/github/Poetics2.0/data/tlg0086/tlg034/tlg0086.tlg034.dacier1692-commentary-fra2.xml",
                "label": "Commentary (André Dacier, 1692)",
                "class": "commentary-text",
                "parse_mode": "milestones",
            },
             "twining-com-eng1": {
                "path": "/Users/gcrane/github/Poetics2.0/data/tlg0086/tlg034/tlg0086.tlg034.twining1789-commentary-eng2.xml",
                "label": "Commentary (T. Twining, 1789)",
                "class": "commentary-text",
                "parse_mode": "milestones",
            },
       },
        "treebanks": {
            "kassel-tb-grc1": {
                "path": "/Users/gcrane/github/Poetics2.0/data/tlg0086/tlg034/tlg0086.tlg034.kassel.tb.conllu",
                "label": "Treebank (Kassel, UD 2026)",
                "class": "treebank-text",
                "parse_mode": "conllu"
            },
            "bishrmatta-tb-ara1": {
                "path": "/Users/gcrane/github/Poetics2.0/data/tlg0086/tlg034/tlg0086.tlg034.matta-ara.conllu",
                "label": "Treebank (Bishr Mattā Arabic, UD)",
                "class": "treebank-text",
                "parse_mode": "conllu"
            }
        },
        "alignments": {
            "bywater1909-grc1__bywater1909-eng1": {
                "path": "/Users/gcrane/github/Poetics2.0/data/tlg0086/tlg034/tlg0086.tlg034.bywater1909-grc1__bywater1909-eng1.json",
                "label": "Bywater GRC → Bywater ENG (1909)",
                "src_version": "bywater1909-grc1",
                "tgt_version": "bywater1909-eng1"
            },
            "bywater1909-grc1__butcher1911-eng2": {
                "path": "/Users/gcrane/github/Poetics2.0/data/tlg0086/tlg034/tlg0086.tlg034.bywater1909-grc1__butcher1911-eng2.json",
                "label": "Bywater GRC → Butcher ENG (1911)",
                "src_version": "bywater1909-grc1",
                "tgt_version": "butcher1911-eng2"
            }
        }
    },
    
    "tlg0011.tlg001": {
        "textgroup": "tlg0011",

        "work": "tlg001",
        "editions": {
            "perseus-grc2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0011/tlg001/tlg0011.tlg001.perseus-grc2.xml", "label": "Greek (F. Storr, 1912)", "class": "greek-text", "parse_mode": "poetry_cards"}
        },
        "appcrits": {},
        "translations": {
            "perseus-eng3": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0011/tlg001/tlg0011.tlg001.perseus-eng3.xml", "label": "English (Sir Richard Jebb, 1904)", "class": "english-text", "parse_mode": "poetry_cards"}
        },
        "commentaries": {
            "jebb-com-eng1": {
                "path": "/Users/gcrane/github/canonical_pdlrefwk/data/viaf2603144/viaf007/viaf2603144.viaf007.perseus-eng1.xml",
                "label": "Commentary (R. C. Jebb)",
                "class": "commentary-text",
                "parse_mode": "line_commentary",
            },
        },        "treebanks": {
            "daphne-tb-grc1": {
                "path": "/Users/gcrane/github/Daphne/data/annotation/latest/tlg0011/tlg001/tlg0011.tlg001.daphne_tb-grc1.conllu",
                "label": "Treebank (Daphne AGDT)",
                "class": "treebank-text",
                "parse_mode": "conllu",
                "speakers_csv": "/Users/gcrane/github/Daphne/data/annotation/latest/tlg0011/tlg001/speakers.csv"
            },

             "daphne-tb-cn1": {
                "path": "/Users/gcrane/github/AntigonesPublic/data/tlg0011/tlg001/tlg0011.tlg001.daphne-cn.conllu",
                "label": "Treebank (Daphne AGDT CN)",
                "class": "treebank-text",
                "parse_mode": "conllu",
                "speakers_csv": "/Users/gcrane/github/Daphne/data/annotation/latest/tlg0011/tlg001/speakers.csv"
            }

        },
    },
    "tlg0011.tlg002": {
        "textgroup": "tlg0011",
            "work": "tlg002",
        "editions": {
            "perseus-grc2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0011/tlg002/tlg0011.tlg002.perseus-grc2.xml", "label": "Greek (F. Storr, 1912)", "class": "greek-text", "parse_mode": "poetry_cards"},
            "boeckh-grc1": {"path": "/Users/gcrane/github/AntigonesPublic/data/tlg0011/tlg002/tlg0011.tlg002.boeckh1884-grc1.xml", "label": "Greek (A. Boeckh, 1884)", "class": "greek-text", "parse_mode": "poetry_cards", "lineno_sigil": "B."}
        },
        "appcrits": {},
        "translations": {
            "perseus-eng2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0011/tlg002/tlg0011.tlg002.perseus-eng2.xml", "label": "English (Sir Richard Jebb2, 1904)", "class": "english-text", "parse_mode": "poetry_cards"},
            "hoelderlin1805-deu1": {"path": "/Users/gcrane/github/AntigonesPublic/data/tlg0011/tlg002/tlg0011.tlg002.hoelderlin1805-deu1.xml", "label": "German (F. H\u00f6lderlin, 1804)", "class": "german-text", "parse_mode": "poetry_cards", "card_anchor": "milestones", "lineno_sigil": "H."},
            "boeckh-deu2": {"path": "/Users/gcrane/github/AntigonesPublic/data/tlg0011/tlg002/tlg0011.tlg002.boeckh1884-deu2.xml", "label": "German (A. Boeckh, 1884)", "class": "german-text", "parse_mode": "poetry_cards", "lineno_sigil": "B."},
            "bothe1806-lat1": {"path": "/Users/gcrane/github/AntigonesPublic/data/tlg0011/tlg002/tlg0011.tlg002.bothe1806-lat1.xml", "label": "Latin (F. H. Bothe, 1806)", "class": "latin-text", "parse_mode": "card_prose"},
            "garbin1889-spa2": {"path": "/Users/gcrane/github/AntigonesPublic/data/tlg0011/tlg002/tlg0011.tlg002.garbin1889-spa2.xml", "label": "Spanish (A. G. Garbin, 1889)", "class": "spanish-text", "parse_mode": "card_prose"},
            "bolufer1921-spa2": {"path": "/Users/gcrane/github/AntigonesPublic/data/tlg0011/tlg002/tlg0011.tlg002.bolufer1921-spa2.xml", "label": "Spanish (J. A. Bolufer, 1921)", "class": "spanish-text", "parse_mode": "card_prose"},
            "jebb1904-eng1": {"path": "/Users/gcrane/github/AntigonesPublic/data/tlg0011/tlg002/tlg0011.tlg002.jebb1904-eng1.xml", "label": "English (Sir Richard Jebb1, 1904)", "class": "english-text", "parse_mode": "card_prose"}
        },
        "commentaries": {
            "jebb-com-eng1": {
                "path": "/Users/gcrane/github/canonical_pdlrefwk/data/viaf2603144/viaf003/viaf2603144.viaf003.perseus-eng1.xml",
                "label": "Commentary (R. C. Jebb)",
                "class": "commentary-text",
                "parse_mode": "line_commentary",
            },
        },
        "treebanks": {
            "daphne-tb-grc1": {
                "path": "/Users/gcrane/github/Daphne/data/annotation/latest/tlg0011/tlg002/tlg0011.tlg002.daphne_tb-grc1.conllu",
                "label": "Treebank (Daphne AGDT)",
                "class": "treebank-text",
                "parse_mode": "conllu",
                "speakers_csv": "/Users/gcrane/github/Daphne/data/annotation/latest/tlg0011/tlg002/speakers.csv"
            },
             "daphne-tb-cn1": {
                "path": "/Users/gcrane/github/AntigonesPublic/data/tlg0011/tlg002/tlg0011.tlg002.daphne-cn.conllu",
                "label": "Treebank (Daphne AGDT CN)",
                "class": "treebank-text",
                "parse_mode": "conllu",
                "speakers_csv": "/Users/gcrane/github/Daphne/data/annotation/latest/tlg0011/tlg004/speakers.csv"
            }

        },
        "edition_alignments": [
            "/Users/gcrane/github/AntigonesPublic/data/tlg0011/tlg002/tlg0011.tlg002.boeckh-grc1--perseus-grc2.align.tsv"
        ]
    },
    "tlg0011.tlg003": {
        "textgroup": "tlg0011",

        "work": "tlg003",
        "editions": {
            "perseus-grc2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0011/tlg003/tlg0011.tlg003.perseus-grc2.xml", "label": "Greek (F. Storr, 1912)", "class": "greek-text", "parse_mode": "poetry_cards"}
        },
        "appcrits": {},
        "translations": {
            "perseus-eng2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0011/tlg003/tlg0011.tlg003.perseus-eng2.xml", "label": "English (Sir Richard Jebb, 1904)", "class": "english-text", "parse_mode": "poetry_cards"}
        },
        "commentaries": {
            "jebb-com-eng1": {
                "path": "/Users/gcrane/github/canonical_pdlrefwk/data/viaf2603144/viaf004/viaf2603144.viaf004.perseus-eng1.xml",
                "label": "Commentary (R. C. Jebb)",
                "class": "commentary-text",
                "parse_mode": "line_commentary",
            },
        },        "treebanks": {
            "daphne-tb-grc1": {
                "path": "/Users/gcrane/github/Daphne/data/annotation/latest/tlg0011/tlg003/tlg0011.tlg003.daphne_tb-grc1.conllu",
                "label": "Treebank (Daphne AGDT)",
                "class": "treebank-text",
                "parse_mode": "conllu",
                "speakers_csv": "/Users/gcrane/github/Daphne/data/annotation/latest/tlg0011/tlg003/speakers.csv"
            },

             "daphne-tb-cn1": {
                "path": "/Users/gcrane/github/AntigonesPublic/data/tlg0011/tlg003/tlg0011.tlg003.daphne-cn.conllu",
                "label": "Treebank (Daphne AGDT CN)",
                "class": "treebank-text",
                "parse_mode": "conllu",
                "speakers_csv": "/Users/gcrane/github/Daphne/data/annotation/latest/tlg0011/tlg003/speakers.csv"
            }

            
        },
    },
    "tlg0011.tlg005": {
        "textgroup": "tlg0011",

        "work": "tlg005",
        "editions": {
            "perseus-grc2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0011/tlg005/tlg0011.tlg005.perseus-grc2.xml", "label": "Greek (F. Storr, 1912)", "class": "greek-text", "parse_mode": "poetry_cards"}
        },
        "appcrits": {},
        "translations": {
            "perseus-eng3": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0011/tlg005/tlg0011.tlg005.perseus-eng2.xml", "label": "English (Sir Richard Jebb, 1904)", "class": "english-text", "parse_mode": "poetry_cards"}
        },
        "commentaries": {
            "jebb-com-eng1": {
                "path": "/Users/gcrane/github/canonical_pdlrefwk/data/viaf2603144/viaf005/viaf2603144.viaf005.perseus-eng1.xml",
                "label": "Commentary (R. C. Jebb)",
                "class": "commentary-text",
                "parse_mode": "line_commentary",
            },
        },        "treebanks": {
            "daphne-tb-grc1": {
                "path": "/Users/gcrane/github/Daphne/data/annotation/latest/tlg0011/tlg005/tlg0011.tlg005.daphne_tb-grc1.conllu",
                "label": "Treebank (Daphne AGDT)",
                "class": "treebank-text",
                "parse_mode": "conllu",
                "speakers_csv": "/Users/gcrane/github/Daphne/data/annotation/latest/tlg0011/tlg005/speakers.csv"
            },

             "daphne-tb-cn1": {
                "path": "/Users/gcrane/github/AntigonesPublic/data/tlg0011/tlg005/tlg0011.tlg005.daphne-cn.conllu",
                "label": "Treebank (Daphne AGDT CN)",
                "class": "treebank-text",
                "parse_mode": "conllu",
                "speakers_csv": "/Users/gcrane/github/Daphne/data/annotation/latest/tlg0011/tlg005/speakers.csv"
            },

            
        },
    },
    "tlg0011.tlg006": {
        "textgroup": "tlg0011",

        "work": "tlg006",
        "editions": {
            "perseus-grc2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0011/tlg006/tlg0011.tlg006.perseus-grc2.xml", "label": "Greek (F. Storr, 1912)", "class": "greek-text", "parse_mode": "poetry_cards"}
        },
        "appcrits": {},
        "translations": {
            "perseus-eng3": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0011/tlg006/tlg0011.tlg006.perseus-eng2.xml", "label": "English (Sir Richard Jebb, 1904)", "class": "english-text", "parse_mode": "poetry_cards"}
        },
        "commentaries": {
            "jebb-com-eng1": {
                "path": "/Users/gcrane/github/canonical_pdlrefwk/data/viaf2603144/viaf006/viaf2603144.viaf006.perseus-eng1.xml",
                "label": "Commentary (R. C. Jebb)",
                "class": "commentary-text",
                "parse_mode": "line_commentary",
            },
        },        "treebanks": {
            "daphne-tb-grc1": {
                "path": "/Users/gcrane/github/Daphne/data/annotation/latest/tlg0011/tlg006/tlg0011.tlg006.daphne_tb-grc1.conllu",
                "label": "Treebank (Daphne AGDT)",
                "class": "treebank-text",
                "parse_mode": "conllu",
                "speakers_csv": "/Users/gcrane/github/Daphne/data/annotation/latest/tlg0011/tlg006/speakers.csv"
            },
             "daphne-tb-cn1": {
                "path": "/Users/gcrane/github/AntigonesPublic/data/tlg0011/tlg006/tlg0011.tlg006.daphne-cn.conllu",
                "label": "Treebank (Daphne AGDT CN)",
                "class": "treebank-text",
                "parse_mode": "conllu",
                "speakers_csv": "/Users/gcrane/github/Daphne/data/annotation/latest/tlg0011/tlg006/speakers.csv"
            },

            
        },
    },
    "tlg0011.tlg007": {
        "textgroup": "tlg0011",

        "work": "tlg007",
        "editions": {
            "perseus-grc2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0011/tlg007/tlg0011.tlg007.perseus-grc2.xml", "label": "Greek (F. Storr, 1912)", "class": "greek-text", "parse_mode": "poetry_cards"}
        },
        "appcrits": {},
        "translations": {
            "perseus-eng3": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0011/tlg007/tlg0011.tlg007.perseus-eng2.xml", "label": "English (Sir Richard Jebb, 1904)", "class": "english-text", "parse_mode": "poetry_cards"}
        },
        "commentaries": {
            "jebb-com-eng1": {
                "path": "/Users/gcrane/github/canonical_pdlrefwk/data/viaf2603144/viaf002/viaf2603144.viaf002.perseus-eng1.xml",
                "label": "Commentary (R. C. Jebb)",
                "class": "commentary-text",
                "parse_mode": "line_commentary",
            },
        },        "treebanks": {
            "daphne-tb-grc1": {
                "path": "/Users/gcrane/github/Daphne/data/annotation/latest/tlg0011/tlg007/tlg0011.tlg007.daphne_tb-grc1.conllu",
                "label": "Treebank (Daphne AGDT)",
                "class": "treebank-text",
                "parse_mode": "conllu",
                "speakers_csv": "/Users/gcrane/github/Daphne/data/annotation/latest/tlg0011/tlg007/speakers.csv"
            },

            
        },
    },
    "tlg0011.tlg004": {
        "textgroup": "tlg0011",

        "work": "tlg004",
        "editions": {
            "perseus-grc2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0011/tlg004/tlg0011.tlg004.perseus-grc2.xml", "label": "Greek (F. Storr, 1912)", "class": "greek-text", "parse_mode": "poetry_cards"}
        },
        "appcrits": {},
        "translations": {
            "perseus-eng2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0011/tlg004/tlg0011.tlg004.perseus-eng2.xml", "label": "English (Sir Richard Jebb, 1904)", "class": "english-text", "parse_mode": "poetry_cards"}
        },
        "commentaries": {},
        "treebanks": {
            "daphne-tb-grc1": {
                "path": "/Users/gcrane/github/Daphne/data/annotation/latest/tlg0011/tlg004/tlg0011.tlg004.daphne_tb-grc1.conllu",
                "label": "Treebank (Daphne AGDT)",
                "class": "treebank-text",
                "parse_mode": "conllu",
                "speakers_csv": "/Users/gcrane/github/Daphne/data/annotation/latest/tlg0011/tlg004/speakers.csv"
            },
             "daphne-tb-cn1": {
                "path": "/Users/gcrane/github/AntigonesPublic/data/tlg0011/tlg004/tlg0011.tlg004.daphne-cn.conllu",
                "label": "Treebank (Daphne AGDT CN)",
                "class": "treebank-text",
                "parse_mode": "conllu",
                "speakers_csv": "/Users/gcrane/github/Daphne/data/annotation/latest/tlg0011/tlg004/speakers.csv"
            },
        }
    }
}

WORK_REGISTRY["tlg0020.tlg001"] = {
    "textgroup": "tlg0020",
    "work": "tlg001",
    "editions": {
        "perseus-grc2": {
            "path": "/Users/gcrane/github/canonical-greekLit/data/tlg0020/tlg001/tlg0020.tlg001.perseus-grc2.xml",
            "label": "Greek (Hugh G. Evelyn-White, 1914)",
            "class": "greek-text",
            "parse_mode": "poetry_cards"
        }
    },
    "appcrits": {},
    "translations": {
        "perseus-eng2": {
            "path": "/Users/gcrane/github/canonical-greekLit/data/tlg0020/tlg001/tlg0020.tlg001.perseus-eng2.xml",
            "label": "English (Hugh G. Evelyn-White, 1914)",
            "class": "english-text",
            "parse_mode": "poetry_cards"
        }
    },
    "commentaries": {},
    "treebanks": {
        "daphne_perstbgl-grc1": {
            "path": "/Users/gcrane/github/Daphne/data/annotation/latest/tlg0020/grc_versions/tlg0020.tlg001.daphne_perstbgl-grc1.corrected.conllu",
            "label": "Treebank (Daphne AGDT)",
            "class": "treebank-text",
            "parse_mode": "conllu"
        }
    }
}


WORK_REGISTRY["tlg0020.tlg002"] = {
    "textgroup": "tlg0020",
    "work": "tlg002",
    "editions": {
        "perseus-grc2": {
            "path": "/Users/gcrane/github/canonical-greekLit/data/tlg0020/tlg002/tlg0020.tlg002.perseus-grc2.xml",
            "label": "Greek (Hugh G. Evelyn-White, 1914)",
            "class": "greek-text",
            "parse_mode": "poetry_cards"
        }
    },
    "appcrits": {},
    "translations": {
        "perseus-eng2": {
            "path": "/Users/gcrane/github/canonical-greekLit/data/tlg0020/tlg002/tlg0020.tlg002.perseus-eng2.xml",
            "label": "English (Hugh G. Evelyn-White, 1914)",
            "class": "english-text",
            "parse_mode": "poetry_cards"
        }
    },
    "commentaries": {},
    "treebanks": {
        "daphne_perstbgl-grc1": {
            "path": "/Users/gcrane/github/Daphne/data/annotation/latest/tlg0020/grc_versions/tlg0020.tlg002.daphne_perstbgl-grc1.corrected.conllu",
            "label": "Treebank (Daphne AGDT)",
            "class": "treebank-text",
            "parse_mode": "conllu"
        }
    }
}


WORK_REGISTRY["tlg0020.tlg003"] = {
    "textgroup": "tlg0020",
    "work": "tlg003",
    "editions": {
        "perseus-grc2": {
            "path": "/Users/gcrane/github/canonical-greekLit/data/tlg0020/tlg003/tlg0020.tlg003.perseus-grc2.xml",
            "label": "Greek (Hugh G. Evelyn-White, 1914)",
            "class": "greek-text",
            "parse_mode": "poetry_cards"
        }
    },
    "appcrits": {},
    "translations": {
        "perseus-eng2": {
            "path": "/Users/gcrane/github/canonical-greekLit/data/tlg0020/tlg003/tlg0020.tlg003.perseus-eng2.xml",
            "label": "English (Hugh G. Evelyn-White, 1914)",
            "class": "english-text",
            "parse_mode": "poetry_cards"
        }
    },
    "commentaries": {},
    "treebanks": {
        "daphne_perstbgl-grc1": {
            "path": "/Users/gcrane/github/Daphne/data/annotation/latest/tlg0020/grc_versions/tlg0020.tlg003.daphne_perstbgl-grc1.corrected.conllu",
            "label": "Treebank (Daphne AGDT)",
            "class": "treebank-text",
            "parse_mode": "conllu"
        }
    }
}


WORK_REGISTRY["ferdowsi.shahnameh"] = {
    "textgroup": "ferdowsi",
    "work": "shahnameh",
    "editions": {
        "pizzi1883-fas1": {
            "path": "/Users/gcrane/github/Shahnameh/pizzi-readings.fa.xml",
            "label": "Persian (I. Pizzi, ed., 1883)",
            "class": "persian-text",
            "parse_mode": "reading_lines"   # new mode — see #2 below
        }
    },
    "appcrits": {},
    "translations": {
        "pizzi1883-eng1": {
            "path": "/Users/gcrane/github/Shahnameh/pizzi-readings.eng.xml",
            "label": "English (trans. Pizzi selections, 1883)",
            "class": "english-text",
            "parse_mode": "reading_lines"
        }
    },
    "commentaries": {},
    "treebanks": {
        "pizzi-tb-fas1": {
            "path": "/Users/gcrane/github/Shahnameh/pizzi.readings.fa.conllu",
            "label": "Treebank (Pizzi Persian selections, UD)",
            "class": "treebank-text",
            "parse_mode": "conllu"
        }
    }
}


WORK_REGISTRY["tlg0085.tlg001"] = {
    "textgroup": "tlg0085",
    "work": "tlg001",
    "editions": {
        "perseus-grc2": {
            "path": "/Users/gcrane/github/canonical-greekLit/data/tlg0085/tlg001/tlg0085.tlg001.perseus-grc2.xml",
            "label": "Greek (Herbert Weir Smyth, 1926)",
            "class": "greek-text",
            "parse_mode": "poetry_cards"
        }
    },
    "appcrits": {},
    "translations": {
        "perseus-eng2": {
            "path": "/Users/gcrane/github/canonical-greekLit/data/tlg0085/tlg001/tlg0085.tlg001.perseus-eng2.xml",
            "label": "English (Herbert Weir Smyth, 1926)",
            "class": "english-text",
            "parse_mode": "poetry_cards"
        }
    },
    "commentaries": {},
    "treebanks": {
        "perseus-tb-grc1": {
            "path": "/Users/gcrane/github/GRC_misc/aesch/AeschTBTrans/tlg0085.tlg001.tbanktrans2.xml",
            "label": "Treebank (Perseus AGDT)",
            "class": "treebank-text",
            "parse_mode": "agdt_xml"
        }
    }
}

WORK_REGISTRY["tlg0085.tlg002"] = {
    "textgroup": "tlg0085",
    "work": "tlg002",
    "editions": {
        "perseus-grc2": {
            "path": "/Users/gcrane/github/canonical-greekLit/data/tlg0085/tlg002/tlg0085.tlg002.perseus-grc2.xml",
            "label": "Greek (Herbert Weir Smyth, 1926)",
            "class": "greek-text",
            "parse_mode": "poetry_cards"
        }
    },
    "appcrits": {},
    "translations": {
        "perseus-eng2": {
            "path": "/Users/gcrane/github/canonical-greekLit/data/tlg0085/tlg002/tlg0085.tlg002.perseus-eng2.xml",
            "label": "English (Herbert Weir Smyth, 1926)",
            "class": "english-text",
            "parse_mode": "poetry_cards"
        }
    },
    "commentaries": {},
    "treebanks": {
        "perseus-tb-grc1": {
            "path": "/Users/gcrane/github/GRC_misc/aesch/AeschTBTrans/tlg0085.tlg002.tbanktrans2.xml",
            "label": "Treebank (Perseus AGDT)",
            "class": "treebank-text",
            "parse_mode": "agdt_xml"
        }
    }
}


WORK_REGISTRY["tlg0085.tlg003"] = {
    "textgroup": "tlg0085",
    "work": "tlg003",
    "editions": {
        "perseus-grc2": {
            "path": "/Users/gcrane/github/canonical-greekLit/data/tlg0085/tlg003/tlg0085.tlg003.perseus-grc2.xml",
            "label": "Greek (Herbert Weir Smyth, 1926)",
            "class": "greek-text",
            "parse_mode": "poetry_cards"
        }
    },
    "appcrits": {},
    "translations": {
        "perseus-eng2": {
            "path": "/Users/gcrane/github/canonical-greekLit/data/tlg0085/tlg003/tlg0085.tlg003.perseus-eng2.xml",
            "label": "English (Herbert Weir Smyth, 1926)",
            "class": "english-text",
            "parse_mode": "poetry_cards"
        }
    },
    "commentaries": {},
    "treebanks": {
        "perseus-tb-grc1": {
            "path": "/Users/gcrane/github/GRC_misc/aesch/AeschTBTrans/tlg0085.tlg003.tbanktrans2.conllu",
            "label": "Treebank (Perseus AGDT)",
            "class": "treebank-text",
            "parse_mode": "conllu"
        }
    }
}
WORK_REGISTRY["tlg0085.tlg004"] = {
    "textgroup": "tlg0085",
    "work": "tlg004",
    "editions": {
        "perseus-grc2": {
            "path": "/Users/gcrane/github/canonical-greekLit/data/tlg0085/tlg004/tlg0085.tlg004.perseus-grc2.xml",
            "label": "Greek (Herbert Weir Smyth, 1926)",
            "class": "greek-text",
            "parse_mode": "poetry_cards"
        }
    },
    "appcrits": {},
    "translations": {
        "perseus-eng2": {
            "path": "/Users/gcrane/github/canonical-greekLit/data/tlg0085/tlg004/tlg0085.tlg004.perseus-eng2.xml",
            "label": "English (Herbert Weir Smyth, 1926)",
            "class": "english-text",
            "parse_mode": "poetry_cards"
        }
    },
    "commentaries": {},
    "treebanks": {
        "perseus-tb-grc1": {
            "path": "/Users/gcrane/github/GRC_misc/aesch/AeschTBTrans/tlg0085.tlg004.tbanktrans2.xml",
            "label": "Treebank (Perseus AGDT)",
            "class": "treebank-text",
            "parse_mode": "agdt_xml"
        }
    }
}


WORK_REGISTRY["tlg0085.tlg005"] = {
    "textgroup": "tlg0085",
    "work": "tlg005",
    "editions": {
        "perseus-grc2": {
            "path": "/Users/gcrane/github/canonical-greekLit/data/tlg0085/tlg005/tlg0085.tlg005.perseus-grc2.xml",
            "label": "Greek (Herbert Weir Smyth, 1926)",
            "class": "greek-text",
            "parse_mode": "poetry_cards"
        }
    },
    "appcrits": {},
    "translations": {
        "perseus-eng3": {
            "path": "/Users/gcrane/github/canonical-greekLit/data/tlg0085/tlg005/tlg0085.tlg005.perseus-eng3.xml",
            "label": "English (Herbert Weir Smyth, 1926)",
            "class": "english-text",
            "parse_mode": "poetry_cards"
        },

        "perseus-eng4": {
            "path": "/Users/gcrane/github/canonical-greekLit/data/tlg0085/tlg005/tlg0085.tlg005.perseus-eng4.xml",
            "label": "English (Robert Browning, 1889)",
            "class": "english-text",
            "parse_mode": "poetry_cards"
        }
    },

    "commentaries": {},
    "treebanks": {
        "perseus-tb-grc1": {
            "path": "/Users/gcrane/github/GRC_misc/aesch/AeschTBTrans/tlg0085.tlg005.tbanktrans.conllu",
            "label": "Treebank (Perseus AGDT)",
            "class": "treebank-text",
            "parse_mode": "conllu"
        }
    }
}

WORK_REGISTRY["tlg0085.tlg006"] = {
    "textgroup": "tlg0085",
    "work": "tlg006",
    "editions": {
        "perseus-grc2": {
            "path": "/Users/gcrane/github/canonical-greekLit/data/tlg0085/tlg006/tlg0085.tlg006.perseus-grc2.xml",
            "label": "Greek (Herbert Weir Smyth, 1926)",
            "class": "greek-text",
            "parse_mode": "poetry_cards"
        }
    },
    "appcrits": {},
    "translations": {
        "perseus-eng2": {
            "path": "/Users/gcrane/github/canonical-greekLit/data/tlg0085/tlg006/tlg0085.tlg006.perseus-eng2.xml",
            "label": "English (Herbert Weir Smyth, 1926)",
            "class": "english-text",
            "parse_mode": "poetry_cards"
        }
    },
    "commentaries": {},
    "treebanks": {
        "perseus-tb-grc1": {
            "path": "/Users/gcrane/github/GRC_misc/aesch/AeschTBTrans/tlg0085.tlg006.tbanktrans2.conllu",
            "label": "Treebank (Perseus AGDT)",
            "class": "treebank-text",
            "parse_mode": "conllu"
        }
    }
}



WORK_REGISTRY["tlg0085.tlg007"] = {
    "textgroup": "tlg0085",
    "work": "tlg007",
    "editions": {
        "perseus-grc2": {
            "path": "/Users/gcrane/github/canonical-greekLit/data/tlg0085/tlg007/tlg0085.tlg007.perseus-grc2.xml",
            "label": "Greek (Herbert Weir Smyth, 1926)",
            "class": "greek-text",
            "parse_mode": "poetry_cards"
        }
    },
    "appcrits": {},
    "translations": {
        "perseus-eng2": {
            "path": "/Users/gcrane/github/canonical-greekLit/data/tlg0085/tlg007/tlg0085.tlg007.perseus-eng2.xml",
            "label": "English (Herbert Weir Smyth, 1926)",
            "class": "english-text",
            "parse_mode": "poetry_cards"
        }
    },
    "commentaries": {},
    "treebanks": {
        "perseus-tb-grc1": {
            "path": "/Users/gcrane/github/GRC_misc/aesch/AeschTBTrans/tlg0085.tlg007.tbanktrans2.conllu",
            "label": "Treebank (Perseus AGDT)",
            "class": "treebank-text",
            "parse_mode": "conllu"
        }
    }
}

def ingest_edition_alignment(conn, path, *, textgroup=None, work=None,
                             base_version=None, target_version=None):
    meta = {"work": work, "base_version": base_version, "target_version": target_version}
    header, recs = None, []
    with open(path, encoding="utf-8") as fh:
        for raw in fh:
            line = raw.rstrip("\n")
            if line.startswith("#"):
                if (m := re.match(r"#\s*work:\s*([\w.]+)", line)) and not meta["work"]:
                    meta["work"] = m.group(1)
                if (m := re.match(r"#\s*base[^:]*:\s*([\w-]+)", line)) and not meta["base_version"]:
                    meta["base_version"] = m.group(1)
                if (m := re.match(r"#\s*target:\s*([\w-]+)", line)) and not meta["target_version"]:
                    meta["target_version"] = m.group(1)
                continue
            if not line.strip():
                continue
            cells = line.split("\t")
            if header is None:
                header = cells; continue
            recs.append(dict(zip(header, cells)))

    tg = textgroup or (meta["work"].split(".")[0] if meta["work"] and "." in meta["work"] else None)
    wk = meta["work"].split(".")[1] if (meta["work"] and "." in meta["work"]) else work
    if not (tg and wk and meta["base_version"] and meta["target_version"]):
        raise ValueError(f"Could not resolve identity: tg={tg} wk={wk} meta={meta}")
    pair_id = f"{meta['base_version']}--{meta['target_version']}"

    cells_to_list = lambda c: [x.strip() for x in (c or "").split(",") if x.strip()]

    conn.execute("""
        CREATE TABLE IF NOT EXISTS edition_line_alignments (
            id INTEGER PRIMARY KEY,
            textgroup TEXT NOT NULL, work TEXT NOT NULL, pair_id TEXT NOT NULL,
            base_version TEXT NOT NULL, target_version TEXT NOT NULL,
            seq INTEGER NOT NULL,
            base_lines TEXT NOT NULL, target_lines TEXT NOT NULL,  -- JSON arrays
            type TEXT NOT NULL, score REAL, review INTEGER NOT NULL DEFAULT 0
        )""")
    conn.execute("CREATE INDEX IF NOT EXISTS ix_ela_pair "
                 "ON edition_line_alignments(textgroup, work, pair_id)")
    conn.execute("DELETE FROM edition_line_alignments WHERE textgroup=? AND work=? AND pair_id=?",
                 (tg, wk, pair_id))
    for seq, r in enumerate(recs):
        conn.execute("""INSERT INTO edition_line_alignments
            (textgroup, work, pair_id, base_version, target_version, seq,
             base_lines, target_lines, type, score, review)
            VALUES (?,?,?,?,?,?,?,?,?,?,?)""",
            (tg, wk, pair_id, meta["base_version"], meta["target_version"], seq,
             json.dumps(cells_to_list(r.get("base"))),
             json.dumps(cells_to_list(r.get("target"))),
             r.get("type"), float(r["score"]) if r.get("score") else None,
             1 if (r.get("review") or "").strip().upper() == "REVIEW" else 0))
    conn.commit()
    print(f"  ✓ edition_line_alignments [{tg}.{wk}] {pair_id}: {len(recs)} rows")

    
# ── CTS namespace derivation (textgroup prefix → namespace) ───────────────
TEXTGROUP_NAMESPACE = {"tlg": "greekLit", "phi": "latinLit"}
TEXTGROUP_NAMESPACE = {"tlg": "greekLit", "phi": "latinLit", "ferdowsi": "persLit"}
def _ns(work_key):
    """Derive the CTS namespace from the textgroup prefix, e.g. tlg0012 → greekLit.
    Extend TEXTGROUP_NAMESPACE for other collections (stoa, etc.)."""
    _tg = work_key.split(".")[0]
    _m = re.match(r"[A-Za-z]+", _tg)
    _val = TEXTGROUP_NAMESPACE.get(_m.group(0).lower() if _m else "")
    if _val is None:
        print(f"  \u26a0 unknown textgroup prefix in '{work_key}'; defaulting to greekLit")
        _val = "greekLit"
    return _val


def init_storage_engine(db_path):
    if db_path.exists():
        db_path.unlink()
        
    conn = sqlite3.connect(str(db_path))
    cursor = conn.cursor()
    
    cursor.execute("PRAGMA page_size = 4096;")
    cursor.execute("PRAGMA journal_mode = OFF;")
    cursor.execute("PRAGMA synchronous = OFF;")
    
    cursor.execute("""
        CREATE TABLE text_units (
            canonical_id TEXT PRIMARY KEY,
            urn TEXT NOT NULL,
            label TEXT NOT NULL,
            text_class TEXT NOT NULL,
            textgroup TEXT NOT NULL,
            work TEXT NOT NULL,
            short_id TEXT NOT NULL,
            doc_type TEXT NOT NULL
        );
    """)
    cursor.execute("""
        CREATE TABLE alignment_grid (
            passage_urn TEXT PRIMARY KEY,
            textgroup TEXT NOT NULL,
            work TEXT NOT NULL,
            book TEXT,
            chapter TEXT NOT NULL,
            section TEXT NOT NULL,
            prev_urn TEXT,
            next_urn TEXT,
            sort_order INTEGER NOT NULL
        );
    """)
    cursor.execute("""
        CREATE TABLE text_segments (
            passage_urn TEXT,
            version_short_id TEXT,
            content_html TEXT NOT NULL,
            PRIMARY KEY (passage_urn, version_short_id),
            FOREIGN KEY (passage_urn) REFERENCES alignment_grid(passage_urn)
        );
    """)
    cursor.execute("""
        CREATE TABLE treebank_sentences (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            textgroup TEXT NOT NULL,
            work TEXT NOT NULL,
            version_short_id TEXT NOT NULL,
            subdoc TEXT NOT NULL,
            chapter TEXT NOT NULL,
            section TEXT NOT NULL,
            sentence_json TEXT NOT NULL,
            prose_translation TEXT,
            literal_translation TEXT,
            transliteration TEXT,
            credits_json TEXT
        );
    """)
    cursor.execute("""
        CREATE TABLE treebank_doc_credits (
            textgroup TEXT NOT NULL,
            work TEXT NOT NULL,
            version_short_id TEXT NOT NULL,
            source_repo TEXT,
            credits_json TEXT,
            PRIMARY KEY (textgroup, work, version_short_id)
        );
    """)
    cursor.execute("""
        CREATE TABLE treebank_speakers (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            textgroup TEXT NOT NULL,
            work TEXT NOT NULL,
            subdoc TEXT NOT NULL,
            speaker TEXT NOT NULL
        );
    """)
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS token_alignments (
            id           INTEGER PRIMARY KEY AUTOINCREMENT,
            textgroup    TEXT NOT NULL,
            work         TEXT NOT NULL,
            pair_id      TEXT NOT NULL,
            src_version  TEXT NOT NULL,
            tgt_version  TEXT NOT NULL,
            segment      TEXT NOT NULL,
            src_indices  TEXT NOT NULL,
            tgt_indices  TEXT NOT NULL,
            src_tokens   TEXT NOT NULL,
            tgt_tokens   TEXT NOT NULL,
            score        REAL NOT NULL
        );
    """)
    cursor.executescript('''
        CREATE TABLE IF NOT EXISTS metrical_lines (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            textgroup TEXT NOT NULL,
            work TEXT NOT NULL,
            version_short_id TEXT NOT NULL,
            line_ref TEXT NOT NULL,
            chapter TEXT NOT NULL,
            line_json TEXT NOT NULL
        );
        CREATE INDEX IF NOT EXISTS idx_metrical_chapter
            ON metrical_lines(textgroup, work, version_short_id, chapter);
    ''')
    
    cursor.execute("CREATE INDEX idx_grid_lookup ON alignment_grid(textgroup, work, book, chapter);")
    cursor.execute("CREATE INDEX idx_segments_lookup ON text_segments(passage_urn);")
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_align_lookup ON token_alignments(textgroup, work, pair_id, segment);")
    cursor.execute("CREATE INDEX idx_tb_lookup ON treebank_sentences(textgroup, work, version_short_id, chapter);")
    cursor.execute("CREATE INDEX idx_tb_subdoc ON treebank_sentences(textgroup, work, subdoc);")
    
    conn.commit()
    return conn

def generate_canonical_id(work_key, label_string, doc_type="edition"):
    prefix = work_key.replace('.', '_')
    match = re.search(r'\((.*?)\)', label_string)
    if match:
        content = match.group(1)
        parts = content.split(',')
        editor = parts[0].strip()
        year = parts[1].strip() if len(parts) > 1 else ""
        editor_last = editor.split()[-1].lower().replace('.', '')
        year_clean = re.sub(r'\D', '', year)
        base_id = f"{prefix}_{editor_last}_{year_clean}"
    else:
        fallback_suffix = re.sub(r'\W+', '_', label_string).lower()
        base_id = f"{prefix}_{fallback_suffix}"

    if doc_type != "edition":
        base_id = f"{base_id}_{doc_type}"
    return base_id

def safe_parse(path):
    if _LXML:
        parser = LET.XMLParser(recover=True)
        tree = LET.parse(path, parser=parser)
        import io
        buf = io.BytesIO()
        tree.write(buf)
        buf.seek(0)
        return ET.parse(buf)
    return ET.parse(path)

def find_text_root(root):
    for div in root.findall('.//{http://www.tei-c.org/ns/1.0}div') + root.findall('.//div'):
        if div.get('type') in ('translation', 'commentary'): return div
    return root.find('.//{http://www.tei-c.org/ns/1.0}body') or root.find('.//body') or root.find('.//*body')

def extract_text_recursive(elem, strip_paragraphs=False, lineno_sigil=None):
    parts = []
    tag = elem.tag.split('}')[-1]
    
    if tag == 'l':
        line_num = (elem.get('n') or '').strip()
        if line_num:
            if lineno_sigil:
                parts.append(f'<div class="line-num-cell" data-n="{line_num}">'
                             f'<span class="src-lineno">[{line_num} {lineno_sigil}]</span></div>')
            else:
                parts.append(f'<div class="line-num-cell">{line_num}</div>')
        else:
            parts.append('<div class="line-num-cell\">&nbsp;</div>')
        parts.append('<div class="line-text-cell">')
    
    elif tag == 'sic':
        parts.append('<span class="tei-sic">[')
    elif tag == 'corr':
        parts.append(' <span class="tei-corr">')
    elif tag == 'speaker': 
        parts.append('<strong class="speaker-attr">')
    elif tag == 'stage':
        parts.append('<div class=\"stage-direction\">')
    elif tag == 'hi':
        rend = elem.get('rend', 'italic')
        parts.append(f'<span class="render-{rend}">')
    elif tag == 's':
        parts.append('<span class="lemma">')
    elif tag == 'del':
        parts.append('<span class="tei-del">[')
    elif tag == 'add':
        parts.append('<span class="tei-add">\u27e8')   # ⟨
    elif tag == 'quote':
        q_type = elem.get('type', 'blockquote')
        parts.append(f'<div class="quote-block type-{q_type}">')
    elif tag == 'p' and not strip_paragraphs:
        parts.append('<div class="prose-para">')
    elif tag == 'milestone' and elem.get('unit') == 'line':
        ln_num = (elem.get('n') or '').strip()
        if ln_num:
            parts.append(f'<span class="inline-line-milestone" title="Line Milestone {ln_num}">{ln_num}</span>')
    elif tag == 'milestone' and elem.get('unit') == 'page':
        pg = (elem.get('n') or '').strip()
        resp = (elem.get('resp') or '').strip()
        if pg:
            tip = (resp + ' ' + pg).strip()
            parts.append(f'<span class="milestone bekker-page" data-resp="{resp}" title="{tip}">{pg}</span>')
    elif tag == 'q':
        parts.append('\u201c')
    elif tag == 'foreign':
        lang = (elem.get('{http://www.w3.org/XML/1998/namespace}lang') or '').strip()
        cls  = f'foreign foreign-{lang}' if lang else 'foreign'
        parts.append(f'<span class="{cls}" lang="{lang}">')
    elif tag == 'seg' and elem.get('type') == 'metrical-part':
        part_n = elem.get('n', '')
        parts.append(f'<span class="metrical-part metrical-part-{part_n}">')
    elif tag == 'seg':
        rend = (elem.get('rend') or '').strip()
        parts.append(f'<span class="seg seg-{rend}">' if rend else '<span class="seg">')
    if elem.text: 
        parts.append(elem.text)
        
    for child in elem:
        child_tag = child.tag.split('}')[-1]
        if child_tag == 'note':
            note_text = extract_text_recursive(child, strip_paragraphs).strip()
            if note_text: parts.append(f'<span class="note">[{note_text}]</span>')
        elif child_tag == 'lb': 
            parts.append('<br/>')
        else: 
            parts.append(extract_text_recursive(child, strip_paragraphs))
        # Tail text after a block-level child must go in its own div,
        # not raw text, to avoid invalid HTML and browser reflow bugs.
        if child.tail and child.tail.strip():
            if child_tag in ('quote', 'stage'):
                parts.append(f'<div class="prose-continuation">{child.tail}</div>')
            else:
                parts.append(child.tail)
        elif child.tail:
            parts.append(child.tail)
            
    if tag == 'l': 
        parts.append('</div>')
    elif tag == 'sic': parts.append(']</span>')
    elif tag == 'corr': parts.append('</span>')
    elif tag == 'speaker': parts.append(': </strong>')
    elif tag == 'stage': parts.append('</div>')
    elif tag == 'hi': parts.append('</span>')
    elif tag == 's': parts.append('</span>')
    elif tag == 'del': parts.append(']</span>')
    elif tag == 'add': parts.append('\u27e9</span>')
    elif tag == 'quote': 
        parts.append('</div>')
    elif tag == 'p' and not strip_paragraphs: parts.append('</div>')
    elif tag == 'q':
        parts.append('\u201d')
    elif tag == 'foreign':
        parts.append('</span>')
    elif tag == 'seg' and elem.get('type') == 'metrical-part':
        parts.append('</span>')
    elif tag == 'seg':
        parts.append('</span>')        
    return ''.join(parts)

def build_poetry_canonical_intervals(editions_dict):
    grc_cfg = editions_dict.get("perseus-grc2") or list(editions_dict.values())[0]
    tree = safe_parse(grc_cfg["path"])
    text_entry = find_text_root(tree.getroot())

    landmarks = []  
    current_book = "1"
    
    for elem in text_entry.iter():
        tag = elem.tag.split("}")[-1]
        if tag == "div":
            subtype = (elem.get("subtype") or elem.get("type") or "").lower()
            if subtype == "book" and elem.get("n"):
                current_book = elem.get("n").strip()
        elif tag == "milestone" and elem.get("unit") == "card":
            card_n = (elem.get("n") or "").strip()
            if card_n:
                landmarks.append({"type": "card", "val": card_n, "book": current_book, "lines": []})
        elif tag == "l":
            ln = (elem.get("n") or "").strip()
            if ln and landmarks:
                # A verse @n can be a plain integer ("168") or a lettered
                # sub-line ("162a", used e.g. for an editorially-repeated
                # refrain/ephymnion in Aeschylus). Either way, take the
                # leading digit run as its canonical line number so the
                # card still gets a real interval instead of being dropped.
                m = re.match(r'^(\d+)', ln)
                if m:
                    landmarks[-1]["lines"].append(int(m.group(1)))

    book_intervals = {}
    
    for idx, item in enumerate(landmarks):
        bk = item["book"]
        card_n = item["val"]
        
        if item["lines"]:
            first_l = min(item["lines"])
            last_l = max(item["lines"])
        else:
            # Card had no <l> children at all (empty/structural milestone) --
            # fall back to the card's own number, taking its leading digit
            # run so a lettered milestone value ("162a") still resolves
            # instead of silently defaulting to line 1.
            m = re.match(r'^(\d+)', card_n)
            first_l = int(m.group(1)) if m else 1
            if idx + 1 < len(landmarks) and landmarks[idx+1]["book"] == bk:
                next_card = landmarks[idx+1]["val"]
                m2 = re.match(r'^(\d+)', next_card)
                last_l = (int(m2.group(1)) - 1) if m2 else first_l
            else:
                last_l = first_l

        book_intervals.setdefault(bk, []).append({
            "card_n": card_n, 
            "label": f"{first_l}-{last_l}", 
            "book": bk,
            "start_line": first_l,
            "end_line": last_l
        })

    return book_intervals

def build_line_remap(tsv_path):
    """From an edition-alignment crosswalk TSV, build {target_line(str): base_line(int)}
    so a NON-baseline edition is binned onto the baseline edition's card structure by
    its TRUE counterpart line, not its own raw line number.
      - split / merge  -> first base (baseline) line of the group
      - target_only    -> preceding base line (attach to the preceding card)
    Returns (base_version, target_version, remap_dict)."""
    base_v = target_v = None
    header, rows = None, []
    with open(tsv_path, encoding="utf-8") as fh:
        for raw in fh:
            line = raw.rstrip("\n")
            if line.startswith("#"):
                if (m := re.match(r"#\s*base[^:]*:\s*([\w-]+)", line)) and not base_v:   base_v = m.group(1)
                if (m := re.match(r"#\s*target:\s*([\w-]+)", line))      and not target_v: target_v = m.group(1)
                continue
            if not line.strip():
                continue
            cells = line.split("\t")
            if header is None:
                header = cells; continue
            rows.append(dict(zip(header, cells)))
    _lst = lambda c: [x.strip() for x in (c or "").split(",") if x.strip()]
    remap, last_s = {}, None
    for r in rows:
        base = _lst(r.get("base")); tgt = _lst(r.get("target"))
        s = int(base[0]) if base and base[0].isdigit() else None   # first baseline line; None for target_only
        eff = s if s is not None else last_s                       # target_only -> preceding baseline line
        for b in tgt:
            remap[b] = eff
        if s is not None:
            last_s = s
    return base_v, target_v, remap

def build_milestone_remap(xml_path):
    """For a translation/edition that carries embedded baseline (Storr) card
    milestones <milestone unit='card' edRef='Storr' n='S'/>, map each of its own
    <l n='G'> lines to the Storr line S of the most recent such milestone. This
    bins the text onto Storr's cards by editorial milestone rather than by its
    own (independent) line numbers. Returns {G(str): S(int)}; lines before the
    first card milestone are left unmapped (they fall back to their own number)."""
    root = find_text_root(safe_parse(xml_path).getroot())
    remap, cur_s = {}, None
    for elem in root.iter():
        tag = elem.tag.split('}')[-1]
        if tag == 'milestone' and elem.get('unit') == 'card':
            nv = (elem.get('n') or '').strip()
            if nv.isdigit():
                cur_s = int(nv)
        elif tag == 'l':
            gl = (elem.get('n') or '').strip()
            if gl and cur_s is not None:
                remap[gl] = cur_s
    return remap

def parse_poetry_cards_tei(path, master_intervals, lineno_sigil=None, line_remap=None):
    if not os.path.exists(path): return None
    tree = safe_parse(path)
    text_entry = find_text_root(tree.getroot())

    data = OrderedDict((bk, OrderedDict()) for bk in master_intervals)
    flat_intervals, card_n_to_label = [], {}
    for bk, ivs in master_intervals.items():
        for iv in ivs:
            flat_intervals.append(iv)
            card_n_to_label[(bk, iv["card_n"])] = iv["label"]

    # If this edition carries Storr card milestones, anchor on them and ignore
    # its own <l n> entirely (same treatment the Greek baseline gets).
    has_card_milestones = any(
        e.tag.split("}")[-1] == "milestone" and e.get("unit") == "card"
        for e in text_entry.iter())

    current_book = next(iter(master_intervals))
    cur_label = None
    buffer_map = {}

    def _add_content(bk, label, html):
        if html and html.strip():
            buffer_map.setdefault((bk, label), []).append(html)

    def _find_interval_for_line(bk, line_num):
        if not str(line_num).isdigit(): return None
        ln = int(line_num)
        for iv in flat_intervals:
            if iv["book"] == bk and iv["start_line"] <= ln <= iv["end_line"]: return iv
        for iv in flat_intervals:
            if iv["book"] == bk and iv["start_line"] <= ln: return iv
        return flat_intervals[0]

    def _get_first_line_num(node):
        if node.tag.split("}")[-1] == "l": return node.get("n")
        for ch in node.iter():
            if ch.tag.split("}")[-1] == "l": return ch.get("n")
        return None

    def _walk(elem):
        nonlocal current_book, cur_label
        tag = elem.tag.split("}")[-1]

        # Some editions (e.g. Evelyn-White's 1914 Theogony translation) embed
        # the card milestone AS THE FIRST CHILD of <l> rather than as a
        # preceding sibling (contrast the Greek baseline, which always puts
        # it before <l>). Because <l>/<stage> are leaves below — their full
        # text is pulled in one shot via extract_text_recursive and we never
        # recurse into them with _walk — a milestone nested this way is
        # never otherwise seen, so the card boundary silently never advances
        # and the whole text collapses into the first card. Check for it here.
        if tag in ("l", "stage"):
            for desc in elem.iter():
                if desc is elem:
                    continue
                if desc.tag.split("}")[-1] == "milestone" and desc.get("unit") == "card":
                    n_val = (desc.get("n") or "").strip()
                    if n_val and (current_book, n_val) in card_n_to_label:
                        cur_label = card_n_to_label[(current_book, n_val)]
                    break  # only the leading milestone (if any) should matter

        if tag == "div":
            subtype = (elem.get("subtype") or elem.get("type") or "").lower()
            n_val = (elem.get("n") or "").strip()
            if subtype == "book" and n_val:
                current_book = n_val
            elif subtype == "card" and n_val and (current_book, n_val) in card_n_to_label:
                cur_label = card_n_to_label[(current_book, n_val)]
        elif tag == "milestone" and elem.get("unit") == "card":
            n_val = (elem.get("n") or "").strip()
            if n_val and (current_book, n_val) in card_n_to_label:
                cur_label = card_n_to_label[(current_book, n_val)]
            # else: a milestone whose n is not a master card -> leave cur_label,
            #       its text folds into the current card (and is worth flagging).

        # Line-number fallback ONLY for editions with no milestones at all.
        if not has_card_milestones:
            lh = _get_first_line_num(elem)
            if lh:
                lookup = line_remap.get(str(lh), lh) if line_remap else lh
                iv = _find_interval_for_line(current_book, lookup)
                if iv:
                    cur_label = iv["label"]; current_book = iv["book"]

        label = cur_label
        if not label:
            bk_ivs = master_intervals.get(current_book, [])
            label = bk_ivs[0]["label"] if bk_ivs else "1"

        if tag == "milestone" and elem.get("unit") == "line":
            ln_num = (elem.get("n") or "").strip()
            if ln_num:
                _add_content(current_book, label,
                    f'<span class="inline-line-milestone" title="Line Milestone {ln_num}">{ln_num}</span>')



        if tag in ("l", "stage"):
            _add_content(current_book, label,
                         extract_text_recursive(elem, strip_paragraphs=True, lineno_sigil=lineno_sigil))
            return
        elif tag == "p":
            has_verse_children = any(
                c.tag.split("}")[-1] in ("l", "milestone") for c in elem)
            if has_verse_children:
                if elem.text and elem.text.strip():
                    _add_content(current_book, label, elem.text)
                for ch in elem:
                    _walk(ch)
                if elem.tail:
                    _add_content(current_book, label, elem.tail)
                return
            t = extract_text_recursive(elem, strip_paragraphs=True).strip()
            if t: _add_content(current_book, label, f"<p>{t}</p>")
            return
        elif tag == "note":
            t = extract_text_recursive(elem, strip_paragraphs=True).strip()
            if t: _add_content(current_book, label, f'<span class="note">[{t}]</span>')
            return

        if elem.text and elem.text.strip():
            _add_content(current_book, label, elem.text)
        for ch in elem:
            _walk(ch)
        if elem.tail:
            _add_content(current_book, label, elem.tail)

    _walk(text_entry)
    for (bk, label), snips in buffer_map.items():
        combined = " ".join(t.strip() for t in snips if t.strip())
        if combined:
            data.setdefault(bk, OrderedDict())[label] = {"1": combined}
    return data

    """Prose translation/edition with NO <l> tags, segmented only by embedded
    baseline card milestones <milestone unit='card' edRef='Storr' n='S'/> that
    commonly sit INSIDE <p>/<s>. Streams content in document order and bins each
    run into the Storr card named by the most recent card milestone, so output is
    keyed by Storr card label and aligns to the grid. <s> sentences are rendered
    atomically (balanced markup) and a card milestone that opens a sentence
    switches the card before that sentence is binned."""
def parse_card_prose_tei(path, master_intervals, lineno_sigil=None):
    if not os.path.exists(path): return None
    root = find_text_root(safe_parse(path).getroot())
    if root is None: return None
    
    data = OrderedDict()
    for bk in master_intervals: data[bk] = OrderedDict()
    
    card_n_to_label, first_label = {}, None
    for bk, ivs in master_intervals.items():
        for iv in ivs:
            card_n_to_label[(bk, iv["card_n"])] = iv["label"]
            if first_label is None: first_label = iv["label"]
            
    current_book = next(iter(master_intervals))
    cur_label = first_label
    buf = []

    def flush():
        nonlocal buf
        html = " ".join(t.strip() for t in buf if t and t.strip())
        html = re.sub(r'[ \t\r\n]{2,}', ' ', html)
        if html:
            d = data.setdefault(current_book, OrderedDict())
            # Wrap chunks in a proper prose-para block to relieve crowding
            formatted_html = f'<div class="prose-para">{html}</div>'
            if cur_label in d and "1" in d[cur_label]:
                d[cur_label]["1"] += " " + formatted_html
            else:
                d.setdefault(cur_label, OrderedDict())["1"] = formatted_html
        buf = []

    def emit(s):
        if s and s.strip(): buf.append(s)

    def walk(elem):
        nonlocal cur_label, current_book
        tag = elem.tag.split('}')[-1]
        
        if tag == 'div':
            st = (elem.get('subtype') or elem.get('type') or '').lower()
            if st == 'book' and elem.get('n'):
                flush()
                current_book = elem.get('n').strip()
                _ivs = master_intervals.get(current_book)        # ← add
                if _ivs:                                         # ← add
                    cur_label = _ivs[0]["label"]                 # ← add  (kill stale carry-over)
        elif tag == 'milestone':
            u = elem.get('unit') or ''
            nv = (elem.get('n') or '').strip()
            if u == 'card':
                if nv and (current_book, nv) in card_n_to_label:
                    flush()
                    cur_label = card_n_to_label[(current_book, nv)]
            elif u == 'line' and nv:
                emit(f'<span class="inline-line-milestone" title="Line Milestone {nv}">{nv}</span>')
            elif u == 'page' and nv:
                rp = (elem.get('resp') or '').strip()
                emit(f'<span class="milestone bekker-page" data-resp=\"{rp}\" title=\"{(rp + chr(32) + nv).strip()}\">{nv}</span>')

        elif tag == 'speaker':
            nm = (elem.text or '').strip()
            if nm: 
                flush() # Isolate block context when a new character speaks
                emit(f'<strong class="speaker-attr">{nm}: </strong>')
            # Skip treating inner children as text nodes to prevent duplication
            if elem.tail and elem.tail.strip():
                emit(elem.tail)
            return

        elif tag == 'stage':
            emit(f' <span class="stage-direction">({extract_text_recursive(elem, strip_paragraphs=True).strip()})</span> ')
            if elem.tail and elem.tail.strip(): emit(elem.tail)
            return
        elif tag == 'foreign':
            lang = (elem.get('{http://www.w3.org/XML/1998/namespace}lang') or '').strip()
            cls  = f'foreign foreign-{lang}' if lang else 'foreign'
            emit(f'<span class="{cls}" lang="{lang}">{"".join(elem.itertext())}</span>')
            if elem.tail and elem.tail.strip(): emit(elem.tail)
            return

        elif tag == 'seg':
            rend = (elem.get('rend') or '').strip()
            cls  = f'seg seg-{rend}' if rend else 'seg'
            emit(f'<span class="{cls}">{"".join(elem.itertext())}</span>')
            if elem.tail and elem.tail.strip(): emit(elem.tail)
            return
        elif tag == 'choice':
            sic_txt = corr_txt = ''
            for ch in elem:
                ct = ch.tag.split('}')[-1]
                txt = ''.join(ch.itertext()).strip()
                if ct == 'sic': sic_txt = txt
                elif ct == 'corr': corr_txt = txt
            emit(f'<span class="tei-sic">[{sic_txt}]</span> <span class="tei-corr">{corr_txt}</span>')
            if elem.tail and elem.tail.strip(): emit(elem.tail)
            return

        if elem.text and elem.text.strip(): 
            emit(elem.text)
            
        for ch in elem: 
            walk(ch)
            
        if elem.tail and elem.tail.strip(): 
            emit(elem.tail)

    walk(root)
    flush()
    return data

def parse_milestone_tei(path, milestone_unit="bekker", lineno_sigil=None):
    """Bin running prose by embedded <milestone unit='...' n='CHAPTER.SECTION'/>
    anchors. For translations (e.g. Twining's Poetics) whose own div hierarchy
    (Part/Section) does NOT match the canonical chapter:section citation scheme,
    but which carry inline milestone anchors keyed to it. The run of text following
    a milestone n='10.1' is binned into data[book]['10']['1'] until the next anchor,
    so the column aligns to the baseline chapter:section grid. Text before the first
    anchor (part/section heads) is dropped, matching the other parsers."""
    if not os.path.exists(path): return None
    root = find_text_root(safe_parse(path).getroot())
    if root is None: return None

    data = OrderedDict()
    current_book = "1"
    cur_ch, cur_sec = None, None
    buf = []

    def flush():
        nonlocal buf
        if cur_ch is None or cur_sec is None:
            buf = []          # unanchored preamble (heads etc.) -> discard
            return
        html = " ".join(t.strip() for t in buf if t and t.strip())
        html = re.sub(r'[ \t\r\n]{2,}', ' ', html).strip()
        if html:
            d = data.setdefault(current_book, OrderedDict())
            ch = d.setdefault(cur_ch, OrderedDict())
            ch[cur_sec] = (ch[cur_sec] + " " + html) if cur_sec in ch else html
        buf = []

    def emit(s):
        if s and s.strip(): buf.append(s)

    def walk(elem):
        nonlocal cur_ch, cur_sec, current_book
        tag = elem.tag.split('}')[-1]

        if tag == 'head':
            return            # never emit head text (matches parse_hierarchical_tei)

        if tag == 'div':
            st = (elem.get('subtype') or elem.get('type') or '').lower()
            if st == 'book' and elem.get('n'):
                flush(); current_book = elem.get('n').strip()

        elif tag == 'milestone':
            u = elem.get('unit') or ''
            nv = (elem.get('n') or '').strip()
            if u == milestone_unit and '.' in nv:
                flush()
                c, s = nv.split('.', 1)
                cur_ch, cur_sec = c.strip(), s.strip()
            elif u == 'page' and nv:
                rp = (elem.get('resp') or '').strip()
                emit(f'<span class="milestone bekker-page" data-resp="{rp}" title="{(rp + chr(32) + nv).strip()}">{nv}</span>')
            # other milestone units (line, etc.): no display, no split
            if elem.tail and elem.tail.strip(): emit(elem.tail)
            return

        elif tag == 'hi':
            emit(extract_text_recursive(elem, strip_paragraphs=True))
            if elem.tail and elem.tail.strip(): emit(elem.tail)
            return

        elif tag == 'note':
            note_text = extract_text_recursive(elem, strip_paragraphs=True).strip()
            if note_text: emit(f'<span class="note">[{note_text}]</span>')
            if elem.tail and elem.tail.strip(): emit(elem.tail)
            return

        if elem.text and elem.text.strip():
            emit(elem.text)
        for ch in elem:
            walk(ch)
        if elem.tail and elem.tail.strip():
            emit(elem.tail)

    walk(root)
    flush()
    return data


_RTL_LANG_CODES = {
    'ar', 'ara',            # Arabic
    'fa', 'per', 'fas',     # Persian/Farsi
    'he', 'heb',            # Hebrew
    'ur', 'urd',            # Urdu
    'ae', 'ave',            # Avestan
    'syc', 'syr',           # Syriac
}

def _bidi_dir_for_lang(lang_code):
    """Map an xml:lang value to 'ltr'/'rtl', defaulting to 'ltr' for anything
    not recognized as an RTL script (covers 2- and 3-letter codes, ignoring
    region subtags like 'fa-IR')."""
    if not lang_code:
        return None
    base = lang_code.strip().split('-')[0].lower()
    return 'rtl' if base in _RTL_LANG_CODES else 'ltr'

def _reading_prose_block(div_el, css_class):
    """Render an editorial prose child of a <div subtype="reading"> (intro,
    notes, ...) into one HTML block.

    Walks DIRECT children in document order so <quote> blocks interleaved
    between <p>s (a sibling of <p>, not nested inside one) aren't skipped —
    an earlier version only collected <p> via findall and silently dropped
    any <quote>. <head> becomes a heading; <quote> is handed to
    extract_text_recursive(strip_paragraphs=False) so its own nested <p> gets
    the existing .quote-block treatment (already styled in styles.css); any
    other block-level tag falls back to the same generic extraction rather
    than being dropped.

    Tags the wrapper with dir="ltr"/"rtl" + lang="..." from @xml:lang, since
    these are typically editorial English prose sitting inside a Persian
    (RTL) edition file and would otherwise silently inherit direction:rtl
    from the .persian-text column wrapper they render inside.
    """
    parts = []
    for child in list(div_el):
        ctag = child.tag.split('}')[-1]
        if ctag == 'head':
            if child.text and child.text.strip():
                parts.append(f'<h4 class="{css_class}-head">{child.text.strip()}</h4>')
        elif ctag == 'p':
            t = extract_text_recursive(child, strip_paragraphs=True).strip()
            if t:
                parts.append(f'<div class="prose-para">{t}</div>')
        elif ctag == 'quote':
            q = extract_text_recursive(child, strip_paragraphs=False).strip()
            if q:
                parts.append(q)
        else:
            t = extract_text_recursive(child, strip_paragraphs=False).strip()
            if t:
                parts.append(f'<div class="prose-para">{t}</div>')
    if not parts:
        return None
    lang = (div_el.get('{http://www.w3.org/XML/1998/namespace}lang') or '').strip()
    bidi = _bidi_dir_for_lang(lang)
    attrs = f' dir="{bidi}"' if bidi else ''
    attrs += f' lang="{lang}"' if lang else ''
    return f'<div class="{css_class}"{attrs}>{"".join(parts)}</div>'

def parse_reading_lines_tei(path):
    """Pizzi Shahnameh 'reading' selections.

    Each <div subtype="reading" n="N"> may hold an optional <div subtype="intro">
    (prose head + paragraphs/quotes introducing the passage) and/or a trailing
    <div subtype="notes"> (grammatical/apparatus notes), alongside the verse.
    The verse itself may live inside its own <div type="textpart" subtype="text">
    (or n="text") wrapper sibling of intro/notes, so <l> lines are no longer
    necessarily DIRECT children of the reading div — they're found recursively
    below so nesting depth and the wrapper's exact attributes don't matter.

    intro is emitted as pseudo-line "0intro" and notes as pseudo-line "notes".
    naturalSectionKeys() in app.js sorts numeric-prefixed keys first (by
    number) and any non-numeric-prefixed key after all of them — so "0intro"
    (numeric prefix 0) sorts ahead of "1", "2", ..., while "notes" (no digit
    prefix at all) sorts after every verse line regardless of the reading's
    line count. Neither block's HTML has a .line-num-cell, so the front end's
    poetry-grid detection falls back to plain prose rendering for both.
    """
    if not os.path.exists(path): return None
    tree = safe_parse(path)
    text_entry = find_text_root(tree.getroot())
    if text_entry is None: return None
    data = OrderedDict({"1": OrderedDict()})   # single pseudo-book -> flat structure

    for div in text_entry.iter():
        tag = div.tag.split('}')[-1]
        if tag == 'div' and (div.get('subtype') == 'reading') and div.get('n'):
            reading_n = div.get('n').strip()
            lines = OrderedDict()

            # Intro / notes: only DIRECT children of this reading (so we don't
            # accidentally pull in a sibling reading's apparatus).
            for child in list(div):
                if child.tag.split('}')[-1] != 'div':
                    continue
                st = (child.get('subtype') or child.get('type') or '').lower()
                if st == 'intro':
                    html = _reading_prose_block(child, 'reading-intro')
                    if html:
                        lines["0intro"] = html
                elif st == 'notes':
                    html = _reading_prose_block(child, 'reading-notes')
                    if html:
                        lines["notes"] = html

            # Verse lines: search recursively — no longer assumed to be direct
            # children, since the intro/notes divs forced them into a nested
            # wrapper.
            for l in div.iter():
                if l.tag.split('}')[-1] != 'l':
                    continue
                ln = l.get('n')
                if ln:

                    lines[ln] = extract_text_recursive(l, strip_paragraphs=True)

            data["1"][reading_n] = lines
    return data

def parse_hierarchical_tei(path):
    if not os.path.exists(path): return None
    tree = safe_parse(path)
    text_entry = find_text_root(tree.getroot())
    if text_entry is None: return None
    data = OrderedDict()

    def walk_divisions(node, current_path):
        tag = node.tag.split('}')[-1]
        subtype = node.get('subtype') or node.get('type')
        n_val = node.get('n')
        base_val = node.get('xml:base') or node.get('{http://www.w3.org/XML/1998/namespace}base')
        
        if tag == 'div' and subtype in ('book', 'chapter', 'section', 'subchapter', 'part', 'textpart') and n_val:
            if subtype in ('part', 'textpart') and n_val.isdigit():
                resolved_subtype = 'chapter'
            elif subtype == 'subchapter':
                resolved_subtype = 'section'
            else:
                resolved_subtype = subtype
            
            if base_val and ':' in base_val:
                ch_from_base = base_val.split(':')[-1].strip()
                new_path = current_path + [('chapter', ch_from_base), (resolved_subtype, str(n_val).strip())]
            else:
                new_path = current_path + [(resolved_subtype, str(n_val).strip())]
        else:
            if base_val and ':' in base_val:
                ch_from_base = base_val.split(':')[-1].strip()
                new_path = current_path + [('chapter', ch_from_base)]
            else:
                new_path = current_path

        paragraphs = node.findall('{http://www.tei-c.org/ns/1.0}p') or node.findall('p')
        if paragraphs and new_path:
            key_map = {t: v for t, v in new_path}
            bk = key_map.get('book', '1')
            ch = key_map.get('chapter', key_map.get('part', '1'))
            sec = key_map.get('section', n_val or '1')

            if bk not in data: data[bk] = OrderedDict()
            if ch not in data[bk]: data[bk][ch] = OrderedDict()
            
            combined_txt = ' '.join(extract_text_recursive(p, strip_paragraphs=False).strip() for p in paragraphs)
            if combined_txt:
                if sec in data[bk][ch]:
                    data[bk][ch][sec] += " " + combined_txt
                else:
                    data[bk][ch][sec] = combined_txt

        for child in node: 
            walk_divisions(child, new_path)

    walk_divisions(text_entry, [])
    return data


def parse_line_commentary_tei(path, master_intervals, lineno_sigil=None):
    """Line-keyed verse commentary (e.g. Jebb on Sophocles).

    Bins every comment into the SAME canonical card as the verse baseline, using
    a running *line anchor* drawn (in priority order) from a div's numeric @n
    (commlines carry the exact verse line number) or, failing that, the start
    line of the nearest section's @corresp range. Heads and paragraphs are
    rendered wherever they occur, which tolerates the file's mixed intro
    wrappers (some intros are <div subtype="commline" n="introduction">, others
    <div subtype="section" n="introduction">).

    Every commline (and every @corresp-anchored section) now also emits a
    visible ".comm-lineno" label — e.g. "1" or "4-6" — as the first thing in
    its block, so the rendered commentary column shows which verse line(s)
    each cluster of notes covers. This is purely a display marker; it does not
    affect binning (binning still uses the *first* numeral of the anchor, via
    _interval_for_line).

    Returns data[bk][card_label] = {"1": combined_html}, matching
    parse_poetry_cards_tei so it drops straight onto the alignment grid.
    """
    if not os.path.exists(path):
        return None
    if not master_intervals:
        print("  \u26a0 line_commentary needs a poetry verse edition in the same "
              "work (no master_intervals built) \u2014 skipping.")
        return None
    tree = safe_parse(path)
    text_entry = find_text_root(tree.getroot())
    if text_entry is None:
        return None

    data = OrderedDict((bk, OrderedDict()) for bk in master_intervals)
    flat_intervals = [iv for ivs in master_intervals.values() for iv in ivs]

    def _interval_for_line(bk, line_num):
        if not str(line_num).isdigit():
            return None
        ln = int(line_num)
        for iv in flat_intervals:
            if iv["book"] == bk and iv["start_line"] <= ln <= iv["end_line"]:
                return iv
        cand = [iv for iv in flat_intervals if iv["book"] == bk and iv["start_line"] <= ln]
        return cand[-1] if cand else (flat_intervals[0] if flat_intervals else None)

    default_bk = next(iter(master_intervals))
    buffer_map = {}

    def _emit(bk, anchor, html):
        if not (html and html.strip()):
            return
        iv = _interval_for_line(bk, anchor) if anchor else None
        if iv is None:
            iv = flat_intervals[0] if flat_intervals else None
        label = iv["label"] if iv else "1"
        ebk = iv["book"] if iv else bk
        buffer_map.setdefault((ebk, label), []).append(html)

    def _render_p(p):
        """One <p>: lemma (<mentioned>) + commentary <note>, or plain prose."""
        # NB: the module-level NS is a dict ({'tei': ...}); use the brace-prefixed
        # literal here, matching the other parsers. childless elems are falsy, so
        # test "is None", never "or".
        lemma_el = p.find("{http://www.tei-c.org/ns/1.0}mentioned")
        if lemma_el is None:
            lemma_el = p.find("mentioned")
        note_el = None
        for gc in p.iter():
            if gc is p:
                continue
            if gc.tag.split("}")[-1] == "note" and gc.get("type") == "commentary":
                note_el = gc
                break
        lemma_html = ""
        if lemma_el is not None:
            lt = extract_text_recursive(lemma_el, strip_paragraphs=True).strip()
            if lt:
                lemma_html = f'<span class="lemma">{lt}</span> '   # same class <s> uses, inherits .commentary-text .lemma
        if note_el is not None:
            body = extract_text_recursive(note_el, strip_paragraphs=True).strip()
            body = body.lstrip(". \u00a0").strip()  # notes often open with a stray "."
        else:
            body = extract_text_recursive(p, strip_paragraphs=True).strip()
        if not (lemma_html or body):
            return ""
        return f'<div class="comm-entry">{lemma_html}<span class="comm-note">{body}</span></div>'

    # Leading digit-run of an anchor string, e.g. "4" -> "4", "4_6" -> "4",
    # "117-253" -> "117". Used to decide which card a block belongs to; the
    # FULL raw label (with underscores normalized to a dash) is kept separately
    # for display so a range like "4_6" still reads as "4-6" on screen.
    _LEAD_NUM_RE = re.compile(r"(\d+)")

    def _display_label(raw):
        return raw.replace("_", "-").strip()

    cur_bk = default_bk
    cur_anchor = None          # running verse-line anchor (string of digits), used for binning
    cur_anchor_display = None  # human-facing label for the current anchor, used for the visible badge
    last_emitted_display = None  # avoids repeating the same lineno badge for every sibling <p>

    def _maybe_emit_lineno_badge():
        nonlocal last_emitted_display
        if cur_anchor_display and cur_anchor_display != last_emitted_display:
            _emit(cur_bk, cur_anchor, f'<div class="comm-lineno">{cur_anchor_display}</div>')
            last_emitted_display = cur_anchor_display

    def _walk(elem):
        nonlocal cur_bk, cur_anchor, cur_anchor_display
        tag = elem.tag.split("}")[-1]
        if tag == "div":
            subtype = (elem.get("subtype") or elem.get("type") or "").lower()
            n_val = (elem.get("n") or "").strip()
            if subtype == "book" and n_val:
                cur_bk = n_val
            else:
                corresp = elem.get("corresp") or ""
                if ":" in corresp:
                    rng = corresp.rsplit(":", 1)[-1].strip()
                    m = _LEAD_NUM_RE.match(rng)
                    if m:
                        cur_anchor = m.group(1)         # section line-range start, for binning
                        cur_anchor_display = _display_label(rng)   # full range, for the badge
                elif subtype == "commline" and n_val:
                    m = _LEAD_NUM_RE.match(n_val)
                    if m:
                        cur_anchor = m.group(1)          # commline @n IS the verse line (first numeral if a range)
                        cur_anchor_display = _display_label(n_val)
                # NB: a section's @n is an ordinal (1,2,3...), never a line number.
            for ch in elem:
                _walk(ch)
            return
        if tag == "head":
            h = extract_text_recursive(elem, strip_paragraphs=True).strip()
            if h:
                m = re.match(r"\s*(\d+(?:[-\u2013]\d+)?)", h)     # heads like "117-253: Parodos"
                if m:
                    cur_anchor_display = _display_label(m.group(1))
                    cur_anchor = _LEAD_NUM_RE.match(m.group(1)).group(1)
                _maybe_emit_lineno_badge()
                _emit(cur_bk, cur_anchor, f'<div class="comm-head">{h}</div>')
            return
        if tag == "p":
            _maybe_emit_lineno_badge()
            _emit(cur_bk, cur_anchor, _render_p(elem))
            return
        for ch in elem:
            _walk(ch)

    _walk(text_entry)

    for (bk, label), snips in buffer_map.items():
        combined = "".join(snips)
        if combined.strip():
            data.setdefault(bk, OrderedDict())[label] = {"1": combined}
    return data

def parse_conllu_treebank(path, version_short_id, tg, wk, card_intervals=None):
    paths = [path] if isinstance(path, str) else list(path)
    missing = [p for p in paths if not os.path.exists(p)]
    if missing:
        for p in missing:
            print(f"  ✗ Treebank not found: {p}")
        return [], {'annotators': []}

    line_to_card = {}
    if card_intervals:
        for idx, interval in enumerate(card_intervals):
            try:
                bk    = str(interval['book'])
                start = int(interval['label'].split('-')[0])
                end   = int(interval['label'].split('-')[1])
                for ln in range(start, end + 1):
                    # Key as BOOK.LINE (multi-book) and bare LINE (flat works)
                    line_to_card[f"{bk}.{ln}"] = interval['label']
                    line_to_card[str(ln)] = interval['label']
            except (ValueError, KeyError):
                continue

    def _derive_prose_chapter_section(ref, sid=None):
        """For prose (no card_intervals) book.chapter.section addressing
        (e.g. Thucydides "1.89.3"). treebank_sentences has no separate book
        column, and the client (app.js's _hydrateTreebank) groups sentences
        purely by this chapter string, so collapsing to the bare book
        number merges every chapter of a book together -- and even a bare
        chapter number collides across different books (book 2 chapter 1
        is not book 3 chapter 1). Fold book into the chapter key itself
        ("book.chapter", e.g. "1.89") to keep it globally unique; app.js's
        lookup constructs this same compound key from payload.book/chapter.
        The real section (the passage/sentence number) is the LAST segment,
        not the middle one -- a 2-level "book.section" ref (no chapter
        subdivision) doesn't have a middle level at all.
        """
        parts = ref.split('.')
        if len(parts) >= 3:
            return f"{parts[0]}.{parts[1]}", '.'.join(parts[2:])
        elif len(parts) == 2:
            return f"{parts[0]}.1", parts[1]
        else:
            return f"{parts[0]}.1", (str(sid) if sid else '1')

    def _looks_like_head(v):
        """Valid CoNLL-U HEAD values: a non-negative integer, or '_'. Used to
        detect a blank LEMMA field that whitespace-splitting has silently
        swallowed (some source files leave LEMMA blank by deliberate
        annotator choice for certain forms -- not an error -- but a truly
        empty field disappears rather than surviving as '' once a line is
        whitespace- rather than tab-split, shifting every field after it one
        position to the left; that shift makes something clearly non-head-
        shaped -- e.g. a deprel string -- turn up where HEAD should be)."""
        return v == '_' or v.isdigit()

    def _lookup_card(ref):
        """Resolve a Ref/subdoc line value to its card label. Handles the
        plain-integer case directly, and falls back to the leading digits
        for interpolated/lettered line variants (e.g. Evelyn-White's Theogony
        929a-929t insertion) that sit between two canonical integer lines but
        were never themselves enumerated when line_to_card was built."""
        if ref in line_to_card:
            return line_to_card[ref]
        m = re.match(r'^(\d+)', ref)
        if m and m.group(1) in line_to_card:
            return line_to_card[m.group(1)]
        return None

    # ── Document-level annotator roster ─────────────────────────────────
    # Two header conventions are supported:
    #   1. Tagged roster (one line per annotator, resolved by short code from
    #      a per-sentence "# sentannotators" line):
    #        # annotator <short>millermo</short> <name>Molly Miller</name> <address>Tufts University, Medford, MA, USA</address>
    #   2. Simple form (the common case — one flat credit for the whole
    #      document, no per-sentence variation):
    #        # annotator = Molly Miller, Tufts University, Medford, MA, USA
    # doc_roster maps short-code -> {name, address} (tagged form only).
    # doc_annotators_ordered preserves file order and is used as the
    # document-level fallback when a sentence has no "# sentannotators" line.
    doc_roster = OrderedDict()
    doc_annotators_ordered = []
    ANNOTATOR_TAGGED_RE = re.compile(
        r'^#\s*annotator\s+<short>(.*?)</short>\s*<name>(.*?)</name>\s*<address>(.*?)</address>\s*$'
    )
    ANNOTATOR_SIMPLE_RE = re.compile(r'^#\s*annotator\s*=\s*(.+)$')
    SENTANNOTATORS_RE   = re.compile(r'<(primary|secondary)>(.*?)</\1>')
    SOURCE_RE           = re.compile(r'^#\s*source\s*[:=]?\s*(.+)$', re.IGNORECASE)
    doc_source = None

    sentences = []
    cur = None

    def flush():
        if not cur or not cur.get('tokens'):
            return
        sent = dict(cur)
        # Resolve empty-node HEAD references (ellipsis/gapping): a token
        # whose basic HEAD pointed at an empty node's decimal id (e.g.
        # "10.1") gets reattached to that empty node's own real governor,
        # which we only learn once the whole sentence has been read (the
        # empty node's row can come before or after the token referencing
        # it). Anything unresolved (malformed/missing empty-node row) falls
        # back to root (0) rather than crashing.
        empty_heads = sent.pop('empty_heads', None) or {}
        for tok in sent['tokens']:
            ref = tok.pop('head_empty_ref', None)
            if ref is not None:
                tok['head'] = empty_heads.get(ref, 0)
        # Resolve this sentence's credits: prefer its own "sentannotators"
        # refs (short codes tagged primary/secondary), resolved against the
        # tagged roster; fall back to the whole-document roster/simple list
        # when the sentence didn't specify its own.
        refs = sent.pop('credits_refs', None)
        if refs:
            resolved = []
            for role, short in refs:
                entry = doc_roster.get(short)
                if entry:
                    resolved.append({'name': entry['name'], 'address': entry['address'], 'role': role})
                else:
                    resolved.append({'name': short, 'address': None, 'role': role})
            sent['credits'] = resolved
        else:
            sent['credits'] = None
        if sent.get('subdoc') is None:
            first_ref = None
            for tok in sent['tokens']:
                if tok.get('ref'):
                    first_ref = tok['ref']
                    break
            if first_ref is not None:
                sent['subdoc'] = first_ref
                if card_intervals:
                    # Ref is BOOK.LINE — look up directly, fall back to book
                    book_part = first_ref.split('.')[0] if '.' in first_ref else first_ref
                    sent['chapter'] = _lookup_card(first_ref) or book_part
                else:
                    sent['chapter'], sent['section'] = _derive_prose_chapter_section(first_ref, sent.get('sent_id'))


        else:
            # subdoc came from a "# subdoc =" header; chapter/section were already
            # set when the header was parsed. Only poetry works need a card remap.
            if card_intervals and sent.get('subdoc'):
                subdoc_ref = sent['subdoc']
                book_part = subdoc_ref.split('.')[0] if '.' in subdoc_ref else subdoc_ref
                # Try BOOK.LINE key first, then bare LINE (flat works), fall back to book
                chapter = _lookup_card(subdoc_ref)
                if chapter is None and '.' not in subdoc_ref:
                    chapter = _lookup_card(f'1.{subdoc_ref}')
                sent['chapter'] = chapter or book_part
                if not sent.get('section'):
                    sent['section'] = str(sent.get('sent_id', '1'))
            else:
                # prose work, no card intervals: recompute chapter/section
                # correctly rather than trusting the header's values, since
                # the header-parsing site below has this exact same
                # book.chapter.section derivation and needs the identical
                # fix -- keeping stale/mis-derived header values here would
                # just perpetuate whatever it got wrong.
                sent['chapter'], sent['section'] = _derive_prose_chapter_section(sent['subdoc'], sent.get('sent_id'))
                
        sentences.append(sent)

    for path in paths:
      with open(path, 'r', encoding='utf-8') as fh:
        for raw in fh:
            line = raw.rstrip('\n')

            is_sent_id   = line.startswith('# sentence_id')
            is_sent_id_s = line.startswith('# sent_id')
            if is_sent_id or is_sent_id_s:
                flush()
                sid = line.split('=', 1)[1].strip() if '=' in line else None
                cur = {'tokens': [], 'subdoc': None, 'chapter': None,
                       'section': None, 'prose': None, 'literal': None,
                       'translit': None, 'sent_id': sid, 'credits_refs': None,
                       'empty_heads': {}}
                continue

            if line.startswith('#'):
                # Document header lines (before the first sent_id) still need
                # to be scanned for the annotator roster/source, even though
                # there's no "cur" sentence yet to attach them to.
                m_tagged = ANNOTATOR_TAGGED_RE.match(line)
                if m_tagged:
                    short, name, address = m_tagged.group(1).strip(), m_tagged.group(2).strip(), m_tagged.group(3).strip()
                    entry = {'name': name, 'address': address}
                    doc_roster[short] = entry
                    doc_annotators_ordered.append(entry)
                elif ANNOTATOR_SIMPLE_RE.match(line):
                    name = ANNOTATOR_SIMPLE_RE.match(line).group(1).strip()
                    doc_annotators_ordered.append({'name': name, 'address': None})
                m_source = SOURCE_RE.match(line)
                if m_source and doc_source is None:
                    doc_source = m_source.group(1).strip()

                if cur is None:
                    continue

                m_sentann = re.match(r'^#\s*sentannotators\b(.*)$', line)
                if m_sentann:
                    refs = SENTANNOTATORS_RE.findall(m_sentann.group(1))
                    if refs:
                        cur['credits_refs'] = refs
                    continue

                def mval(key, ln=line):
                    m = re.match(rf'^#\s*{key}\s*=\s*(.+)', ln)
                    return m.group(1).strip() if m else None
                v = mval('subdoc')
                if v:
                    cur['subdoc'] = v
                    cur['chapter'], cur['section'] = _derive_prose_chapter_section(v, cur.get('sent_id'))
                p = mval('prose_translation')
                if p: cur['prose'] = p
                lt = mval('literal_translation')
                if lt: cur['literal'] = lt
                tr = mval('transliteration')
                if tr: cur['translit'] = tr
                continue

            if cur is None:
                continue

            if line == '':
                flush()
                cur = None
                continue

            # Standard CoNLL-U is tab-separated, but some source files (seen
            # in the wild: a large stretch of Aeschylus's Prometheus Bound)
            # use space-padded columns instead of real tabs -- visually
            # aligned, but silently unparseable by a strict tab-split, which
            # would otherwise drop every one of those sentences with zero
            # tokens and no error. Try tabs first (this also preserves the
            # existing colon-typo'd-as-tab tolerance for empty nodes below,
            # which depends on the tab-split column layout); only fall back
            # to whitespace-splitting when tab-splitting clearly failed.
            cols = line.split('\t')
            if len(cols) < 8:
                ws_cols = line.split()
                if len(ws_cols) >= 8:
                    cols = ws_cols
            if len(cols) < 8:
                continue
            id_str = cols[0]
            if '-' in id_str:
                continue  # multiword token range row: no annotation of its own
            if re.match(r'^\d+\.$', id_str):
                # A stray period glued onto the token id with nothing after
                # the dot (seen in the wild when the token's own FORM is
                # itself "." and an export step merged the two with no
                # space, e.g. "13.\t.\tPUNCT..."). NOT a real empty node --
                # those are always "N.M" with digits on both sides of the
                # dot. Strip the glued-on dot and fall through to ordinary
                # token handling below instead of misreading it as ellipsis.
                id_str = id_str[:-1]
            elif '.' in id_str:
                # Empty node (CoNLL-U's standard way of encoding ellipsis/
                # gapping — a word position with no surface form). It carries
                # no token of its own, but real tokens can point to it as
                # their HEAD (see below), so remember its own governor,
                # normally packed as "HEAD:DEPREL" in the DEPS field (index 8).
                # Tolerate the corrupted variant where that colon was typo'd
                # as a tab, splitting "8:acl:compl" into cols[8]="8" and an
                # extra cols[9]="acl:compl" (bumping MISC to cols[10]).
                deps_field = cols[8] if len(cols) > 8 else '_'
                head_part = deps_field.split(':', 1)[0].strip()
                if not head_part.isdigit() and len(cols) > 9 and cols[8].strip().isdigit():
                    head_part = cols[8].strip()
                if head_part.isdigit():
                    cur.setdefault('empty_heads', {})[id_str] = int(head_part)
                continue

            # Realign LEMMA/UPOS/XPOS/FEATS/HEAD/DEPREL/DEPS/MISC from cols[2:],
            # auto-detecting a blank LEMMA (see _looks_like_head above) by
            # checking whether the position HEAD should occupy actually holds
            # a valid head value; if not, retry on the assumption LEMMA was
            # blank and everything after it shifted one column left.
            rest = cols[2:]
            if len(rest) >= 5 and _looks_like_head(rest[4]):
                lemma = rest[0]
                upos, xpos, feats, head_val = rest[1], rest[2], rest[3], rest[4]
                tail = rest[5:]
            elif len(rest) >= 4 and _looks_like_head(rest[3]):
                lemma = ''
                upos, xpos, feats, head_val = rest[0], rest[1], rest[2], rest[3]
                tail = rest[4:]
            else:
                # Neither alignment yields a valid head -- fall back to the
                # naive (lemma-present) reading rather than guessing further.
                lemma = rest[0] if rest else ''
                upos  = rest[1] if len(rest) > 1 else '_'
                xpos  = rest[2] if len(rest) > 2 else '_'
                feats = rest[3] if len(rest) > 3 else '_'
                head_val = rest[4] if len(rest) > 4 else '0'
                tail = rest[5:]
            deprel = tail[0] if len(tail) > 0 else '_'
            # tail[1], if present, is DEPS -- not stored on the token (only
            # used above for empty-node governor resolution).
            misc = tail[2] if len(tail) > 2 else '_'
            # Some source files pack gloss into MISC ("Ref=1|gloss=word");
            # others give gloss its own trailing column instead of using
            # MISC's pipe-joined convention for it at all. Support both:
            # prefer MISC's own gloss= if present, else the trailing column.
            gloss_trailing = tail[3] if len(tail) > 3 else None
            gloss = None
            ref   = None
            translit  = None
            ltranslit = None
            for kv in misc.split('|'):
                k2, _, v2 = kv.partition('=')
                if k2 == 'gloss':     gloss     = v2.strip()
                if k2 == 'Ref':       ref       = v2.strip()
                if k2 == 'Translit':  translit  = v2.strip()
                if k2 == 'LTranslit': ltranslit = v2.strip()
            if gloss is None and gloss_trailing:
                gloss = gloss_trailing.strip()

            # A real token's basic HEAD is normally an integer (or '_'/0 for
            # root). Some Daphne-annotated ellipsis constructions instead
            # point a token straight at an empty node (a decimal id like
            # "10.1"). We can't resolve that until the whole sentence (and
            # that empty node's own row, which may appear later) is read, so
            # stash the raw reference now and resolve it in flush() below.
            head_is_empty_ref = '.' in head_val
            cur['tokens'].append({
                'id':     int(id_str),
                'form':   cols[1],
                'lemma':  lemma,
                'upos':   upos,
                'xpos':   xpos,
                'feats':  feats,
                'head':   0 if head_is_empty_ref else (int(head_val) if head_val not in ('_',) else 0),
                'head_empty_ref': head_val if head_is_empty_ref else None,
                'deprel': deprel,
                'gloss':  gloss,
                'ref':    ref,
                'translit':  translit,
                'ltranslit': ltranslit,
            })

    flush()
    # Document-level credits: the tagged roster (if any annotator lines used
    # that form) plus any simple-form lines, in file order. This is the
    # fallback shown for sentences that don't carry their own sentannotators,
    # and is also stored once per version for the "common case" documents
    # that never use per-sentence credits at all.
    doc_credits = {
        'annotators': doc_annotators_ordered,
        'source': doc_source,
    }
    return sentences, doc_credits


AGDT_POS_MAP = {
    'n': 'NOUN', 'v': 'VERB', 't': 'VERB',
    'a': 'ADJ', 'd': 'ADV', 'l': 'DET', 'g': 'PART',
    'r': 'ADP', 'p': 'PRON', 'm': 'NUM', 'c': 'CCONJ',
    'i': 'INTJ', 'e': 'INTJ', 'u': 'PUNCT',
}

def _agdt_upos(postag):
    """First letter of an AGDT 9-position postag -> a UD-style UPOS label,
    so app.js's PUNCT filtering and TB_POS_COLORS coloring work unchanged.
    Anything unrecognized (including '-' for fragmentary/untagged lyric text
    and 'x' for indeclinables) falls through to 'X'."""
    if not postag:
        return None
    return AGDT_POS_MAP.get(postag[0], 'X')

def _ref_from_cite(cite):
    """'urn:cts:greekLit:tlg0085.tlg001:1' -> '1'"""
    if not cite:
        return None
    tail = cite.rsplit(':', 1)[-1]
    return tail or None


def parse_agdt_treebank(path, version_short_id, tg, wk, card_intervals=None):
    """Parse a Perseus/AGDT Prague-XML treebank file (<sentence subdoc=...>
    <word .../></sentence>) into the same (sentences, doc_credits) shape
    produced by parse_conllu_treebank, so everything downstream (treebank_
    sentences ingestion, Cell 1b's flatten step, app.js rendering) is
    format-agnostic and needs no changes for this input type.

    Field mapping from the Prague-XML attributes to the shared schema:
      relation -> deprel   cite -> ref   postag[0] -> upos (see AGDT_POS_MAP)
      postag (whole)  -> xpos
      translation attribute (the gloss-composed literal rendering already
      verified sentence-by-sentence to be self-consistent) -> literal
    """
    if not os.path.exists(path):
        print(f"  ✗ Treebank not found: {path}")
        return [], {'annotators': []}

    try:
        tree = ET.parse(path)
    except ET.ParseError as e:
        print(f"  ✗ XML parse error in {path}: {e}")
        return [], {'annotators': []}
    root = tree.getroot()

    # Document-level annotator credits from the header's respStmt blocks.
    # Cosmetic metadata only -- falls back to an empty list if the header
    # is absent or shaped differently.
    doc_annotators = []
    seen_names = set()
    for resp_stmt in root.iter('respStmt'):
        name_el = resp_stmt.find('persName')
        name = None
        if name_el is not None:
            name = (name_el.text or '').strip()
            if not name:
                n_el = name_el.find('n')
                if n_el is not None:
                    name = (n_el.text or '').strip()
        resp_el = resp_stmt.find('resp')
        resp = (resp_el.text or '').strip() if resp_el is not None else None
        if name and name not in seen_names:
            seen_names.add(name)
            doc_annotators.append({'name': name, 'address': resp})

    line_to_card = {}
    if card_intervals:
        for interval in card_intervals:
            try:
                bk = str(interval['book'])
                start = int(interval['label'].split('-')[0])
                end = int(interval['label'].split('-')[1])
                for ln in range(start, end + 1):
                    line_to_card[f"{bk}.{ln}"] = interval['label']
                    line_to_card[str(ln)] = interval['label']
            except (ValueError, KeyError):
                continue

    def _lookup_card(ref):
        if ref in line_to_card:
            return line_to_card[ref]
        m = re.match(r'^(\d+)', ref)
        if m and m.group(1) in line_to_card:
            return line_to_card[m.group(1)]
        return None

    sentences = []
    for sent_el in root.iter('sentence'):
        subdoc = sent_el.get('subdoc')
        sid = sent_el.get('id')
        translation = sent_el.get('translation')  # gloss-composed literal translation
        annotator_el = sent_el.find('annotator')
        annotator_name = (annotator_el.text or '').strip() if annotator_el is not None else None

        tokens = []
        for w in sent_el.findall('word'):
            wid = w.get('id')
            if wid is None or not wid.isdigit():
                continue
            postag = w.get('postag') or ''
            head_val = w.get('head') or '0'
            tokens.append({
                'id': int(wid),
                'form': w.get('form') or '',
                'lemma': w.get('lemma'),
                'upos': _agdt_upos(postag),
                'xpos': postag or None,
                'feats': None,
                'head': int(head_val) if head_val.isdigit() else 0,
                'deprel': w.get('relation'),
                'gloss': w.get('gloss'),
                'ref': _ref_from_cite(w.get('cite')),
                'translit': None,
                'ltranslit': None,
            })

        if not tokens or not subdoc:
            continue

        first_ref = subdoc.split('-')[0]
        ref_parts = first_ref.split('.')
        book_part = ref_parts[0]
        if card_intervals:
            chapter = _lookup_card(first_ref) or book_part
            section = str(sid) if sid else '1'
        else:
            # Prose, book.chapter.section addressing (e.g. Thucydides
            # "1.89.3"). treebank_sentences has no separate book column, and
            # the client (app.js's _hydrateTreebank) groups sentences purely
            # by this chapter string -- so collapsing to book_part alone
            # would merge every chapter of a book under one key, and even
            # a bare chapter number would collide across different books
            # (book 2 chapter 1 vs book 3 chapter 1 are NOT the same
            # chapter). Encode book into the chapter key itself ("book.chapter",
            # e.g. "1.89") to keep it globally unique -- app.js's lookup
            # constructs this same compound key from payload.book/chapter.
            # The genuine section (the passage/sentence number) is the LAST
            # segment, not ref_parts[1] -- a 2-level "book.section" ref (no
            # chapter subdivision) doesn't have a middle level at all.
            if len(ref_parts) >= 3:
                chapter = f"{ref_parts[0]}.{ref_parts[1]}"
                section = '.'.join(ref_parts[2:])
            elif len(ref_parts) == 2:
                chapter = f"{ref_parts[0]}.1"
                section = ref_parts[1]
            else:
                chapter = f"{book_part}.1"
                section = str(sid) if sid else '1'

        credits = [{'name': annotator_name, 'address': None, 'role': 'primary'}] if annotator_name else None

        sentences.append({
            'subdoc': subdoc,
            'chapter': chapter,
            'section': section,
            'tokens': tokens,
            'prose': None,
            'literal': translation,
            'translit': None,
            'sent_id': sid,
            'credits': credits,
        })

    doc_credits = {'annotators': doc_annotators, 'source': None}
    return sentences, doc_credits

def parse_speakers_csv(path):
    import csv
    speakers = {}
    if not path or not os.path.exists(path):
        return speakers
    with open(path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            subdoc  = (row.get('subdoc') or row.get('sentence_id') or row.get('id') or '').strip()
            speaker = (row.get('speaker') or row.get('Speaker') or '').strip()
            if subdoc and speaker:
                speakers[subdoc] = speaker
    return speakers

# ── Processing & Database Compilations ────────────────────────
conn = init_storage_engine(DB_PATH)
cursor = conn.cursor()
global_sort_index = 0

def parse_metrical_tsv(path):
    """Parse a tab-separated metrical annotation file.

    Each row: URN_REF<TAB>WORD<TAB>START<TAB>END<TAB>SCANSION<TAB>HEMI[<TAB>TAG...]
    Returns a dict: { "BOOK.LINE": [ {word, start, end, hemi, syls, tags}, ...] }
    where syls = [ {q: "long"|"short"|"onset", text: str}, ...]
    and mora positions 1-24 encode metrical weight (long=2, short=1).
    """
    import re
    if not os.path.exists(path):
        print(f"  ✗ Metrical file not found: {path}")
        return {}
    lines_data = {}
    with open(path, 'r', encoding='utf-8') as fh:
        for raw in fh:
            raw = raw.rstrip('\n')
            if not raw or raw.startswith('#'):
                continue
            parts = raw.split('\t')
            if len(parts) < 6:
                continue
            urn_ref  = parts[0]   # e.g. tlg001:1.1
            word     = parts[1]
            try:
                start = int(parts[2]); end = int(parts[3])
            except ValueError:
                continue
            scansion = parts[4]
            hemi     = parts[5]
            tags     = [t for t in parts[6:] if t]

            # Extract BOOK.LINE reference
            ref = urn_ref.split(':')[1] if ':' in urn_ref else urn_ref

            # Parse syllables from scansion string
            syls = []
            for tok in re.split(r'(?=(?:long|short)-)', scansion):
                if not tok:
                    continue
                if tok.startswith('long-'):
                    syls.append({'q': 'long',   'text': tok[5:].rstrip('-')})
                elif tok.startswith('short-'):
                    syls.append({'q': 'short',  'text': tok[6:].rstrip('-')})
                else:
                    syls.append({'q': 'onset',  'text': tok.rstrip('-')})

            word_obj = {'word': word, 'start': start, 'end': end,
                        'hemi': hemi, 'syls': syls, 'tags': tags}
            lines_data.setdefault(ref, []).append(word_obj)
    return lines_data


# ─────────────────────────────────────────────────────────────────────────────
# REGISTERING A LINE-KEYED VERSE COMMENTARY (e.g. Jebb on Sophocles)
# ─────────────────────────────────────────────────────────────────────────────
# A line commentary aligns by VERSE LINE, not by its own section ordinals, so it
# only works inside a work that ALSO carries a poetry verse edition (parse_mode
# "poetry_cards") — that edition builds master_intervals, and the commentary is
# binned into those exact cards by its commline @n / section @corresp.
#
# NOTE: viaf2603144.viaf002.perseus-eng1 is Jebb on Sophocles' OEDIPUS AT COLONUS
#       (tlg0011.tlg007), NOT Antigone (tlg0011.tlg002). Register it under the OC
#       work, alongside the OC Greek verse text. Example:
#
# WORK_REGISTRY["tlg0011.tlg007"] = {
#     "textgroup": "tlg0011",
#     "work": "tlg007",
#     "editions": {
#         "perseus-grc2": {   # the verse baseline that defines the card grid
#             "path": "/Users/gcrane/github/canonical-greekLit/data/tlg0011/tlg007/tlg0011.tlg007.perseus-grc2.xml",
#             "label": "Greek (Storr/baseline)", "class": "greek-text",
#             "parse_mode": "poetry_cards",
#         },
#     },
#     "appcrits": {},
#     "translations": {},
#     "commentaries": {
#         "jebb1899-com-eng1": {
#             "path": "/Users/gcrane/github/.../viaf2603144_viaf002_perseus-eng1.xml",
#             "label": "Commentary (R. C. Jebb, 1899)",
#             "class": "commentary-text",
#             "parse_mode": "line_commentary",   # <- routes to parse_line_commentary_tei
#         },
#     },
# }

_pending_metrical = {}
for work_key, work_meta in WORK_REGISTRY.items():
    tg = work_meta["textgroup"]
    wk = work_meta["work"]
    
    editions_combined = {}
    doc_types_map = {}
    
    for category in ["editions", "appcrits", "translations", "commentaries"]:
        category_dict = work_meta.get(category, {})
        editions_combined.update(category_dict)
        for v_id in category_dict:
            doc_types_map[v_id] = {"editions": "edition", "appcrits": "appcrit", "translations": "translation", "commentaries": "commentary"}[category]

    for v_id, cfg in work_meta.get("treebanks", {}).items():
        canonical_id = f"{tg}_{wk}_{v_id}_treebank"
        cursor.execute("""
            INSERT OR REPLACE INTO text_units (canonical_id, urn, label, text_class, textgroup, work, short_id, doc_type)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?);
        """, (canonical_id, f"urn:cts:{_ns(work_key)}:{work_key}.{v_id}",
              cfg["label"], cfg["class"], tg, wk, v_id, "treebank"))

    # ── Ingest metrical annotation files ──────────────────────────────
    for v_id, cfg in work_meta.get("metrics", {}).items():
        canonical_id = f"{tg}_{wk}_{v_id}_metrical"
        cursor.execute("""
            INSERT OR REPLACE INTO text_units (canonical_id, urn, label, text_class, textgroup, work, short_id, doc_type)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?);
        """, (canonical_id, f"urn:cts:{_ns(work_key)}:{work_key}.{v_id}",
              cfg["label"], cfg["class"], tg, wk, v_id, "metrical"))
        lines_data = parse_metrical_tsv(cfg["path"])
        if not lines_data:
            print(f"  ⚠ No metrical data loaded for {v_id}")
            continue
        # Map line_ref -> chapter using card intervals (built later);
        # store temporarily and re-ingest after master_intervals are built.
        # For now store lines keyed by line_ref; chapter assigned below.
        _pending_metrical[work_key] = _pending_metrical.get(work_key, {})
        _pending_metrical[work_key][v_id] = {"cfg": cfg, "lines_data": lines_data,
                                              "canonical_id": canonical_id}
        print(f"  ✓ Loaded {len(lines_data)} metrical lines for {v_id}")

    print(f"Ingesting structural alignment mappings for work: {work_key}...")
    
    for v_id, cfg in editions_combined.items():
        doc_type = doc_types_map[v_id]
        canonical_id = generate_canonical_id(work_key, cfg["label"], doc_type)
        cursor.execute("""
            INSERT OR REPLACE INTO text_units (canonical_id, urn, label, text_class, textgroup, work, short_id, doc_type)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?);
        """, (canonical_id, f"urn:cts:{_ns(work_key)}:{work_key}.{v_id}", cfg["label"], cfg["class"], tg, wk, v_id, doc_type))

    master_intervals = None
    is_poetry = any(cfg.get("parse_mode") in ("poetry_cards", "card_prose") for cfg in editions_combined.values())
    if is_poetry:
        master_intervals = build_poetry_canonical_intervals(editions_combined)

    # Card structure is built from the baseline edition (perseus-grc2 = Storr).
    # For every edition-alignment crosswalk whose base axis IS that baseline,
    # build a {target_line: baseline_line} remap so the target edition is binned
    # by its true Storr counterpart line (split/merge -> first Storr line of the
    # group; target_only -> preceding Storr line). Display keeps the native @n.
    baseline_vid = ("perseus-grc2" if "perseus-grc2" in editions_combined
                    else (next(iter(editions_combined)) if editions_combined else None))
    edition_remaps = {}
    for _tsv in work_meta.get("edition_alignments", []):
        if not os.path.exists(_tsv):
            continue
        _bv, _tv, _map = build_line_remap(_tsv)
        if _tv:
            edition_remaps[_tv] = {"base": _bv, "map": _map}
            print(f"  \u21b3 line remap {_tv}\u2192{_bv}: {len(_map)} lines (baseline={baseline_vid})")

    work_corpus = OrderedDict()
    for v_id, cfg in editions_combined.items():
        p_mode = cfg.get("parse_mode")
        if p_mode == "poetry_cards":
            if cfg.get("card_anchor") == "milestones":
                # Carries embedded Storr card milestones: bin by those, ignoring
                # its own independent line numbers (which remain display-only).
                _rm = build_milestone_remap(cfg["path"])
            else:
                _info = edition_remaps.get(v_id)
                _rm = _info["map"] if (_info and _info["base"] == baseline_vid) else None
            parsed = parse_poetry_cards_tei(cfg["path"], master_intervals,
                                            lineno_sigil=cfg.get("lineno_sigil"),
                                            line_remap=_rm)
        elif p_mode == "card_prose":
            # Prose (no <l>) segmented only by embedded Storr card milestones.
            parsed = parse_card_prose_tei(cfg["path"], master_intervals,
                                          lineno_sigil=cfg.get("lineno_sigil"))
        elif p_mode == "line_commentary":
            # Line-keyed verse commentary (e.g. Jebb): bins each note into the
            # SAME canonical card as the verse baseline via its commline @n /
            # section @corresp. Requires a poetry verse edition in this work so
            # master_intervals exists; returns None (skipped) if it does not.
            parsed = parse_line_commentary_tei(cfg["path"], master_intervals,
                                               lineno_sigil=cfg.get("lineno_sigil"))
        elif p_mode == "milestones":
            # Translation whose own div hierarchy (e.g. Twining's Part/Section) does
            # NOT match the canonical chapter:section scheme, but which carries inline
            # <milestone unit='bekker' n='CHAPTER.SECTION'/> anchors. Bin running prose
            # by those so it aligns to the baseline chapter:section grid.
            parsed = parse_milestone_tei(cfg["path"],
                                         milestone_unit=cfg.get("milestone_unit", "bekker"),
                                         lineno_sigil=cfg.get("lineno_sigil"))
        elif p_mode == "reading_lines":
            parsed = parse_reading_lines_tei(cfg["path"])
        else:
            parsed = parse_hierarchical_tei(cfg["path"])
            
        if parsed is not None and sum(len(secs) for chs in parsed.values() for secs in chs.values()) > 0:
            work_corpus[v_id] = parsed
            print(f"  ✓ {v_id}: {sum(len(secs) for chs in parsed.values() for secs in chs.values())} segments parsed")
        else:
            print(f"  ✗ {v_id}: Failed to parse completely.")

    if not work_corpus: continue

    first_version = list(work_corpus.keys())[0]
    baseline_corpus = work_corpus[first_version]
    has_multiple_books = len(baseline_corpus.keys()) > 1

    chapter_sequence = []
    for b_k, ch_v in baseline_corpus.items():
        for c_k in ch_v.keys():
            chapter_sequence.append({'book': b_k if has_multiple_books else None, 'chapter': c_k})

    for c_idx, coord in enumerate(chapter_sequence):
        bk_id = coord['book']
        ch_id = coord['chapter']
        lookup_bk = bk_id if bk_id else list(baseline_corpus.keys())[0]
        baseline_secs = list(baseline_corpus[lookup_bk][ch_id].keys())
        
        for sec in baseline_secs:
            global_sort_index += 1
            if bk_id:
                passage_urn = f"urn:cts:{_ns(work_key)}:{work_key}:{bk_id}.{ch_id}.{sec}"
                prev_urn = f"urn:cts:{_ns(work_key)}:{work_key}:{chapter_sequence[c_idx-1]['book']}.{chapter_sequence[c_idx-1]['chapter']}.{sec}" if c_idx > 0 else None
                next_urn = f"urn:cts:{_ns(work_key)}:{work_key}:{chapter_sequence[c_idx+1]['book']}.{chapter_sequence[c_idx+1]['chapter']}.{sec}" if c_idx < len(chapter_sequence) - 1 else None
            else:
                passage_urn = f"urn:cts:{_ns(work_key)}:{work_key}:{ch_id}.{sec}"
                prev_urn = f"urn:cts:{_ns(work_key)}:{work_key}:{chapter_sequence[c_idx-1]['chapter']}.{sec}" if c_idx > 0 else None
                next_urn = f"urn:cts:{_ns(work_key)}:{work_key}:{chapter_sequence[c_idx+1]['chapter']}.{sec}" if c_idx < len(chapter_sequence) - 1 else None
            
            cursor.execute("""
                INSERT OR REPLACE INTO alignment_grid 
                (passage_urn, textgroup, work, book, chapter, section, prev_urn, next_urn, sort_order)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?);
            """, (passage_urn, tg, wk, bk_id, ch_id, sec, prev_urn, next_urn, global_sort_index))
            
            for v_id in editions_combined:
                ch_data = work_corpus.get(v_id, {}).get(lookup_bk, {}).get(ch_id, {})
                raw_html = ch_data.get(sec)
                if raw_html is None:
                    raw_html = ch_data.get("1", "[Text range missing in alignment layer]")                
                cursor.execute("""
                    INSERT OR REPLACE INTO text_segments (passage_urn, version_short_id, content_html)
                    VALUES (?, ?, ?);
                """, (passage_urn, v_id, raw_html))

conn.commit()

# ── Ingest treebank sentences and metrical lines ──────────────────────────────
print("\nIngesting treebank and metrical data...")
for work_key, work_meta in WORK_REGISTRY.items():
    tg = work_meta["textgroup"]
    wk = work_meta["work"]

    # Build card intervals once per work (needed by both treebank and metrical)
    tb_card_intervals = None
    if any(c.get("parse_mode") == "poetry_cards"
           for c in work_meta.get("editions", {}).values()):
        try:
            tb_card_intervals = build_poetry_canonical_intervals(work_meta["editions"])
            tb_card_intervals = [iv for bk in tb_card_intervals.values() for iv in bk]
            print(f"  ↳ {work_key}: {len(tb_card_intervals)} card intervals")
        except Exception as e:
            print(f"  ⚠ Could not build card intervals for {work_key}: {e}")

    # ── Treebank sentences ────────────────────────────────────────────
    for v_id, cfg in work_meta.get("treebanks", {}).items():
        p_mode = cfg.get("parse_mode")
        if p_mode == "conllu":
            sentences, doc_credits = parse_conllu_treebank(cfg["path"], v_id, tg, wk,
                                              card_intervals=tb_card_intervals)
        elif p_mode == "agdt_xml":
            sentences, doc_credits = parse_agdt_treebank(cfg["path"], v_id, tg, wk,
                                              card_intervals=tb_card_intervals)
        else:
            print(f"  ✗ {v_id}: unsupported treebank parse_mode {p_mode!r}")
            continue
        speakers  = parse_speakers_csv(cfg.get("speakers_csv", ""))
        cursor.execute("""
            INSERT OR REPLACE INTO treebank_doc_credits
            (textgroup, work, version_short_id, source_repo, credits_json)
            VALUES (?, ?, ?, ?, ?)
        """, (tg, wk, v_id, cfg.get("source_repo"),
              json.dumps(doc_credits.get('annotators', []), ensure_ascii=False)))
        for sent in sentences:
            if not sent.get('subdoc'): continue
            cursor.execute("""
                INSERT INTO treebank_sentences
                (textgroup, work, version_short_id, subdoc, chapter, section,
                 sentence_json, prose_translation, literal_translation, transliteration,
                 credits_json)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (tg, wk, v_id, sent['subdoc'], sent['chapter'], sent.get('section') or '1',
                  json.dumps(sent['tokens'], ensure_ascii=False),
                  sent.get('prose'), sent.get('literal'), sent.get('translit'),
                  json.dumps(sent['credits'], ensure_ascii=False) if sent.get('credits') else None))
            if sent['subdoc'] in speakers:
                cursor.execute("""
                    INSERT INTO treebank_speakers (textgroup, work, subdoc, speaker)
                    VALUES (?, ?, ?, ?)
                """, (tg, wk, sent['subdoc'], speakers[sent['subdoc']]))
        print(f"  ✓ {v_id}: {len(sentences)} sentences ingested"
              + (f" ({len(doc_credits.get('annotators', []))} doc-level annotator(s))"
                 if doc_credits.get('annotators') else ""))

    # ── Metrical lines ────────────────────────────────────────────────
    if work_key in _pending_metrical:
        if not tb_card_intervals:
            print(f"  ⚠ No card intervals for {work_key} — metrical chapters will use book number")
        _m_line_to_card = {}
        if tb_card_intervals:
            for _iv in tb_card_intervals:
                try:
                    _bk = str(_iv['book'])
                    _s  = int(_iv['label'].split('-')[0])
                    _e  = int(_iv['label'].split('-')[1])
                    for _ln in range(_s, _e + 1):
                        _m_line_to_card[f"{_bk}.{_ln}"] = _iv['label']
                except (ValueError, KeyError):
                    continue
        for v_id, entry in _pending_metrical[work_key].items():
            rows_inserted = 0
            for line_ref, words in entry["lines_data"].items():
                chapter = _m_line_to_card.get(line_ref, line_ref.split('.')[0])
                cursor.execute(
                    "INSERT INTO metrical_lines "
                    "(textgroup, work, version_short_id, line_ref, chapter, line_json) "
                    "VALUES (?, ?, ?, ?, ?, ?)",
                    (tg, wk, v_id, line_ref, chapter, json.dumps(words, ensure_ascii=False))
                )
                rows_inserted += 1
            print(f"  ✓ {v_id}: {rows_inserted} metrical lines inserted")
conn.commit()

# ── Ingest token alignment files ──────────────────────────────────────────
print("\nIngesting token alignment files...")
for work_key, work_meta in WORK_REGISTRY.items():
    tg = work_meta["textgroup"]
    wk = work_meta["work"]
    for pair_id, aln_cfg in work_meta.get("alignments", {}).items():
        aln_path = aln_cfg["path"]
        if not os.path.exists(aln_path):
            print(f"  ✗ {pair_id}: file not found at {aln_path}")
            continue
        with open(aln_path, encoding="utf-8") as f:
            aln_data = json.load(f)
        meta     = aln_data.get("alignment_meta", {})
        segments = aln_data.get("segments", {})
        src_ver  = aln_cfg["src_version"]
        tgt_ver  = aln_cfg["tgt_version"]
        rows = 0
        for seg_id, seg in segments.items():
            for grp in seg.get("alignments", []):
                cursor.execute("""
                    INSERT INTO token_alignments
                    (textgroup, work, pair_id, src_version, tgt_version,
                     segment, src_indices, tgt_indices, src_tokens, tgt_tokens, score)
                    VALUES (?,?,?,?,?,?,?,?,?,?,?)
                """, (
                    tg, wk, pair_id, src_ver, tgt_ver, seg_id,
                    json.dumps(grp["src_indices"]),
                    json.dumps(grp["tgt_indices"]),
                    json.dumps(grp.get("src_tokens", [])),
                    json.dumps(grp.get("tgt_tokens", [])),
                    grp["score"]
                ))
                rows += 1
        conn.commit()
        print(f"  ✓ {pair_id}: {len(segments)} segments, {rows} alignment groups ingested")

# ── Ingest edition line-alignment crosswalks ─────────────────────────
print("\nIngesting edition line-alignment files...")
for work_key, work_meta in WORK_REGISTRY.items():
    for tsv in work_meta.get("edition_alignments", []):
        if not os.path.exists(tsv):
            print(f"  ✗ edition alignment: file not found at {tsv}")
            continue
        ingest_edition_alignment(conn, tsv)

cursor.execute("VACUUM;")
conn.close()

print(f"\n[SUCCESS] Serverless Relational Storage Asset compiled cleanly at: {DB_PATH}")

Ingesting structural alignment mappings for work: tlg0003.tlg001...
  ✓ perseus-grc2: 3587 segments parsed
  ✓ 1st1K-eng1: 3580 segments parsed
  ✓ 1st1K-eng2: 3580 segments parsed
  ✓ perseus-eng4: 3590 segments parsed
  ✓ perseus-eng6: 3587 segments parsed
  ✓ 1st1k-fre1: 1364 segments parsed
  ✓ 1st1K-fre2: 917 segments parsed
  ✓ 1st1K-ger1: 59 segments parsed
  ✓ 1st1K-ger2: 930 segments parsed
  ✓ 1st1K-ger3: 923 segments parsed
  ✓ 1st1K-ger4: 50 segments parsed
  ✓ 1st1K-ita1: 918 segments parsed
  ✓ 1st1k-lat2: 3624 segments parsed
  ✓ Loaded 15682 metrical lines for perstb-meter-grc1
Ingesting structural alignment mappings for work: tlg0012.tlg001...
  ✓ perseus-grc2: 425 segments parsed
  ✓ perseus-eng3: 423 segments parsed
  ✓ butlernagy2020-eng2: 425 segments parsed
  ✓ bryant1870-eng2: 448 segments parsed
  ✓ pope1720watson1857-eng2: 448 segments parsed
  ✓ hobbes1667molesworth-eng2: 447 segments parsed
  ✓ chapman1611hooper-eng2: 448 segments parsed
  ✓ cowper-eng2: 448 

In [2]:
# ── CELL 1b: Flatten treebank_sentences -> treebank_tokens ─────────────────
# Explodes each sentence's `sentence_json` (a JSON array of token dicts) into
# one row per token in a new `treebank_tokens` table, so the search app can
# do `WHERE lemma_norm=?` / `WHERE feats LIKE '%Mood=Opt%'` directly in SQL
# instead of deserializing every sentence's JSON in the browser.
#
# Run this AFTER the monolith is built (the cell above) and BEFORE Cell 2
# (sharding). It rides Cell 2's existing generic per-work copy loop for free
# -- the ONLY other change needed is adding "treebank_tokens" to Cell 2's
# TABLES list (done below in this notebook already).

import sqlite3, json as _json, unicodedata
from pathlib import Path

BUILD_DIR = Path("/tmp/persvers_build")   # must match Cell 1/Cell 0 above
DB_PATH = BUILD_DIR / "corpus_alignment_grid.db"


def norm_key(s):
    """Accent/diacritic- and case-insensitive search key.

    Strips Unicode combining marks (category Mn) -- this covers Greek
    polytonic accents/breathings/iota subscript AND Arabic tashkeel in one
    pass -- then folds Greek final sigma and lowercases. Plain Latin/Persian
    strings pass through unchanged (aside from lowercasing).
    """
    if not s:
        return None
    t = unicodedata.normalize('NFD', s)
    t = ''.join(ch for ch in t if unicodedata.category(ch) != 'Mn')
    t = unicodedata.normalize('NFC', t).lower()
    t = t.replace('\u03c2', '\u03c3')  # final sigma -> medial sigma
    return t


conn = sqlite3.connect(str(DB_PATH))
cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS treebank_tokens")
cur.execute("""
    CREATE TABLE treebank_tokens (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        textgroup TEXT NOT NULL,
        work TEXT NOT NULL,
        version_short_id TEXT NOT NULL,
        sentence_id INTEGER NOT NULL,
        chapter TEXT NOT NULL,
        section TEXT,
        subdoc TEXT,
        tok_id INTEGER NOT NULL,
        form TEXT NOT NULL,
        form_norm TEXT NOT NULL,
        lemma TEXT,
        lemma_norm TEXT,
        upos TEXT,
        xpos TEXT,
        feats TEXT,
        head INTEGER,
        deprel TEXT,
        gloss TEXT,
        ref TEXT,
        translit TEXT,
        ltranslit TEXT
    )
""")
cur.execute("CREATE INDEX idx_tt_lemma_norm ON treebank_tokens(lemma_norm)")
cur.execute("CREATE INDEX idx_tt_form_norm  ON treebank_tokens(form_norm)")
cur.execute("CREATE INDEX idx_tt_work       ON treebank_tokens(textgroup, work)")
cur.execute("CREATE INDEX idx_tt_upos       ON treebank_tokens(upos)")

rows = cur.execute("""
    SELECT id, textgroup, work, version_short_id, chapter, section, subdoc, sentence_json
    FROM treebank_sentences
""").fetchall()

n_sent, n_tok, n_skipped = 0, 0, 0
for sid, tg, wk, ver, chapter, section, subdoc, sjson in rows:
    n_sent += 1
    try:
        tokens = _json.loads(sjson)  # sentence_json is a JSON ARRAY of token dicts
    except (TypeError, ValueError):
        n_skipped += 1
        continue
    if not isinstance(tokens, list):
        n_skipped += 1
        continue
    for tok in tokens:
        form = tok.get('form')
        if not form:
            continue
        lemma = tok.get('lemma')
        cur.execute("""
            INSERT INTO treebank_tokens
                (textgroup, work, version_short_id, sentence_id, chapter, section, subdoc,
                 tok_id, form, form_norm, lemma, lemma_norm, upos, xpos, feats,
                 head, deprel, gloss, ref, translit, ltranslit)
            VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
        """, (
            tg, wk, ver, sid, chapter, section, subdoc,
            tok.get('id'), form, norm_key(form), lemma, norm_key(lemma),
            tok.get('upos'), tok.get('xpos'), tok.get('feats'),
            tok.get('head'), tok.get('deprel'), tok.get('gloss'),
            tok.get('ref'), tok.get('translit'), tok.get('ltranslit'),
        ))
        n_tok += 1

conn.commit()
conn.close()
print(f"\u2713 treebank_tokens: {n_tok} tokens from {n_sent} sentences ({n_skipped} sentences skipped/unparseable)")


✓ treebank_tokens: 625584 tokens from 41076 sentences (0 sentences skipped/unparseable)


# Sharded deployment (v37)

**Principle:** the corpus is split into one read-only SQLite **shard per work**, laid
out by CTS author/work structure. The app fetches a shard *whole* (each is small)
and queries it with sql.js. No server, no Range requests, no per-file header tuning —
so it runs identically on GitHub Pages, on Tufts nginx, or on a laptop behind any
static file server. Moving hosts is a copy + (optional) header config.

## Why whole-shard fetch instead of sql.js-httpvfs
Sharding and httpvfs solve the same problem (don't ship 2 GB) two ways: make the unit
small, or page into a big unit. Per-work sharding makes each unit small enough to
fetch whole, so httpvfs's complexity — and the GitHub-Pages gzip-on-HEAD breakage that
forces a manual `fileLength` — simply don't apply. Reserve httpvfs for the *one* thing
sharding can't help: a future corpus-wide search index.

## Layout
```
site/
  catalog.json                                     # work list + PARTS + versions + byte sizes
  data/
    tlg0012/tlg001/tlg0012.tlg001.part1.db         # Homer Iliad, books 1-N (whatever fits)
    tlg0012/tlg001/tlg0012.tlg001.part2.db         # Homer Iliad, books N+1-24
    phi0690/phi003/phi0690.phi003.part1.db         # a smaller work: still just one part
```
Each part has the **identical schema** to the monolith (same tables + indexes); it just
holds a subset of rows for one `(textgroup, work)`. `text_segments` (no tg/wk columns) is
filtered through `alignment_grid`. Every work uses the `partN.db` naming uniformly, even
when it only has one part — one file-naming convention for every consumer to handle,
not two.

### Book-aware auto-splitting (v47)
A work whose shard would exceed ~80MB of estimated content (leaving headroom under
GitHub's **100MB hard per-file limit**) is automatically split into multiple parts along
its **ancient book divisions** — `alignment_grid.book` is the source of truth for where
those divisions fall; every other passage-keyed table is mapped back to a book via its
`chapter` (or `chapter+section`, if `chapter` turns out to be reused across books) rather
than by parsing book numbers out of strings. Works with no book divisions at all (flat
prose) fall back to splitting by contiguous chapter ranges instead. See Cell 2 for the
full algorithm; Cell 2b's chunking guard is part-aware too (it unions chapter counts
across every `part*.db` for a work rather than assuming one file).

Bundling all *versions* of a work across its parts preserves fetch locality for the
parallel multi-column view and the diff — the whole reason to shard per work (not per
edition) in the first place; splitting only kicks in along the book axis when a work's
total size demands it.

Consumers reconstruct "one work = one queryable database" by fetching every part and
merging them in memory (`app.js`'s `getDbForWork` → `loadAndMergeParts`) — this is
intentionally invisible below that point: `treebankForChapter`, `alignmentsForPair`,
`metricalForChapter`, and everything else still just take a single `db` handle.

## Routing (deep-link, no lookup)
A CTS URN's namespace *is* the path. `urn:cts:greekLit:tlg0012.tlg001.grc2:1.10` →
`data/tlg0012/tlg001/tlg0012.tlg001.db`, then query passage `1.10`, version `grc2`.
`catalog.json` is only needed for browse / the version picker, not for resolution.

## Build pipeline
1. Run the existing **Cell 0** to build the monolith as today.
2. Run the **shard splitter** (next cell) — non-destructive: it reads the monolith and
   writes the `site/` tree + `catalog.json`. Verified to copy each work's rows with zero
   foreign-work leakage and all indexes intact.
3. Generate the app (existing app-generator cell), then apply the **four integration swaps**
   listed in the loader cell so the running app reads per-work shards instead of one monolith
   plus inlined JSON.

## Integration swaps (in the app/HTML generator)
The current app reads text from the monolith via SQL **and** reads registry / treebank /
alignments / metrical from JSON baked into the HTML (`*_REPLACE`). Both break at 3,860 works.
The loader cell replaces them: fetch the work's shard, point `window.dbInstance` at it, and
read the former-inlined data from the shard via SQL. Details in that cell.

## What sharding does NOT solve
Corpus-wide search (a lemma across all 3,860 versions) can't fan across thousands of shards.
That needs a separate precomputed inverted index — the one piece that may reintroduce a
backend and a security surface. Decide scope deliberately.


In [3]:
# ── CELL 2: Shard corpus by work (book-aware auto-splitting) ───────────────
#
# Splits each work into one or more book-range "parts" so no single shard
# file exceeds GitHub's 100MB hard per-file limit. Book boundaries come from
# alignment_grid.book (the one table that records book divisions natively);
# every other passage-keyed table is mapped back to a book via its chapter
# (or, if the same chapter string turns out to be reused across more than
# one book in this work, via chapter+section) using a lookup built from
# alignment_grid itself -- never by parsing book numbers out of strings.
#
# Tables that only carry work/edition/pair-level metadata (no passage-level
# book info) are copied WHOLESALE into every part, since a part needs to be
# self-sufficient on its own (e.g. the edition dropdown needs every version
# regardless of which book-range part happens to be loaded).
#
# Every work gets a uniform "parts" list in catalog.json, even ones that
# don't need splitting (a single-element list) -- one shape for consumers
# to handle, not two.

import sqlite3
import json
from pathlib import Path

PART_SIZE_CEILING_BYTES = 60 * 1024 * 1024  # SOFT ceiling used only to produce a
                                              # reasonable first-guess grouping from
                                              # the cheap content-byte estimate below
                                              # -- kept conservative because that
                                              # estimate can undershoot real file size
                                              # substantially at high row counts.
HARD_SIZE_CEILING_BYTES = 95 * 1024 * 1024  # AUTHORITATIVE ceiling, checked against
                                              # each part's ACTUAL written file size
                                              # (see _resolve_under_ceiling) -- this
                                              # is what actually guarantees staying
                                              # under GitHub's 100MB hard cap; the
                                              # soft ceiling above is just a starting
                                              # point to minimize re-splitting work.

# Split by book range -- these tables are keyed at passage/sentence granularity.
PARTITIONED_TABLES = ["alignment_grid", "text_segments", "treebank_sentences",
                       "treebank_tokens", "metrical_lines"]

# Copied in full into every part -- work/edition/pair-level, no per-passage
# book info to split on.
WHOLESALE_TABLES = ["text_units", "treebank_speakers", "token_alignments",
                     "edition_line_alignments"]

ALL_TABLES = PARTITIONED_TABLES + WHOLESALE_TABLES

_HEAVY_COLS = {
    "treebank_sentences": ["sentence_json", "prose_translation",
                            "literal_translation", "transliteration", "credits_json"],
    "treebank_tokens": ["form", "lemma", "feats", "gloss", "translit", "ltranslit"],
    "metrical_lines": ["line_json"],
}


def _table_cols(src, table):
    return [c[1] for c in src.execute(f"PRAGMA table_info({table})").fetchall()]


def _book_lookup(src, tg, wk):
    """Returns (ordered_books, chapter_book, cs_book, ambiguous).
    `chapter_book` (chapter -> book) is ALWAYS built, even when ambiguous --
    it's a best-effort last-write-wins mapping in that case, but it must
    still exist for tables with no `section` column (metrical_lines) that
    have nothing else to look up by. `cs_book` ((chapter, section) -> book)
    is only built when needed (chapter turned out to be reused across more
    than one book), for tables that DO have a section column to disambiguate
    with; it's None when chapter alone was already unambiguous.

    BUG HISTORY: an earlier version returned *either* chapter_book *or*
    cs_book depending on ambiguity, never both. That silently dropped every
    metrical_lines row whenever a work's chapters were ambiguous, since
    metrical_lines deliberately avoids the (chapter, section) lookup (no
    section column) but was then handed nothing else to look up by."""
    rows = src.execute(
        "SELECT book, chapter, section FROM alignment_grid "
        "WHERE textgroup=? AND work=? ORDER BY sort_order", (tg, wk)).fetchall()

    ordered_books, seen = [], set()
    for book, chapter, section in rows:
        if book not in seen:
            seen.add(book)
            ordered_books.append(book)

    chapter_book, ambiguous = {}, False
    for book, chapter, section in rows:
        if chapter in chapter_book and chapter_book[chapter] != book:
            ambiguous = True
        chapter_book[chapter] = book  # last-write-wins; only exact if not ambiguous

    cs_book = {(chapter, section): book for book, chapter, section in rows} if ambiguous else None
    return ordered_books, chapter_book, cs_book, ambiguous


def _estimate_book_sizes(src, tg, wk, ordered_books, chapter_book, cs_book):
    """Rough per-book byte estimate from the heavy TEXT columns -- used only
    to decide where part boundaries fall, not an exact file-size prediction."""
    sizes = {b: 0 for b in ordered_books}
    sizes[None] = 0  # bucket for rows that couldn't be mapped to a book

    def bump(book, n):
        sizes[book] = sizes.get(book, 0) + (n or 0)

    for book, n in src.execute("""
        SELECT ag.book, SUM(LENGTH(ts.content_html))
        FROM text_segments ts JOIN alignment_grid ag ON ts.passage_urn = ag.passage_urn
        WHERE ag.textgroup=? AND ag.work=? GROUP BY ag.book""", (tg, wk)).fetchall():
        bump(book, n)

    for table, cols in _HEAVY_COLS.items():
        cols = [c for c in cols if c in _table_cols(src, table)]
        if not cols:
            continue
        len_expr = " + ".join(f"COALESCE(LENGTH({c}),0)" for c in cols)
        # Use (chapter, section) only for tables that actually HAVE a section
        # column and only when it's needed (cs_book is None otherwise) --
        # metrical_lines never qualifies (no section column), so it always
        # falls back to chapter_book, which is always populated.
        use_cs = cs_book is not None and "section" in _table_cols(src, table)
        key_cols = "chapter, section" if use_cs else "chapter"
        lookup = cs_book if use_cs else chapter_book
        for row in src.execute(
                f"SELECT {key_cols}, SUM({len_expr}) FROM {table} "
                f"WHERE textgroup=? AND work=? GROUP BY {key_cols}", (tg, wk)).fetchall():
            *key, n = row
            k = tuple(key) if use_cs else key[0]
            bump(lookup.get(k), n)

    return sizes


def _estimate_flat_chapter_sizes(src, tg, wk):
    """Fallback for works with no book divisions at all: same idea as
    _estimate_book_sizes, but keyed directly by chapter."""
    sizes = {}

    def bump(chapter, n):
        sizes[chapter] = sizes.get(chapter, 0) + (n or 0)

    for chapter, n in src.execute("""
        SELECT ag.chapter, SUM(LENGTH(ts.content_html))
        FROM text_segments ts JOIN alignment_grid ag ON ts.passage_urn = ag.passage_urn
        WHERE ag.textgroup=? AND ag.work=? GROUP BY ag.chapter""", (tg, wk)).fetchall():
        bump(chapter, n)

    for table, cols in _HEAVY_COLS.items():
        cols = [c for c in cols if c in _table_cols(src, table)]
        if not cols:
            continue
        len_expr = " + ".join(f"COALESCE(LENGTH({c}),0)" for c in cols)
        for chapter, n in src.execute(
                f"SELECT chapter, SUM({len_expr}) FROM {table} "
                f"WHERE textgroup=? AND work=? GROUP BY chapter", (tg, wk)).fetchall():
            bump(chapter, n)

    return sizes


def _bin_into_parts(ordered_keys, sizes, ceiling):
    """Greedily group ordered keys (books, or chapters for the no-book
    fallback) into contiguous parts, each capped at `ceiling` estimated
    bytes."""
    parts, current, current_size = [], [], 0
    for key in ordered_keys:
        k_size = sizes.get(key, 0)
        if current and current_size + k_size > ceiling:
            parts.append(current)
            current, current_size = [], 0
        current.append(key)
        current_size += k_size
    if current:
        parts.append(current)
    return [p for p in parts if p]


def split_corpus_by_work(monolith_path, out_root):
    """Split monolith into per-work, per-book-range shard parts."""
    monolith_path = Path(monolith_path)
    out_root = Path(out_root)
    out_root.mkdir(parents=True, exist_ok=True)

    src = sqlite3.connect(str(monolith_path))
    src.row_factory = sqlite3.Row

    works = [dict(r) for r in src.execute(
        "SELECT DISTINCT textgroup, work FROM text_units ORDER BY textgroup, work")]

    catalog = {"works": {}}

    for tg, wk in [(w["textgroup"], w["work"]) for w in works]:
        work_key = f"{tg}.{wk}"
        print(f"\n[shard] {work_key}")

        work_dir = out_root / "data" / tg / wk
        work_dir.mkdir(parents=True, exist_ok=True)
        for stale in work_dir.glob(f"{work_key}.*"):
            stale.unlink()

        ordered_books, chapter_book, cs_book, ambiguous = _book_lookup(src, tg, wk)
        if ambiguous:
            print(f"  \u26a0 chapter values are reused across books for {work_key}; "
                  f"using chapter+section for tables that have a section column "
                  f"(treebank_sentences, treebank_tokens), and a best-effort "
                  f"last-write-wins chapter-only mapping for metrical_lines, which "
                  f"has no section column to disambiguate with -- worth a manual "
                  f"spot check of metrical_lines placement for this work.")

        if ordered_books == [None]:
            # No book divisions at all (flat prose work) -- fall back to
            # splitting by contiguous chapters (in reading order) instead.
            ordered_keys = [r[0] for r in src.execute(
                "SELECT chapter FROM alignment_grid WHERE textgroup=? AND work=? "
                "GROUP BY chapter ORDER BY MIN(sort_order)", (tg, wk)).fetchall()]
            sizes = _estimate_flat_chapter_sizes(src, tg, wk)
            part_groups = _bin_into_parts(ordered_keys, sizes, PART_SIZE_CEILING_BYTES)
            split_mode = "chapter"
        else:
            sizes = _estimate_book_sizes(src, tg, wk, ordered_books, chapter_book, cs_book)
            part_groups = _bin_into_parts(ordered_books, sizes, PART_SIZE_CEILING_BYTES)
            split_mode = "book"

        if not part_groups:
            print(f"  ! no passages found for {work_key} -- skipping (nothing to shard)")
            continue

        n_parts = len(part_groups)
        if n_parts > 1:
            print(f"  \u2192 splitting into {n_parts} parts by {split_mode} "
                  f"({[len(p) for p in part_groups]} {split_mode}s each)")

        def _write_group_to_path(shard_path, group):
            """Writes one part's data (for `group`, a list of book or
            chapter keys) to `shard_path`. Returns per-table row counts."""
            shard_path.unlink(missing_ok=True)
            dst = sqlite3.connect(str(shard_path))
            dst.execute("PRAGMA synchronous=OFF")

            for table in ALL_TABLES:
                schema = src.execute(
                    "SELECT sql FROM sqlite_master WHERE type='table' AND name=?",
                    (table,)).fetchone()
                if schema and schema[0]:
                    dst.execute(schema[0])
            dst.commit()

            group_set = set(group)
            row_counts = {}
            for table in ALL_TABLES:
                try:
                    col_names = _table_cols(src, table)

                    if table in WHOLESALE_TABLES:
                        if "textgroup" in col_names and "work" in col_names:
                            rows_data = src.execute(
                                f"SELECT * FROM {table} WHERE textgroup=? AND work=?",
                                (tg, wk)).fetchall()
                        else:
                            rows_data = []

                    elif table == "alignment_grid":
                        key_col = "book" if split_mode == "book" else "chapter"
                        placeholders = ", ".join("?" for _ in group)
                        rows_data = src.execute(
                            f"SELECT * FROM alignment_grid WHERE textgroup=? AND work=? "
                            f"AND {key_col} IN ({placeholders})", (tg, wk, *group)).fetchall()

                    elif table == "text_segments":
                        key_col = "book" if split_mode == "book" else "chapter"
                        placeholders = ", ".join("?" for _ in group)
                        rows_data = src.execute(f"""
                            SELECT ts.* FROM text_segments ts
                            JOIN alignment_grid ag ON ts.passage_urn = ag.passage_urn
                            WHERE ag.textgroup=? AND ag.work=? AND ag.{key_col} IN ({placeholders})
                        """, (tg, wk, *group)).fetchall()

                    else:
                        # treebank_sentences / treebank_tokens / metrical_lines --
                        # keyed by chapter(+section); filtered in Python via the
                        # book lookup (or directly by chapter in the flat-work
                        # fallback), since there's no book column to filter on
                        # directly in SQL here.
                        all_rows = src.execute(
                            f"SELECT * FROM {table} WHERE textgroup=? AND work=?",
                            (tg, wk)).fetchall()
                        chapter_idx = col_names.index("chapter")
                        section_idx = col_names.index("section") if "section" in col_names else None
                        use_cs = cs_book is not None and section_idx is not None
                        rows_data = []
                        for row in all_rows:
                            if split_mode == "chapter":
                                key_of_row = row[chapter_idx]
                            elif use_cs:
                                key_of_row = cs_book.get((row[chapter_idx], row[section_idx]))
                            else:
                                key_of_row = chapter_book.get(row[chapter_idx])
                            if key_of_row in group_set:
                                rows_data.append(row)

                    if rows_data:
                        col_list = ", ".join(col_names)
                        placeholders = ", ".join("?" for _ in col_names)
                        insert_sql = f"INSERT INTO {table} ({col_list}) VALUES ({placeholders})"
                        for row in rows_data:
                            dst.execute(insert_sql, row)
                        dst.commit()
                        row_counts[table] = len(rows_data)

                except Exception as e:
                    print(f"  ! {shard_path.name} {table}: {e}")

            dst.close()
            return row_counts

        def _resolve_under_ceiling(group):
            """Writes `group` to a scratch file and checks its REAL size --
            not the estimate -- since the estimate can undershoot badly at
            high row counts (SQLite per-row overhead isn't something a raw
            content-byte sum captures well). If it's over HARD_SIZE_CEILING_BYTES
            and still has more than one book/chapter to split, bisects and
            recurses. Returns a flat list of (group, row_counts) tuples, each
            verified to fit (or a single book/chapter that can't be split
            further, flagged loudly instead of silently shipped oversized)."""
            scratch = work_dir / f"{work_key}.scratch.db"
            row_counts = _write_group_to_path(scratch, group)
            actual_bytes = scratch.stat().st_size
            scratch.unlink(missing_ok=True)

            if actual_bytes <= HARD_SIZE_CEILING_BYTES or len(group) <= 1:
                if actual_bytes > HARD_SIZE_CEILING_BYTES:
                    print(f"  \u26a0 a single {split_mode} ({group[0]}) is "
                          f"{actual_bytes/1e6:.1f}MB on its own -- can't split further "
                          f"at this granularity; will ship oversized.")
                return [(group, row_counts)]

            mid = len(group) // 2
            print(f"  \u21bb estimate undershot: a {len(group)}-{split_mode} group came out "
                  f"{actual_bytes/1e6:.1f}MB (over the {HARD_SIZE_CEILING_BYTES/1e6:.0f}MB hard "
                  f"ceiling) -- bisecting and re-checking")
            return _resolve_under_ceiling(group[:mid]) + _resolve_under_ceiling(group[mid:])

        resolved_groups = []
        for group in part_groups:
            resolved_groups.extend(_resolve_under_ceiling(group))

        parts_meta = []
        for i, (group, _unused_counts) in enumerate(resolved_groups, start=1):
            part_file = f"{work_key}.part{i}.db"
            shard_path = work_dir / part_file
            row_counts = _write_group_to_path(shard_path, group)  # final numbered write
            for table, n in row_counts.items():
                print(f"    part{i} {table}: {n}")

            sz = shard_path.stat().st_size
            flag = "  \u26a0 STILL OVER 100MB" if sz > 100 * 1024 * 1024 else ""
            print(f"  \u2713 part{i}: {sz/1e6:.1f}MB ({split_mode}s {group[0]}\u2026{group[-1]}){flag}")
            parts_meta.append({
                "part": i, "file": part_file,
                split_mode + "s": [str(x) for x in group],
                "bytes": sz,
            })

        # ── Work-level metadata (unchanged in spirit: computed against the
        # FULL work from the monolith, not any individual part) ────────────
        metadata = {"textgroup": tg, "work": wk, "parts": parts_meta, "annotations": {}}

        versions = [dict(r) for r in src.execute(
            "SELECT short_id, urn, label, doc_type, text_class FROM text_units WHERE textgroup=? AND work=?",
            (tg, wk))]
        metadata["versions"] = versions

        for key, sql in [
            ("treebanks", "SELECT COUNT(DISTINCT version_short_id) FROM treebank_sentences WHERE textgroup=? AND work=?"),
            ("alignments", "SELECT COUNT(DISTINCT pair_id) FROM token_alignments WHERE textgroup=? AND work=?"),
            ("commentaries", "SELECT COUNT(DISTINCT subdoc) FROM treebank_speakers WHERE textgroup=? AND work=?"),
            ("metrical_lines", "SELECT COUNT(*) FROM metrical_lines WHERE textgroup=? AND work=?"),
        ]:
            try:
                metadata["annotations"][key] = src.execute(sql, (tg, wk)).fetchone()[0]
            except Exception:
                metadata["annotations"][key] = 0

        work_label = src.execute(
            "SELECT MIN(label) FROM text_units WHERE textgroup=? AND work=?", (tg, wk)).fetchone()
        metadata["label"] = work_label[0] if work_label[0] else work_key

        catalog["works"][work_key] = metadata
        print(f"    Annotations: {metadata['annotations']}")

    src.close()
    (out_root / "catalog.json").write_text(
        json.dumps(catalog, ensure_ascii=False, indent=1), encoding="utf-8")
    print(f"\n\u2713 Done.")


# Monolith lives in the temp build dir (never in the repo); shards -> site/.
from pathlib import Path
BUILD_DIR = Path("/tmp/persvers_build")          # must match Cell 1
MONOLITH  = BUILD_DIR / "corpus_alignment_grid.db"
if not MONOLITH.exists():
    raise FileNotFoundError(f"{MONOLITH} not found - run Cell 1 (build) first.")

split_corpus_by_work(
    str(MONOLITH),
    "/Users/gcrane/github/persverscomp/site"
)



[shard] ferdowsi.shahnameh
    part1 alignment_grid: 4289
    part1 text_segments: 8578
    part1 text_units: 3
  ✓ part1: 3.2MB (chapters 1…20)
    Annotations: {'treebanks': 1, 'alignments': 0, 'commentaries': 0, 'metrical_lines': 0}

[shard] tlg0003.tlg001
    part1 alignment_grid: 3587
    part1 text_segments: 46631
    part1 treebank_sentences: 6060
    part1 treebank_tokens: 169992
    part1 text_units: 14
  ✓ part1: 86.8MB (books 1…8)
    Annotations: {'treebanks': 1, 'alignments': 0, 'commentaries': 0, 'metrical_lines': 0}

[shard] tlg0011.tlg001
    part1 alignment_grid: 65
    part1 text_segments: 195
    part1 treebank_sentences: 1324
    part1 treebank_tokens: 17480
    part1 text_units: 5
  ✓ part1: 9.6MB (chapters 1-48…1259-1278)
    Annotations: {'treebanks': 2, 'alignments': 0, 'commentaries': 0, 'metrical_lines': 0}

[shard] tlg0011.tlg002
    part1 alignment_grid: 81
    part1 text_segments: 810
    part1 treebank_sentences: 1350
    part1 treebank_tokens: 17238
    

In [4]:
# v40 (FINAL - CLEAN STRINGS) — fixed quote escaping in onclick
# v39 — slimmed index.html. Treebank / speaker / metrical / alignment data
# are no longer inlined; they are read on demand from corpus_alignment_grid.db
# (already loaded by the page) via lazy DB-backed accessors. No loss of
# functionality vs v38. See the 'Lazy, DB-backed annotation accessors' block.
import json
import sqlite3
from pathlib import Path

# ── Target Configuration ───────────────────────────────────────
WORKSPACE_DIR = Path("/Users/gcrane/github/persverscomp")
BUILD_DIR = Path("/tmp/persvers_build")          # must match Cell 1
DB_PATH = BUILD_DIR / "corpus_alignment_grid.db"

# ── Dynamic Relational Extraction for Frontend State ───────────
if not DB_PATH.exists():
    raise FileNotFoundError(f"Monolith not found at {DB_PATH}. Run Cell 1 (build) - it now writes to the temp build dir, not the repo.")

conn = sqlite3.connect(str(DB_PATH))
cursor = conn.cursor()

# 1. Dynamically reconstruct GLOBAL_REGISTRIES from text_units table
cursor.execute("SELECT canonical_id, urn, label, text_class, textgroup, work, short_id, doc_type FROM text_units")
extracted_registries = {}
for row in cursor.fetchall():
    extracted_registries[row[0]] = {
        "urn": row[1],
        "label": row[2],
        "class": row[3],
        "textgroup": row[4],
        "work": row[5],
        "short_id": row[6],
        "doc_type": row[7]
    }

# 2. Dynamically reconstruct GLOBAL_STRUCTURES from alignment_grid table
extracted_structures = {}
cursor.execute("SELECT DISTINCT textgroup || '.' || work FROM alignment_grid")
work_keys = [r[0] for r in cursor.fetchall() ]
for w_key in work_keys:
    tg, wk = w_key.split('.')
    
    # Check if this work uses multi-book navigation or a flat structure
    cursor.execute("SELECT DISTINCT book FROM alignment_grid WHERE textgroup=? AND work=? AND book IS NOT NULL", (tg, wk))
    books = [r[0] for r in cursor.fetchall()]
    
    if books:
        # Multi-book configuration (e.g., Thucydides)
        book_map = {}
        for bk in sorted(books, key=lambda x: int(x) if x.isdigit() else x):
            cursor.execute("SELECT DISTINCT chapter FROM alignment_grid WHERE textgroup=? AND work=? AND book=? ORDER BY sort_order", (tg, wk, bk))
            book_map[bk] = [r[0] for r in cursor.fetchall()]
        extracted_structures[w_key] = book_map
    else:
        # Flat configuration (e.g., Aristotle, Sophocles)
        cursor.execute("SELECT DISTINCT chapter FROM alignment_grid WHERE textgroup=? AND work=? ORDER BY sort_order", (tg, wk))
        extracted_structures[w_key] = [r[0] for r in cursor.fetchall()]

# 3. Extract treebank sentences keyed by work+version+chapter
extracted_treebanks = {}   # { "tg.wk/v_id": { chapter: [ {subdoc, tokens, prose, literal}, ...] } }
extracted_speakers  = {}   # { "tg.wk/v_id": { subdoc: speaker } }

# Guard: treebank tables only exist if Cell 1 was run with treebank support
_tb_tables = {r[0] for r in cursor.execute(
    "SELECT name FROM sqlite_master WHERE type='table' AND name IN ('treebank_sentences','treebank_speakers')"
).fetchall()}

if 'treebank_sentences' in _tb_tables:
    cursor.execute("""
        SELECT textgroup, work, version_short_id, subdoc, chapter, section,
               sentence_json, prose_translation, literal_translation
        FROM treebank_sentences
        ORDER BY textgroup, work, version_short_id, id
    """)
    for row in cursor.fetchall():
        tg2, wk2, vid, subdoc, chapter, section, sjson, prose, literal = row
        key = f"{tg2}.{wk2}/{vid}"
        if key not in extracted_treebanks:
            extracted_treebanks[key] = {}
        if chapter not in extracted_treebanks[key]:
            extracted_treebanks[key][chapter] = []
        extracted_treebanks[key][chapter].append({
            "subdoc": subdoc,
            "section": section,
            "tokens": json.loads(sjson),
            "prose": prose,
            "literal": literal,
        })
    print(f"  ✓ Extracted treebank data: {sum(sum(len(v) for v in ch.values()) for ch in extracted_treebanks.values())} sentences")
else:
    print("  ⚠ treebank_sentences table not found — run Cell 1 to ingest treebanks")

if 'treebank_speakers' in _tb_tables:
    cursor.execute("SELECT textgroup, work, subdoc, speaker FROM treebank_speakers")
    for row in cursor.fetchall():
        tg2, wk2, subdoc, speaker = row
        skey = f"{tg2}.{wk2}"
        if skey not in extracted_speakers:
            extracted_speakers[skey] = {}
        extracted_speakers[skey][subdoc] = speaker

# 4. Extract metrical lines keyed by work+version+chapter
extracted_metrics = {}  # { "tg.wk/v_id": { chapter: { line_ref: [words] } } }
_mt_tables = {r[0] for r in cursor.execute(
    "SELECT name FROM sqlite_master WHERE type='table' AND name='metrical_lines'"
).fetchall()}
if 'metrical_lines' in _mt_tables:
    cursor.execute("""
        SELECT textgroup, work, version_short_id, line_ref, chapter, line_json
        FROM metrical_lines
        ORDER BY textgroup, work, version_short_id, id
    """)
    for row in cursor.fetchall():
        tg2, wk2, vid, line_ref, chapter, ljson = row
        mkey = f"{tg2}.{wk2}/{vid}"
        if mkey not in extracted_metrics:
            extracted_metrics[mkey] = {}
        if chapter not in extracted_metrics[mkey]:
            extracted_metrics[mkey][chapter] = {}
        extracted_metrics[mkey][chapter][line_ref] = json.loads(ljson)
    total_ml = sum(sum(len(ch) for ch in wk.values()) for wk in extracted_metrics.values())
    print(f"  \u2713 Extracted metrical data: {total_ml} lines")
else:
    print("  \u26a0 metrical_lines table not found \u2014 run Cell 1 to ingest")

# 5. Extract token alignment pairs
# Structure: { "tg.wk": { "pair_id": { "label": "...", "src_version": "...", "tgt_version": "...",
#                           "segments": { "1.1": [ {src_indices, tgt_indices, src_tokens, tgt_tokens, score}, ...] } } } }
extracted_alignments = {}

_aln_tables = {r[0] for r in cursor.execute(
    "SELECT name FROM sqlite_master WHERE type='table' AND name='token_alignments'"
).fetchall()}

if 'token_alignments' in _aln_tables:
    # Load alignment pair metadata from work registry (stored in WORK_REGISTRY in cell 0,
    # but we re-derive it here from the DB content so cell 2 stays self-contained)
    cursor.execute("""
        SELECT DISTINCT textgroup, work, pair_id, src_version, tgt_version
        FROM token_alignments
    """)
    pair_rows = cursor.fetchall()
    for tg2, wk2, pair_id, src_ver, tgt_ver in pair_rows:
        wkey = f"{tg2}.{wk2}"
        extracted_alignments.setdefault(wkey, {})
        extracted_alignments[wkey][pair_id] = {
            "src_version": src_ver,
            "tgt_version": tgt_ver,
            "segments": {}
        }
        cursor.execute("""
            SELECT segment, src_indices, tgt_indices, src_tokens, tgt_tokens, score
            FROM token_alignments
            WHERE textgroup=? AND work=? AND pair_id=?
            ORDER BY segment, id
        """, (tg2, wk2, pair_id))
        for row in cursor.fetchall():
            seg_id = row[0]
            extracted_alignments[wkey][pair_id]["segments"].setdefault(seg_id, [])
            extracted_alignments[wkey][pair_id]["segments"][seg_id].append({
                "s": json.loads(row[1]),   # src_indices
                "t": json.loads(row[2]),   # tgt_indices
                "st": json.loads(row[3]),  # src_tokens
                "tt": json.loads(row[4]),  # tgt_tokens
                "sc": round(row[5], 4)     # score
            })

conn.close()

# ── Alignment extraction summary ──────────────────────────────────────────
if extracted_alignments:
    for wk, pairs in extracted_alignments.items():
        for pid, pdata in pairs.items():
            n_segs = len(pdata.get("segments", {}))
            n_groups = sum(len(g) for g in pdata["segments"].values())
            print(f"  ✓ Alignments [{wk}] {pid}: {n_segs} segments, {n_groups} groups")
else:
    print("  ⚠ No alignment data found in DB.")
    print("    → Did you re-run Cell 0 after adding the 'alignments' keys to WORK_REGISTRY?")
    print("    → Check that the JSON files exist at the paths in WORK_REGISTRY.")

# Serialize configurations directly into JS injection tokens
struct_map_json      = json.dumps(extracted_structures)
text_registry_json   = json.dumps(extracted_registries)
# v39: the heavy annotation layers (treebank / speakers / metrical / alignments)
# are NO LONGER inlined into index.html. They already live in
# corpus_alignment_grid.db, which the page loads at startup, and are read on
# demand by the DB-backed accessors in the app (see the accessor block below).
# The extracted_* dicts are kept only for the console summary prints; the
# *_REPLACE tokens for these four layers were removed from the HTML template,
# so these strings inject nothing.
treebank_data_json   = '{}'
speakers_data_json   = '{}'
metrics_data_json    = '{}'
alignment_data_json  = '{}'


# ── 1. Compile Repository README.md Asset ─────────────────────
README_MD_CONTENT = """# Perseus 6 -- Serverless Prototype

Our goal is to provide public facing versions of the Perseus Digital Library that can run for as long as possible with minimal -- and ideally no -- changes. The Canadian [Endings Project](https://endings.uvic.ca/) provided an initial inspiration for this work but it was not clear to me whether this approach could accommodate the demands of the Perseus Digital Library. In his contributions to the Ajax Multicommentary Project, Charles Pletcher, however, created a [minimal computing version of this challenging philological use case](https://multi.ajmc.ch/passages/urn:cts:greekLit:tlg0011.tlg003:1-133). Pletcher's work made it clear that we could build every feature of the Perseus Digital Library, current and planned, in format that Tufts could serve far more easily than the traditional serverb-based versions of Perseus and that was designed to run, securely and without modification,for a long period of time. 

In summer 2026, there are two serverless Perseus efforts. Peter Nadel, Charles Pletcher, and Clifford Wulfman are working on a Minimum Viable Perseus (MVP) -- essentially a replacement for Perseus 4: the Hopper, a version that David Mimno first designed in 2003, that was developed through 2013 and has been running on virtual servers unchanged ever since. Minimum Viable Perseus aims to provide the functionality of Perseus 4 for all Perseus textual data -- essentially, MVP provides a streamlined digital library that can be updated and expanded over time.

In 2018, support from the Alexander von Humboldt Foundation allowed us to create Perseus 5: the Scaife Viewer. Scaife did not fully replicate the core functionality of Perseus 4 - it did not include the morphological services, dictionary lookups or commentaries that Scaife now supports -- but it did provide us with scalable reading environment that we could update and expand. 

Support from the Mellon Foundation, Harvard's Center for Hellenic Studies, the National Endowment for the Humanities, and Tufts University allowed us to prototype a next generation version of the Perseus Digital Library, one that built on the 2018 Scaife Viewer. Support from Schmidt Sciences has allowed us to carry this work forward and to build this experimental serverless implementation.

An NEH-funded project to create a digital edition of Aristotle's Poetics in Greek, Arabic and Latin has the driving force behind this particular effort to build a Perseus 6. We needed to be able to compare multiple versions of the Poetics, to include commentaries, to show word and phrase alignments between source texts and translations and to provide the rich linguistic annotations that treebanks in Greek, Latin and other languages have begun to make available.

Using Claude and Gemini, Gregory Crane developed the basic version of this Serverless Perseus 6 over ten days. We are are using GitHub Pages both because GitHub Pages can publish the front end work and because the constraints of GitHub Pages help us develop something sufficiently secure and lightweight that Tufts University -- and other institutions -- could easily host. Storage limits will prevent us from hosting the entire content of Perseus 6 on GitHub pages but the 1 gigabyte limitation should allow us to include enough content that we can thoroughly test the architecture and prepare for a complete implementation at Tufts.

## Infrastructure Sustainability Paradigm

Rather than relying on a continuous, monolithic server daemon—or fragmenting the corpus into hundreds of thousands of brittle static chunks strewn across a fragile local filesystem—durable architecture calls for a **decentralized, serverless delivery model**. The corpus is compiled into many small, self-contained binary shards: one read-only SQLite database per work, each holding all of that work's editions, translations, commentaries, and annotation layers, arranged as a flat directory tree addressed directly by CTS URN. Because SQLite's on-disk format is openly documented and exceptionally long-lived, every shard is a preservation-grade asset, and the deployed corpus is simply a tree of such files that can be copied, mirrored, or checksummed wholesale.

Since each shard is small, the client fetches only the single work it needs—whole—and queries it in the browser through WebAssembly, with no byte-range paging, no special server behavior, and no headers to tune. That property is what makes the system genuinely portable: it runs identically on a globally distributed object store, an institutional web server, or a researcher's laptop behind any static file server, with no backend process, no database daemon, and no writable endpoint to maintain or attack. The result is a robust, replicable foundation for permanent digital philology that scales gracefully—simply by adding shards—toward the full extent of the textual record."""
(WORKSPACE_DIR / "README.md").write_text(README_MD_CONTENT, encoding='utf-8')


# ── 2. Compile Branded HTML Workspace Application ──────────────
# ── Front-end source lives as real files under web/ (no Python-string escaping) ──
# Edit web/styles.css, web/app.js, web/index_shell.html directly: backslashes, regexes
# and quotes are authored normally. This cell only assembles them and injects the
# DB-derived JSON, producing a byte-identical index.html to the previous inline version.
#WEB_SRC = pathlib.Path("web")
WEB_SRC = Path("web")
if not (WEB_SRC / "index_shell.html").exists():
    WEB_SRC = WORKSPACE_DIR / "web"          # fallback: web/ alongside the workspace
_shell  = open(WEB_SRC / "index_shell.html", encoding="utf-8", newline="").read()
_styles = open(WEB_SRC / "styles.css",       encoding="utf-8", newline="").read()
_app_js = open(WEB_SRC / "app.js",           encoding="utf-8", newline="").read()
INDEX_HTML_CONTENT = (
    _shell
    .replace("%%PERSEUS_STYLES%%", _styles)
    .replace("%%PERSEUS_APP_JS%%", _app_js)
    .replace("STRUCT_REPLACE",    struct_map_json)
    .replace("REGISTRY_REPLACE",  text_registry_json)
    .replace("TREEBANK_REPLACE",  treebank_data_json)
    .replace("SPEAKERS_REPLACE",  speakers_data_json)
    .replace("METRICAL_REPLACE",  metrics_data_json)
    .replace("ALIGNMENT_REPLACE", alignment_data_json)
)

(WORKSPACE_DIR / "index.html").write_text(INDEX_HTML_CONTENT, encoding='utf-8')
(WORKSPACE_DIR / ".nojekyll").write_text("", encoding='utf-8')
print("[SUCCESS] Production Standalone Workspace compiled cleanly.")

  ✓ Extracted treebank data: 41076 sentences
  ✓ Extracted metrical data: 27792 lines
  ✓ Alignments [tlg0086.tlg034] bywater1909-grc1__butcher1911-eng2: 381 segments, 8962 groups
  ✓ Alignments [tlg0086.tlg034] bywater1909-grc1__bywater1909-eng1: 380 segments, 9126 groups
[SUCCESS] Production Standalone Workspace compiled cleanly.


In [5]:
from pathlib import Path
# ── Runtime shard loader (JavaScript) ─────────────────────────────────────
# Writes shard_loader.js next to the app and prints the integration swaps.
# Splice SHARD_LOADER_JS into your app template (or <script src="shard_loader.js">).

SHARD_LOADER_JS = r"""
// ── Sharded corpus loader ─────────────────────────────────────────────────
// Replaces the single-monolith fetch. Each work lives in its own small SQLite
// shard; we fetch the whole shard (it's small), open it with sql.js, and cache
// it. No httpvfs / Range requests -> runs on any static server, incl. file-less
// local `python -m http.server`, with no header tuning.

let CATALOG = null;
const SHARD_CACHE = new Map();   // workKey -> sql.js Database
const SHARD_INFLIGHT = new Map(); // workKey -> Promise (dedupe concurrent loads)

const DATA_DIR = "data";

// urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.10  ->  parts
function parseCtsUrn(urn) {
    const m = /^urn:cts:([^:]+):([^.]+)\.([^.:]+)(?:\.([^:]+))?(?::(.*))?$/.exec(urn || "");
    if (!m) return null;
    return { textClass: m[1], textgroup: m[2], work: m[3],
             version: m[4] || null, passage: m[5] || null,
             workKey: `${m[2]}.${m[3]}` };
}

// FLAT layout: data/<textgroup>/<work>/<tg>.<wk>.db. The path is fully
// determined by the work key alone — no namespace tier, no catalog lookup.
function shardPathFor(textgroup, work) {
    return `${DATA_DIR}/${textgroup}/${work}/${textgroup}.${work}.db`;
}
function shardPathForWorkKey(workKey) {
    const [tg, wk] = workKey.split(".");
    return shardPathFor(tg, wk);
}

async function loadCatalog() {
    if (CATALOG) return CATALOG;
    // catalog.json is tiny and must always be fresh: bypass the HTTP cache so a
    // rebuild's new sizes/versions show up without a manual hard-refresh.
    const r = await fetch(`./${DATA_DIR}/../catalog.json?v=${Date.now()}`, { cache: "no-store" });
    if (!r.ok) throw new Error("catalog.json not found");
    CATALOG = await r.json();
    return CATALOG;
}

// Fetch + open a shard for a work; cached and de-duplicated.
async function getDbForWork(workKey, shardPathHint) {
    if (SHARD_CACHE.has(workKey)) return SHARD_CACHE.get(workKey);
    if (SHARD_INFLIGHT.has(workKey)) return SHARD_INFLIGHT.get(workKey);

    const path = shardPathHint || shardPathForWorkKey(workKey);

    const p = (async () => {
        const resp = await fetch(`./${path}`);
        if (!resp.ok) throw new Error(`Shard not found: ${path}`);
        const buf = new Uint8Array(await resp.arrayBuffer());
        const db = new window.SQL_WASM_ENGINE.Database(buf);
        SHARD_CACHE.set(workKey, db);
        SHARD_INFLIGHT.delete(workKey);
        return db;
    })();
    SHARD_INFLIGHT.set(workKey, p);
    return p;
}

// Optional memory hygiene for long sessions / "own machine" use.
function evictWorkExcept(keepWorkKey) {
    for (const [k, db] of SHARD_CACHE) {
        if (k !== keepWorkKey) { try { db.close(); } catch (e) {} SHARD_CACHE.delete(k); }
    }
}

// ── Per-work data access (replaces the inlined *_REPLACE globals) ──────────
// These read from the loaded shard instead of giant in-HTML JSON blobs.
function queryAll(db, sql, params = []) {
    const out = []; const st = db.prepare(sql); st.bind(params);
    while (st.step()) out.push(st.getAsObject());
    st.free(); return out;
}
function registryForWork(db) {
    return queryAll(db, "SELECT short_id, urn, label, doc_type, text_class FROM text_units ORDER BY doc_type, short_id");
}
function treebankForChapter(db, version, chapter) {
    return queryAll(db,
        "SELECT subdoc, chapter, section, sentence_json, prose_translation, literal_translation " +
        "FROM treebank_sentences WHERE version_short_id=? AND chapter=? ORDER BY id",
        [version, chapter]);
}
function alignmentsForPair(db, pairId, segment) {
    return queryAll(db,
        "SELECT src_indices, tgt_indices, src_tokens, tgt_tokens, score " +
        "FROM token_alignments WHERE pair_id=? AND segment=?", [pairId, segment]);
}
function metricalForChapter(db, version, chapter) {
    return queryAll(db,
        "SELECT line_ref, line_json FROM metrical_lines WHERE version_short_id=? AND chapter=?",
        [version, chapter]);
}

// ── Entry point: deep-link routing ─────────────────────────────────────────
async function routeToUrn(urn) {
    const parsed = parseCtsUrn(urn);
    if (!parsed) throw new Error("Unparseable CTS URN: " + urn);
    const path = shardPathFor(parsed.textgroup, parsed.work);
    const db = await getDbForWork(parsed.workKey, path);
    window.dbInstance = db;          // existing text_segments / grid queries now hit the shard
    return { parsed, db };
}

"""

try:
    _site = WORKSPACE_DIR / 'site'
except NameError:
    _site = Path('site')
_site.mkdir(parents=True, exist_ok=True)
(_site / 'shard_loader.js').write_text(SHARD_LOADER_JS, encoding='utf-8')
print('wrote', _site / 'shard_loader.js')

print(r"""
INTEGRATION SWAPS in the app/HTML generator (current monolith app → sharded):

  1. STARTUP: remove the single  fetch('./corpus_alignment_grid.db')  block.
     Instead, after initSqlJs resolves, call  loadCatalog()  then, if the URL has a
     URN,  await routeToUrn(urn)  (sets window.dbInstance to that work's shard);
     otherwise show the splash / work picker built from CATALOG.works.

  2. WORK SWITCH: wherever activeWorkKey changes, call
        await getDbForWork(activeWorkKey);  window.dbInstance = SHARD_CACHE.get(activeWorkKey);
     (optionally evictWorkExcept(activeWorkKey) for long sessions). All existing
     text_segments / alignment_grid queries then hit the shard unchanged.

  3. DROP the inlined globals fed by *_REPLACE (TREEBANK_DATA, SPEAKERS_DATA,
     METRICAL_DATA, GLOBAL_ALIGNMENTS, and the all-works slice of TEXT_REGISTRY).
     Replace each lookup with a shard query on the active db:
        TREEBANK_DATA[wk][ch]      → treebankForChapter(db, version, ch)
        GLOBAL_ALIGNMENTS[wk][pid] → alignmentsForPair(db, pid, segment)
        METRICAL_DATA[wk][ch]      → metricalForChapter(db, version, ch)
        TEXT_REGISTRY (this work)  → registryForWork(db)
     Keep GLOBAL_STRUCTURES per-work only (or read alignment_grid from the shard).

  4. CATALOG is the only global JSON left (small) — used for the work picker and
     version lists. Routing needs NO catalog now: both full URNs and bare workKeys
     map to data/<textgroup>/<work>/<tg>.<wk>.db by string split alone.
""")


wrote /Users/gcrane/github/persverscomp/site/shard_loader.js

INTEGRATION SWAPS in the app/HTML generator (current monolith app → sharded):

  1. STARTUP: remove the single  fetch('./corpus_alignment_grid.db')  block.
     Instead, after initSqlJs resolves, call  loadCatalog()  then, if the URL has a
     URN,  await routeToUrn(urn)  (sets window.dbInstance to that work's shard);
     otherwise show the splash / work picker built from CATALOG.works.

  2. WORK SWITCH: wherever activeWorkKey changes, call
        await getDbForWork(activeWorkKey);  window.dbInstance = SHARD_CACHE.get(activeWorkKey);
     (optionally evictWorkExcept(activeWorkKey) for long sessions). All existing
     text_segments / alignment_grid queries then hit the shard unchanged.

  3. DROP the inlined globals fed by *_REPLACE (TREEBANK_DATA, SPEAKERS_DATA,
     METRICAL_DATA, GLOBAL_ALIGNMENTS, and the all-works slice of TEXT_REGISTRY).
     Replace each lookup with a shard query on the active db:
        TREEBANK

In [6]:
# ── CELL 1c: Author-level lexicon ingestion (Cunliffe, Dindorf, Pizzi, Betant) ─
#
# Lexica are scoped to a TEXTGROUP (an author/tradition), not to a single
# work, and are shared across every work in that textgroup -- unlike
# editions/treebanks/commentaries, which are one row per WORK_REGISTRY
# entry. So they get their own small registry (LEXICON_REGISTRY) and their
# own tables, independent of WORK_REGISTRY, but written into the SAME
# monolith DB so Cell 2c can shard them the same way everything else is
# sharded.
#
# Three source TEI shapes are handled, converging on one output row shape:
#   - "cunliffe"  : nested <div type="textpart" xml:id="..." n="HEADWORD">
#                   entries (Cunliffe's Homeric lexicon format).
#   - "tei_dict"  : standard TEI dictionary <entry xml:id="..."><form>
#                   <orth>HEADWORD</orth></form><sense>...</sense>
#                   <cit><bibl>...</bibl><quote>...</quote></cit></entry>
#                   (Dindorf's Aeschylus lexicon format).
#   - "pizzi"     : <div type="textpart" subtype="entry" xml:id="..."> with
#                   TWO <head> elements (Persian script + Latin
#                   transliteration), optional <div subtype="wsense"> senses,
#                   and untargeted <ref>/"see X" cross-references that must
#                   be resolved by matching transliterated text against a
#                   headword index rather than by xml:id (Pizzi's Persian
#                   dictionary). headword_key is built from the SCRIPT head
#                   (matches the treebank's lemma column, which is Persian
#                   script, not transliteration) while headword_translit is
#                   carried separately for display and for resolving the
#                   untargeted cross-references.
#
# Run this AFTER Cell 1b (it reuses Cell 1b's norm_key logic for headword
# matching, redefined here so this cell stays self-contained) and BEFORE
# Cell 2c.

import sqlite3
import re
import unicodedata
import xml.etree.ElementTree as ET
from pathlib import Path

BUILD_DIR = Path("/tmp/persvers_build")   # must match Cell 0/1/1b
DB_PATH = BUILD_DIR / "corpus_alignment_grid.db"

TEI_NS = "{http://www.tei-c.org/ns/1.0}"
XML_ID = "{http://www.w3.org/XML/1998/namespace}id"
XML_LANG = "{http://www.w3.org/XML/1998/namespace}lang"


def norm_key(s):
    """Accent/diacritic- and case-insensitive lookup key. Same logic as
    Cell 1b's norm_key -- kept in sync deliberately so a treebank token's
    lemma_norm and a lexicon entry's headword_key are directly comparable."""
    if not s:
        return None
    t = unicodedata.normalize('NFD', s)
    t = ''.join(ch for ch in t if unicodedata.category(ch) != 'Mn')
    t = unicodedata.normalize('NFC', t).lower()
    t = t.replace('\u03c2', '\u03c3')  # final sigma -> medial sigma
    return t


def _tag(el):
    return el.tag.split('}')[-1]


def safe_parse_lexicon(path):
    """Same recover-on-error strategy as the main safe_parse(), redefined
    here so this cell doesn't depend on Cell 0 still being in memory."""
    try:
        import lxml.etree as LET
        parser = LET.XMLParser(recover=True)
        tree = LET.parse(str(path), parser=parser)
        import io
        buf = io.BytesIO()
        tree.write(buf)
        buf.seek(0)
        return ET.parse(buf)
    except ImportError:
        return ET.parse(str(path))


# ─────────────────────────────────────────────────────────────────────────
# Registry: one entry per SOURCE FILE. `textgroups` is the list of CTS
# textgroup ids this lexicon should be offered for. `shard_file` groups
# lexica that should ship in the same .db (see Cell 2c) -- lexica that
# will plausibly be consulted together (e.g. Cunliffe words + Cunliffe
# names, both used while reading Homer) share a shard_file; lexica for
# an unrelated author get their own.
# ─────────────────────────────────────────────────────────────────────────
LEXICON_REGISTRY = {
    "cunliffe-words": {
        "path": "/Users/gcrane/github/Homerica/cunliffe.lexentries.unicode.xml",
        "format": "cunliffe",
        "title": "A Lexicon of the Homeric Dialect",
        "author": "Richard John Cunliffe",
        "citation": "Cunliffe, R. J. (1924). A Lexicon of the Homeric Dialect. Blackie and Son.",
        "entry_kind": "word",
        "textgroups": ["tlg0012"],           # Homer
        "shard_file": "lexica_homer.db",
    },
    "cunliffe-names": {
        "path": "/Users/gcrane/github/Homerica/cunliffe.hompers.unicode.xml",
        "format": "cunliffe",
        "title": "A Lexicon of the Homeric Dialect (Proper Names)",
        "author": "Richard John Cunliffe",
        "citation": "Cunliffe, R. J. (1924). A Lexicon of the Homeric Dialect. Blackie and Son.",
        "entry_kind": "name",
        "textgroups": ["tlg0012"],           # Homer
        "shard_file": "lexica_homer.db",     # same shard as cunliffe-words
    },
    "dindorf-aeschylus": {
        "path": "/Users/gcrane/github/GRC_misc/aeschylus-gemini.dindorf.lexicon.xml",
        "format": "tei_dict",
        "title": "Lexicon Aeschyleum",
        "author": "Wilhelm Dindorf",
        "citation": "Dindorf, W. (1873\u20131876). Lexicon Aeschyleum. B. G. Teubner.",
        "entry_kind": "word",
        "textgroups": ["tlg0085"],           # Aeschylus
        "shard_file": "lexica_aeschylus.db",
    },
    "pizzi-persian": {
        "path": "/Users/gcrane/github/Shahnameh/pizzi-glossary.en.xml",
        "format": "pizzi",
        "title": "Manual of the Persian Language (Glossary)",
        "author": "Italo Pizzi",
        "citation": "Pizzi, I. (1891). Firdusiana anthology with a compendium of Persian "
                    "grammar and a dictionary. W. Gerhard.",
        "entry_kind": "word",
        "textgroups": ["ferdowsi"],           # matches WORK_REGISTRY["ferdowsi.shahnameh"]["textgroup"]
        "shard_file": "lexica_persian.db",
    },
    "betant-thucydides": {
        "path": "/Users/gcrane/github/Thucydides-new-working-materials/betant.thuclex_lateng4.xml",
        "format": "betant",
        "title": "Lexicon Thucydideum",
        "author": "\u00c9lie-Ami B\u00e9tant",
        "citation": "B\u00e9tant, \u00c9.-A. (1843\u20131847). Lexicon Thucydideum. Carey.",
        "entry_kind": "word",
        "textgroups": ["tlg0003"],           # Thucydides (no WORK_REGISTRY entry yet --
                                              # kept ready for when an edition is added)
        "shard_file": "lexica_thucydides.db",
    },
}


# ─────────────────────────────────────────────────────────────────────────
# Shared inline-content renderer. Small, deliberately duplicated from
# extract_text_recursive() in Cell 0 rather than extended in place --
# lexicon markup (<ref>, <term>, <gloss>, <sense>, <cit>, <bibl>, <quote>)
# is different enough from reading-text markup that bolting it onto the
# text renderer risked regressing existing edition/commentary rendering.
# ─────────────────────────────────────────────────────────────────────────
def _lex_inline_html(elem, lexicon_id, entry_id_by_target=None):
    """Renders one lexicon entry element (and its children) to HTML.
    `entry_id_by_target` is unused here (cross-refs are resolved at click
    time in the browser, not at build time), kept as a parameter in case a
    future pass wants to grey out dead links whose target never got
    ingested.
    """
    tag = _tag(elem)
    parts = []

    if tag == 'p':
        parts.append('<div class="tb-lex-p">')
    elif tag == 'sense':
        parts.append('<div class="tb-lex-sense">')
    elif tag == 'cit':
        parts.append('<div class="tb-lex-cit">')
    elif tag == 'def':
        parts.append('<span class="tb-lex-def">')
    elif tag == 'note':
        parts.append('<span class="tb-lex-note">(')
    elif tag == 'bibl':
        parts.append('<span class="tb-lex-cite">')
    elif tag == 'quote':
        # Betant's teiHeader notes the Greek citation text was never keyed in
        # for budget reasons -- <quote/> ships as an empty placeholder in
        # ~35k spots. Rendering an empty <div> for each would be pure clutter,
        # so bail out before the wrapper is even opened.
        if not elem.text and len(elem) == 0:
            return ''
        lang = elem.get('{http://www.w3.org/XML/1998/namespace}lang', '')
        cls = 'tb-lex-quote tb-greek' if lang == 'grc' else 'tb-lex-quote'
        parts.append(f'<div class="{cls}">')
    elif tag == 'foreign':
        lang = elem.get('{http://www.w3.org/XML/1998/namespace}lang', '')
        cls = 'tb-greek' if lang in ('greek', 'grc') else ''
        parts.append(f'<span class="{cls}">')
    elif tag == 'term':
        parts.append('<span class="tb-lex-term">')
    elif tag == 'gloss':
        # Betant pairs a Latin definition with an added English translation as
        # two sibling <gloss> elements -- tag each with its xml:lang so the
        # reader-side "Hide Latin" toggle can target tb-lex-lat via CSS.
        lang = elem.get(XML_LANG, '')
        cls = 'tb-lex-gloss'
        if lang == 'lat':
            cls += ' tb-lex-lat'
        elif lang == 'eng':
            cls += ' tb-lex-eng'
        parts.append(f'<span class="{cls}">')
    elif tag == 'hi':
        parts.append('<span class="tb-lex-hi">')
    elif tag == 'ref':
        target = (elem.get('target') or '').strip()
        if target:
            parts.append(
                f'<a href="javascript:void(0)" class="tb-lex-ref" '
                f'onclick="openLexiconEntry(\'{lexicon_id}\',\'{target}\')">'
            )
        else:
            parts.append('<span class="tb-lex-ref-dead">')
    # head / form / orth are handled separately by the caller (they supply
    # the entry's own headword, shown once in the panel title) -- skip here
    # so the headword doesn't get duplicated inside the entry body.
    elif tag in ('head', 'form', 'orth'):
        return ''

    if elem.text:
        parts.append(elem.text)
    for child in elem:
        parts.append(_lex_inline_html(child, lexicon_id, entry_id_by_target))
        if child.tail:
            parts.append(child.tail)

    if tag == 'p':
        parts.append('</div>')
    elif tag == 'sense':
        parts.append('</div>')
    elif tag == 'cit':
        parts.append('</div>')
    elif tag == 'def':
        parts.append('</span>')
    elif tag == 'note':
        parts.append(')</span>')
    elif tag == 'bibl':
        parts.append('</span>')
    elif tag == 'quote':
        parts.append('</div>')
    elif tag in ('foreign', 'term', 'gloss', 'hi'):
        parts.append('</span>')
    elif tag == 'ref':
        target = (elem.get('target') or '').strip()
        parts.append('</a>' if target else '</span>')

    return ''.join(parts)


def _plain_text_len(elem):
    """Rough visible-character count of an element's rendered content --
    used only to flag very short 'see X' pointer entries for aliasing."""
    return len(re.sub(r'<[^>]+>', '', _lex_inline_html(elem, '_probe_')))


# ─────────────────────────────────────────────────────────────────────────
# Format 1: Cunliffe -- <div type="textpart" xml:id="..." n="HEADWORD">
# ─────────────────────────────────────────────────────────────────────────
def parse_lexicon_cunliffe_tei(path, lexicon_id):
    tree = safe_parse_lexicon(path)
    root = tree.getroot()
    body = root.find(f'.//{TEI_NS}body')
    if body is None:
        body = root.find('.//body')
    if body is None:
        print(f"  \u26a0 no <body> found in {path}")
        return [], [], []

    entries, aliases, citations = [], [], []

    for div in body:
        if _tag(div) != 'div':
            continue
        entry_id = div.get(XML_ID)
        if not entry_id:
            continue  # nested/unlabeled sub-divs are handled inside their parent, not iterated here
        headword_display = (div.get('n') or '').strip()
        if not headword_display:
            head_el = div.find(f'{TEI_NS}head')
            if head_el is None:
                head_el = div.find('head')
            headword_display = (''.join(head_el.itertext()).strip() if head_el is not None else entry_id)

        headword_key = norm_key(headword_display)

        # Render every child except <head> (already captured as the headword).
        body_html = ''.join(
            _lex_inline_html(child, lexicon_id)
            for child in div if _tag(child) != 'head'
        )

        entries.append({
            "entry_id": entry_id,
            "headword_display": headword_display,
            "headword_key": headword_key,
            "sort_key": headword_key or '',
            "entry_html": body_html,
        })

        # Short "see X" pointer entries: alias this headword straight to
        # the single ref's target so a lookup resolves to the fuller entry.
        refs = div.findall(f'.//{TEI_NS}ref') or div.findall('.//ref')
        if len(refs) == 1 and _plain_text_len(div) < 60:
            target = (refs[0].get('target') or '').strip()
            if target:
                aliases.append({"alias_key": headword_key, "entry_id": target})

        # Citations with real CTS URNs: <bibl n="Perseus:abo:tlg,0012,001:14:78">
        for bibl in div.findall(f'.//{TEI_NS}bibl') or div.findall('.//bibl'):
            n = (bibl.get('n') or '').strip()
            m = re.match(r'^Perseus:abo:tlg,(\d+),(\d+):(.+)$', n)
            if m:
                tg_num, wk_num, ref_parts = m.groups()
                citations.append({
                    "entry_id": entry_id,
                    "textgroup": f"tlg{tg_num}",
                    "work": f"tlg{wk_num}",
                    "ref": ref_parts.replace(':', '.'),
                    "urn": f"urn:cts:greekLit:tlg{tg_num}.tlg{wk_num}:{ref_parts.replace(':', '.')}",
                })

    return entries, aliases, citations


# ─────────────────────────────────────────────────────────────────────────
# Format 2: TEI dictionary -- <entry xml:id="..."><form><orth>...
# ─────────────────────────────────────────────────────────────────────────
# NOTE: Dindorf's <bibl> citations are plain text ("Agamemnon 1087"), not
# CTS URNs, so they can't be resolved to a passage_urn without a play-name
# -> work-id lookup this cell doesn't have. `lexicon_citations` still
# records them with urn=NULL so they're captured now and resolvable later
# (Cell 2c / app.js should treat urn IS NULL citations as "ref text only,
# no jump-to-passage link yet").
PLAY_NAME_TO_WORK = {
    # Fill in once the Aeschylus WORK_REGISTRY entries exist, e.g.:
    # "Agamemnon": "tlg0003", "Choephori": "tlg0004", ...
}


def parse_lexicon_dict_tei(path, lexicon_id):
    tree = safe_parse_lexicon(path)
    root = tree.getroot()
    edition_div = None
    for div in root.iter(f'{TEI_NS}div'):
        if div.get('type') == 'edition':
            edition_div = div
            break
    if edition_div is None:
        print(f"  \u26a0 no <div type=\"edition\"> found in {path}")
        return [], [], []

    entries, aliases, citations = [], [], []

    for entry in edition_div.findall(f'{TEI_NS}entry') or edition_div.findall('entry'):
        entry_id = entry.get(XML_ID)
        if not entry_id:
            continue

        orth_el = entry.find(f'.//{TEI_NS}orth')
        if orth_el is None:
            orth_el = entry.find('.//orth')
        headword_display = (''.join(orth_el.itertext()).strip() if orth_el is not None else entry_id)
        headword_key = norm_key(headword_display)

        body_html = ''.join(
            _lex_inline_html(child, lexicon_id)
            for child in entry if _tag(child) != 'form'
        )

        entries.append({
            "entry_id": entry_id,
            "headword_display": headword_display,
            "headword_key": headword_key,
            "sort_key": headword_key or '',
            "entry_html": body_html,
        })

        for cit in entry.findall(f'{TEI_NS}cit') or entry.findall('cit'):
            bibl_el = cit.find(f'{TEI_NS}bibl')
            if bibl_el is None:
                bibl_el = cit.find('bibl')
            bibl_text = (''.join(bibl_el.itertext()).strip() if bibl_el is not None else '')
            if not bibl_text:
                continue
            m = re.match(r'^([A-Za-z]+)\s+(?:\(fr\.\s*)?(\d+)\)?$', bibl_text)
            if not m:
                continue  # descriptive/essay-style bibl, not a "Play ####" citation -- skip
            play, line = m.groups()
            work_id = PLAY_NAME_TO_WORK.get(play)
            citations.append({
                "entry_id": entry_id,
                "textgroup": "tlg0085" if work_id else None,
                "work": work_id,
                "ref": line,
                "urn": (f"urn:cts:greekLit:tlg0085.{work_id}:{line}"
                        if work_id else None),
            })

    return entries, aliases, citations


# ─────────────────────────────────────────────────────────────────────────
# Format 3: Pizzi -- dual-script headwords, <ref>/"see X" resolved by text
# match against transliteration rather than by xml:id target.
# ─────────────────────────────────────────────────────────────────────────
def _pizzi_render_child(elem, lexicon_id, translit_index):
    """Like _lex_inline_html, but: (a) <div subtype="wsense"> gets a visible
    sense-number wrapper, (b) <ref> and the target of a 'see X' <term n="v">
    have no target attribute in the source -- they're resolved here by
    normalizing their text and looking it up in translit_index (built from
    every entry's headword_translit). A miss renders as plain/muted text
    instead of a broken link -- about a third of refs won't resolve (they
    point to inflected forms, not headwords), and that's fine."""
    tag = _tag(elem)
    parts = []

    if tag == 'div' and elem.get('subtype') == 'wsense':
        n = elem.get('n', '')
        parts.append(f'<div class="tb-lex-wsense"><span class="tb-lex-sense-num">{n}.</span> ')
    elif tag == 'p':
        parts.append('<div class="tb-lex-p">')
    elif tag == 'gloss':
        parts.append('<span class="tb-lex-gloss">')
    elif tag == 'term':
        n = elem.get('n', '')
        url = n.split(':', 1)[1] if ':' in n and n.split(':', 1)[1].startswith('http') else None
        parts.append(f'<a href="{url}" target="_blank" rel="noopener" class="tb-lex-term-link">' if url
                      else '<span class="tb-lex-term">')
    elif tag == 'foreign':
        lang = elem.get(XML_LANG, '')
        cls = 'tb-lex-fa tb-lex-hi' if lang.startswith('fas') else 'tb-lex-hi'
        parts.append(f'<span class="{cls}">')
    elif tag == 'ref':
        text = ''.join(elem.itertext()).strip()
        target = translit_index.get(norm_key(text))
        if target:
            parts.append(f'<a href="javascript:void(0)" class="tb-lex-ref" '
                         f'onclick="openLexiconEntry(\'{lexicon_id}\',\'{target}\')">')
        else:
            parts.append('<span class="tb-lex-ref-dead">')

    if elem.text:
        parts.append(elem.text)
    for child in elem:
        parts.append(_pizzi_render_child(child, lexicon_id, translit_index))
        if child.tail:
            parts.append(child.tail)

    if tag == 'div' and elem.get('subtype') == 'wsense':
        parts.append('</div>')
    elif tag == 'p':
        parts.append('</div>')
    elif tag == 'gloss':
        parts.append('</span>')
    elif tag == 'term':
        n = elem.get('n', '')
        url = n.split(':', 1)[1] if ':' in n and n.split(':', 1)[1].startswith('http') else None
        parts.append('</a>' if url else '</span>')
    elif tag == 'foreign':
        parts.append('</span>')
    elif tag == 'ref':
        text = ''.join(elem.itertext()).strip()
        target = translit_index.get(norm_key(text))
        parts.append('</a>' if target else '</span>')

    return ''.join(parts)


def parse_lexicon_pizzi_tei(path, lexicon_id):
    tree = safe_parse_lexicon(path)
    root = tree.getroot()
    body = root.find(f'.//{TEI_NS}body')
    if body is None:
        print(f"  \u26a0 no <body> found in {path}")
        return [], [], []

    raw_entries = []          # (entry_id, headword_display, headword_translit, children)
    translit_index = {}       # norm_key(translit) -> entry_id, first-wins on homographs

    for div in body:
        if _tag(div) != 'div' or div.get('subtype') != 'entry':
            continue
        entry_id = div.get(XML_ID)
        if not entry_id:
            continue

        heads = div.findall(f'{TEI_NS}head')
        script_head = next((h for h in heads if h.get(XML_LANG) == 'fas'), None)
        translit_head = next((h for h in heads if (h.get(XML_LANG) or '').startswith('fas-t')), None)
        headword_display = (''.join(script_head.itertext()).strip()
                             if script_head is not None else entry_id)
        headword_translit = (''.join(translit_head.itertext()).strip()
                              if translit_head is not None else None)

        children = [c for c in div if _tag(c) != 'head']
        raw_entries.append((entry_id, headword_display, headword_translit, children))

        tkey = norm_key(headword_translit)
        if tkey and tkey not in translit_index:  # first entry wins a homograph collision
            translit_index[tkey] = entry_id

    entries, aliases = [], []
    for entry_id, headword_display, headword_translit, children in raw_entries:
        # headword_key matches the SCRIPT form -- this is what has to line up
        # with the treebank's lemma column (Persian script, per the .fa.conllu
        # naming in WORK_REGISTRY["ferdowsi.shahnameh"]).
        headword_key = norm_key(headword_display)
        body_html = ''.join(_pizzi_render_child(c, lexicon_id, translit_index) for c in children)

        entries.append({
            "entry_id": entry_id,
            "headword_display": headword_display,
            "headword_translit": headword_translit,
            "headword_key": headword_key,
            "sort_key": norm_key(headword_translit) or headword_key or '',
            "entry_html": body_html,
        })

        # "see X" pointer entries: <term n="v">see</term> <foreign>TARGET</foreign>,
        # short enough that the whole entry is really just a cross-reference.
        plain_len = len(re.sub(r'<[^>]+>', '', body_html))
        if plain_len < 80:
            see_terms = [c for c in children if _tag(c) == 'p']
            for p in see_terms:
                kids = list(p)
                for i, k in enumerate(kids):
                    if _tag(k) == 'term' and k.get('n') == 'v' and i + 1 < len(kids) \
                            and _tag(kids[i + 1]) == 'foreign':
                        target_text = ''.join(kids[i + 1].itertext()).strip()
                        target_id = translit_index.get(norm_key(target_text))
                        if target_id:
                            aliases.append({"alias_key": headword_key, "entry_id": target_id})

    # Pizzi is a dictionary/grammar, not a citation-heavy commentary -- no
    # <bibl n="Perseus:..."> anywhere in the source, so citations stay empty
    # (same as Cunliffe's names lexicon). Nothing to parse here.
    citations = []
    return entries, aliases, citations


# ─────────────────────────────────────────────────────────────────────────
# Format 4: Betant -- <div type="textpart" subtype="entry" n="lsj-...">.
#   Unlike Cunliffe, `n` here is an LSJ cross-reference id, not the headword
#   display form, AND it's absent on ~52 of the 4864 entries, so it can't be
#   used as entry_id on its own -- entries with no `n` get an auto-generated
#   fallback id instead. The headword itself lives in
#   <head><foreign xml:lang="grc">...</foreign></head>.
#
#   Body is one or more <p> containing, per sense:
#     <gloss xml:lang="lat">Latin definition</gloss>
#     <gloss xml:lang="eng">English translation</gloss>, then one or more
#     <bibl n="thuc. BOOK.CHAPTER.SECTION">display</bibl> citations, each
#     optionally followed by an empty <quote/> (a Greek citation that was
#     never keyed in -- see teiHeader encodingDesc). _lex_inline_html
#     already tags the two <gloss> languages (tb-lex-lat / tb-lex-eng) and
#     skips empty <quote/>, so this parser just needs to walk entries and
#     pull citations out of the `n` attribute.
# ─────────────────────────────────────────────────────────────────────────
BETANT_BIBL_RE = re.compile(r'^(?:thuc\.|Thuc\.)\s*(\d+)\.(\d+)(?:\.(\d+))?$')

def parse_lexicon_betant_tei(path, lexicon_id):
    tree = safe_parse_lexicon(path)
    root = tree.getroot()
    body = root.find(f'.//{TEI_NS}body')
    if body is None:
        body = root.find('.//body')
    if body is None:
        print(f"  \u26a0 no <body> found in {path}")
        return [], [], []

    entries, aliases, citations = [], [], []
    auto_counter = 0
    seen_ids = {}  # base id -> occurrence count -- Betant's `n` (an LSJ beta-code
                   # id) is occasionally reused across 2+ entries (158 cases: e.g.
                   # an adjective and adverb, or two distinct senses, sharing one
                   # LSJ cross-reference), so it isn't unique on its own the way
                   # Cunliffe's xml:id is.

    # iter() rather than a direct child walk: entries sit one level down
    # inside <div type="textpart" subtype="section"> letter-groupings
    # (24 of them, e.g. alpha/beta/...), not directly under <body>.
    for entry_div in body.iter(f'{TEI_NS}div'):
        if entry_div.get('type') != 'textpart' or entry_div.get('subtype') != 'entry':
            continue

        n_attr = (entry_div.get('n') or '').strip()
        if n_attr:
            base_id = n_attr
        else:
            auto_counter += 1
            base_id = f"betant-auto-{auto_counter}"

        seen_ids[base_id] = seen_ids.get(base_id, 0) + 1
        entry_id = base_id if seen_ids[base_id] == 1 else f"{base_id}-{seen_ids[base_id]}"

        head_el = entry_div.find(f'{TEI_NS}head')
        if head_el is None:
            head_el = entry_div.find('head')
        headword_display = (''.join(head_el.itertext()).strip() if head_el is not None else entry_id)
        headword_key = norm_key(headword_display)

        body_html = ''.join(
            _lex_inline_html(child, lexicon_id)
            for child in entry_div if _tag(child) != 'head'
        )

        entries.append({
            "entry_id": entry_id,
            "headword_display": headword_display,
            "headword_key": headword_key,
            "sort_key": headword_key or '',
            "entry_html": body_html,
        })

        for bibl in entry_div.iter(f'{TEI_NS}bibl'):
            ref_attr = (bibl.get('n') or '').strip()
            m = BETANT_BIBL_RE.match(ref_attr)
            if not m:
                continue  # shouldn't happen -- every bibl in this source matched at ingest time
            book, chapter, section = m.groups()
            ref = f"{book}.{chapter}" + (f".{section}" if section else "")
            citations.append({
                "entry_id": entry_id,
                "textgroup": "tlg0003",
                "work": "tlg001",
                "ref": ref,
                "urn": f"urn:cts:greekLit:tlg0003.tlg001:{ref}",
            })

    return entries, aliases, citations


# ─────────────────────────────────────────────────────────────────────────
# Ingestion loop
# ─────────────────────────────────────────────────────────────────────────
PARSERS = {
    "cunliffe": parse_lexicon_cunliffe_tei,
    "tei_dict": parse_lexicon_dict_tei,
    "pizzi": parse_lexicon_pizzi_tei,
    "betant": parse_lexicon_betant_tei,
}

BUILD_DIR.mkdir(parents=True, exist_ok=True)  # sqlite3.connect can create the .db file
                                                # itself, but not a missing parent dir --
                                                # that's what "unable to open database
                                                # file" actually means when it's thrown here.
if not DB_PATH.exists():
    raise FileNotFoundError(
        f"{DB_PATH} does not exist yet -- run Cell 0 (builds the monolith from "
        f"WORK_REGISTRY) before this cell. This cell only ADDS lexicon_* tables "
        f"to an already-built monolith; it doesn't build the corpus itself."
    )

conn = sqlite3.connect(str(DB_PATH))
cur = conn.cursor()

cur.executescript("""
    CREATE TABLE IF NOT EXISTS lexicon_meta (
        lexicon_id TEXT PRIMARY KEY,
        title TEXT NOT NULL,
        author TEXT,
        citation TEXT,
        entry_kind TEXT NOT NULL,
        shard_file TEXT NOT NULL
    );
    CREATE TABLE IF NOT EXISTS lexicon_scope (
        lexicon_id TEXT NOT NULL,
        textgroup TEXT NOT NULL,
        PRIMARY KEY (lexicon_id, textgroup)
    );
    CREATE TABLE IF NOT EXISTS lexicon_entries (
        lexicon_id TEXT NOT NULL,
        entry_id TEXT NOT NULL,
        headword_display TEXT NOT NULL,
        headword_translit TEXT,
        headword_key TEXT NOT NULL,
        sort_key TEXT NOT NULL,
        entry_html TEXT NOT NULL,
        PRIMARY KEY (lexicon_id, entry_id)
    );
    CREATE INDEX IF NOT EXISTS idx_lex_headword ON lexicon_entries(lexicon_id, headword_key);
    CREATE TABLE IF NOT EXISTS lexicon_aliases (
        lexicon_id TEXT NOT NULL,
        alias_key TEXT NOT NULL,
        entry_id TEXT NOT NULL,
        PRIMARY KEY (lexicon_id, alias_key, entry_id)
    );
    CREATE INDEX IF NOT EXISTS idx_lex_alias ON lexicon_aliases(lexicon_id, alias_key);
    CREATE TABLE IF NOT EXISTS lexicon_citations (
        lexicon_id TEXT NOT NULL,
        entry_id TEXT NOT NULL,
        textgroup TEXT,
        work TEXT,
        ref TEXT,
        urn TEXT
    );
    CREATE INDEX IF NOT EXISTS idx_lex_cit_urn ON lexicon_citations(urn);
""")

# Migration guard: CREATE TABLE IF NOT EXISTS is a no-op against a table
# that already exists from an earlier run of this cell (e.g. before Pizzi
# was added) -- it won't retroactively add headword_translit. Add it here
# if missing, so re-running this cell on an older monolith doesn't break.
_existing_cols = [r[1] for r in cur.execute("PRAGMA table_info(lexicon_entries)").fetchall()]
if "headword_translit" not in _existing_cols:
    cur.execute("ALTER TABLE lexicon_entries ADD COLUMN headword_translit TEXT")
    print("  (migrated lexicon_entries: added headword_translit column)")

for lexicon_id, cfg in LEXICON_REGISTRY.items():
    if not Path(cfg["path"]).exists():
        print(f"  ! skipping {lexicon_id}: file not found at {cfg['path']}")
        continue

    print(f"Ingesting lexicon: {lexicon_id} ({cfg['format']})...")
    parser = PARSERS[cfg["format"]]
    entries, aliases, citations = parser(cfg["path"], lexicon_id)

    cur.execute("DELETE FROM lexicon_meta WHERE lexicon_id=?", (lexicon_id,))
    cur.execute("DELETE FROM lexicon_scope WHERE lexicon_id=?", (lexicon_id,))
    cur.execute("DELETE FROM lexicon_entries WHERE lexicon_id=?", (lexicon_id,))
    cur.execute("DELETE FROM lexicon_aliases WHERE lexicon_id=?", (lexicon_id,))
    cur.execute("DELETE FROM lexicon_citations WHERE lexicon_id=?", (lexicon_id,))

    cur.execute(
        "INSERT INTO lexicon_meta (lexicon_id, title, author, citation, entry_kind, shard_file) "
        "VALUES (?, ?, ?, ?, ?, ?)",
        (lexicon_id, cfg["title"], cfg["author"], cfg["citation"], cfg["entry_kind"], cfg["shard_file"]),
    )
    cur.executemany(
        "INSERT INTO lexicon_scope (lexicon_id, textgroup) VALUES (?, ?)",
        [(lexicon_id, tg) for tg in cfg["textgroups"]],
    )
    cur.executemany(
        "INSERT INTO lexicon_entries (lexicon_id, entry_id, headword_display, headword_translit, headword_key, sort_key, entry_html) "
        "VALUES (?, ?, ?, ?, ?, ?, ?)",
        [(lexicon_id, e["entry_id"], e["headword_display"], e.get("headword_translit"),
          e["headword_key"], e["sort_key"], e["entry_html"])
         for e in entries],
    )
    cur.executemany(
        "INSERT OR IGNORE INTO lexicon_aliases (lexicon_id, alias_key, entry_id) VALUES (?, ?, ?)",
        [(lexicon_id, a["alias_key"], a["entry_id"]) for a in aliases],
    )
    cur.executemany(
        "INSERT INTO lexicon_citations (lexicon_id, entry_id, textgroup, work, ref, urn) VALUES (?, ?, ?, ?, ?, ?)",
        [(lexicon_id, c["entry_id"], c["textgroup"], c["work"], c["ref"], c["urn"]) for c in citations],
    )
    conn.commit()
    print(f"  \u2713 {len(entries)} entries, {len(aliases)} aliases, {len(citations)} citations "
          f"\u2192 shard {cfg['shard_file']}")

conn.close()
print("\nDone.")


Ingesting lexicon: cunliffe-words (cunliffe)...
  ✓ 9824 entries, 2401 aliases, 92929 citations → shard lexica_homer.db
Ingesting lexicon: cunliffe-names (cunliffe)...
  ✓ 1592 entries, 0 aliases, 0 citations → shard lexica_homer.db
Ingesting lexicon: dindorf-aeschylus (tei_dict)...
  ✓ 7488 entries, 0 aliases, 1646 citations → shard lexica_aeschylus.db
Ingesting lexicon: pizzi-persian (pizzi)...
  ✓ 2380 entries, 25 aliases, 0 citations → shard lexica_persian.db
Ingesting lexicon: betant-thucydides (betant)...
  ✓ 4864 entries, 0 aliases, 63599 citations → shard lexica_thucydides.db

Done.


In [7]:
# ── CELL 2c: Shard lexica -> site/data/lexica/*.db + lexica.json ──────────
#
# Unlike Cell 2 (per-work, book-aware splitting), lexicon shards are small
# (Cunliffe ~14MB, Dindorf ~7.6MB uncompressed source -- nowhere near
# GitHub's 100MB cap) and are grouped by `shard_file`, not by size or by
# work. Everything needed to build a shard's contents lives in the
# monolith's lexicon_* tables (via lexicon_meta.shard_file), so this cell
# doesn't need Cell 1c's LEXICON_REGISTRY still in memory -- same
# self-contained-cell convention as the rest of this notebook.
#
# Run this AFTER Cell 1c and independently of Cell 2/2b (order between
# the two doesn't matter -- they touch disjoint tables/files).

import sqlite3
import json
from pathlib import Path

BUILD_DIR = Path("/tmp/persvers_build")                          # must match Cell 1c
MONOLITH_DB = BUILD_DIR / "corpus_alignment_grid.db"
SITE_ROOT = Path("/Users/gcrane/github/persverscomp/site")       # must match Cell 2
LEXICA_DIR = SITE_ROOT / "data" / "lexica"

LEXICON_TABLES = ["lexicon_meta", "lexicon_scope", "lexicon_entries",
                  "lexicon_aliases", "lexicon_citations"]


def shard_lexica(monolith_path, lexica_dir):
    monolith_path = Path(monolith_path)
    lexica_dir = Path(lexica_dir)
    lexica_dir.mkdir(parents=True, exist_ok=True)

    src = sqlite3.connect(str(monolith_path))
    src.row_factory = sqlite3.Row

    shard_files = [r[0] for r in src.execute(
        "SELECT DISTINCT shard_file FROM lexicon_meta ORDER BY shard_file")]

    if not shard_files:
        print("  ! no rows in lexicon_meta -- run Cell 1c first")
        src.close()
        return

    manifest = {"lexica": {}, "textgroups": {}}

    for shard_file in shard_files:
        lexicon_ids = [r[0] for r in src.execute(
            "SELECT lexicon_id FROM lexicon_meta WHERE shard_file=?", (shard_file,))]
        print(f"[lexica-shard] {shard_file}  <-  {lexicon_ids}")

        shard_path = lexica_dir / shard_file
        shard_path.unlink(missing_ok=True)
        dst = sqlite3.connect(str(shard_path))
        dst.execute("PRAGMA synchronous=OFF")

        for table in LEXICON_TABLES:
            schema = src.execute(
                "SELECT sql FROM sqlite_master WHERE type='table' AND name=?",
                (table,)).fetchone()
            if schema and schema[0]:
                dst.execute(schema[0])
        dst.commit()

        placeholders = ", ".join("?" for _ in lexicon_ids)
        for table in LEXICON_TABLES:
            col_names = [c[1] for c in src.execute(f"PRAGMA table_info({table})").fetchall()]
            rows = src.execute(
                f"SELECT * FROM {table} WHERE lexicon_id IN ({placeholders})",
                lexicon_ids).fetchall()
            if not rows:
                continue
            col_list = ", ".join(col_names)
            qmarks = ", ".join("?" for _ in col_names)
            dst.executemany(f"INSERT INTO {table} ({col_list}) VALUES ({qmarks})", rows)
            print(f"    {table}: {len(rows)}")
        dst.commit()
        dst.close()

        sz = shard_path.stat().st_size
        print(f"  \u2713 {shard_file}: {sz/1e6:.2f}MB")

        for lexicon_id in lexicon_ids:
            meta_row = src.execute(
                "SELECT title, author, citation, entry_kind FROM lexicon_meta WHERE lexicon_id=?",
                (lexicon_id,)).fetchone()
            textgroups = [r[0] for r in src.execute(
                "SELECT textgroup FROM lexicon_scope WHERE lexicon_id=?", (lexicon_id,))]
            manifest["lexica"][lexicon_id] = {
                "title": meta_row[0], "author": meta_row[1],
                "citation": meta_row[2], "entry_kind": meta_row[3],
                "shard": shard_file,
            }
            for tg in textgroups:
                manifest["textgroups"].setdefault(tg, [])
                if lexicon_id not in manifest["textgroups"][tg]:
                    manifest["textgroups"][tg].append(lexicon_id)

    src.close()
    (SITE_ROOT / "lexica.json").write_text(
        json.dumps(manifest, ensure_ascii=False, indent=1), encoding="utf-8")
    print(f"\n\u2713 wrote {SITE_ROOT / 'lexica.json'}")


if not MONOLITH_DB.exists():
    print(f"  ! monolith not found at {MONOLITH_DB} -- run Cell 1c first")
else:
    shard_lexica(MONOLITH_DB, LEXICA_DIR)


[lexica-shard] lexica_aeschylus.db  <-  ['dindorf-aeschylus']
    lexicon_meta: 1
    lexicon_scope: 1
    lexicon_entries: 7488
    lexicon_citations: 1646
  ✓ lexica_aeschylus.db: 11.00MB
[lexica-shard] lexica_homer.db  <-  ['cunliffe-words', 'cunliffe-names']
    lexicon_meta: 2
    lexicon_scope: 2
    lexicon_entries: 11416
    lexicon_aliases: 2382
    lexicon_citations: 92929
  ✓ lexica_homer.db: 26.98MB
[lexica-shard] lexica_persian.db  <-  ['pizzi-persian']
    lexicon_meta: 1
    lexicon_scope: 1
    lexicon_entries: 2380
    lexicon_aliases: 25
  ✓ lexica_persian.db: 1.48MB
[lexica-shard] lexica_thucydides.db  <-  ['betant-thucydides']
    lexicon_meta: 1
    lexicon_scope: 1
    lexicon_entries: 4864
    lexicon_citations: 63599
  ✓ lexica_thucydides.db: 12.32MB

✓ wrote /Users/gcrane/github/persverscomp/site/lexica.json


In [8]:
# ── CELL 2b: Treebank chunking guard ──────────────────────────────────────────
# Verifies, AFTER sharding, that the distinct treebank chapters present in each
# deployed shard match what the source .conllu actually contains (and the build
# monolith in between). Catches stale shards, wrong-path source files, and the
# silent chapter='1' fallback before they ship. Raises AssertionError on any
# mismatch so a bad build halts loudly instead of deploying.

import sqlite3, re
from pathlib import Path
from collections import Counter

# --- config you can edit -----------------------------------------------------
SITE_ROOT   = Path("/Users/gcrane/github/persverscomp/site")        # must match Cell 2
MONOLITH_DB = Path("/tmp/persvers_build/corpus_alignment_grid.db")  # must match Cell 1/2
# Known-good distinct chapter counts. When a version appears here, its count is
# hard-asserted — this is what also catches a source file that is itself stale
# (e.g. an old chapter-1-only export silently overwriting the path).
EXPECTED_TB_CHAPTERS = {
    "kassel-tb-grc1": 26,        # Aristotle Poetics, Kassel
    # "bishrmatta-tb-ara1": 26,  # fill in once verified
    # "daphne_perstb-grc1": 24,  # Iliad books 1-24, etc.
}
# -----------------------------------------------------------------------------

def _natkey(c):
    m = re.match(r"(\d+)", str(c) or "")
    return (int(m.group(1)) if m else 10**9, str(c))

def _chset_from_conllu(cfg, tg, wk, work_meta):
    """Expected chapter set = what parse_conllu_treebank would emit from the
    source path(s), using the same card-interval logic the build uses."""
    card_intervals = None
    if any(c.get("parse_mode") == "poetry_cards"
           for c in work_meta.get("editions", {}).values()):
        try:
            ci = build_poetry_canonical_intervals(work_meta["editions"])
            card_intervals = [iv for bk in ci.values() for iv in bk]
        except Exception as e:
            print(f"      ⚠ card-interval build failed ({e}); using None")
    sents, _doc_credits = parse_conllu_treebank(cfg["path"], "?", tg, wk,
                                                 card_intervals=card_intervals)
    return Counter(str(s["chapter"]) for s in sents if s.get("subdoc"))

def _chset_from_db(db_path, tg, wk, vid):
    if not Path(db_path).exists():
        return None
    con = sqlite3.connect(str(db_path))
    try:
        rows = con.execute(
            "SELECT chapter, COUNT(*) FROM treebank_sentences "
            "WHERE textgroup=? AND work=? AND version_short_id=? GROUP BY chapter",
            (tg, wk, vid)).fetchall()
    except sqlite3.OperationalError:
        rows = []
    finally:
        con.close()
    return Counter({str(c): n for c, n in rows})


def _chset_from_all_parts(site_root, tg, wk, vid):
    """A work may now be split into several book-range part files (see
    Cell 2's book-aware sharder); this unions the chapter counts across
    every part*.db so the guard still checks the WHOLE work, not just
    whichever part happens to sort first."""
    part_files = sorted((site_root / "data" / tg / wk).glob(f"{tg}.{wk}.part*.db"))
    if not part_files:
        return None, []
    total = Counter()
    found_any = False
    for pf in part_files:
        c = _chset_from_db(pf, tg, wk, vid)
        if c:
            found_any = True
            total.update(c)
    return (total if found_any else None), part_files

failures = []
print("── Treebank chunking guard ───────────────────────────────────────────")
for work_key, work_meta in WORK_REGISTRY.items():
    tbs = {v: c for v, c in work_meta.get("treebanks", {}).items()
           if c.get("parse_mode") == "conllu"}
    if not tbs:
        continue
    tg, wk = work_meta["textgroup"], work_meta["work"]
    print(f"\n[{work_key}]")

    for vid, cfg in tbs.items():
        src_set  = set(_chset_from_conllu(cfg, tg, wk, work_meta))
        mono_set = set(_chset_from_db(MONOLITH_DB, tg, wk, vid) or Counter())
        shd_ct, part_files = _chset_from_all_parts(SITE_ROOT, tg, wk, vid)
        shd_set = set(shd_ct) if shd_ct is not None else None
        part_names = ", ".join(p.name for p in part_files) if part_files else "none found"

        n_shd = "n/a" if shd_set is None else len(shd_set)
        print(f"  {vid:<22} source={len(src_set):>3} chapters  "
              f"monolith={len(mono_set):>3}  shard={n_shd}  (parts: {part_names})")

        if shd_set is None:
            failures.append(f"{work_key}/{vid}: no shard parts found, or no treebank_sentences rows across them")
            print("      ✗ no shard parts have rows for this version")
            continue

        miss_mono = src_set - mono_set
        if miss_mono:
            failures.append(f"{work_key}/{vid}: monolith missing chapters {sorted(miss_mono, key=_natkey)}")
            print(f"      ✗ in source but NOT in monolith: {sorted(miss_mono, key=_natkey)}  → re-run Cell 1 (ingest)")

        miss_shd = mono_set - shd_set
        if miss_shd:
            failures.append(f"{work_key}/{vid}: shard missing chapters {sorted(miss_shd, key=_natkey)}")
            print(f"      ✗ in monolith but NOT in shard: {sorted(miss_shd, key=_natkey)}  → re-run Cell 2 (shard) + redeploy")

        extra_shd = shd_set - src_set
        if extra_shd:
            failures.append(f"{work_key}/{vid}: shard has unexpected chapters {sorted(extra_shd, key=_natkey)}")
            print(f"      ✗ in shard but NOT in source (stale data?): {sorted(extra_shd, key=_natkey)}")

        if len(src_set) > 1 and shd_set == {"1"}:
            print("      ✗ ALL shard sentences are chapter '1' — classic silent fallback / stale source")

        exp = EXPECTED_TB_CHAPTERS.get(vid)
        if exp is not None and len(shd_set) != exp:
            failures.append(f"{work_key}/{vid}: expected {exp} chapters in shard, got {len(shd_set)}")
            print(f"      ✗ EXPECTED {exp} chapters, shard has {len(shd_set)}")

        if not (miss_mono or miss_shd or extra_shd) and (exp is None or len(shd_set) == exp):
            ordered = sorted(shd_set, key=_natkey)
            print(f"      ✓ source ↔ monolith ↔ shard agree ({ordered[:3]}…{ordered[-1:]})")

print("\n──────────────────────────────────────────────────────────────────────")
if failures:
    print(f"✗ {len(failures)} treebank chunking problem(s):")
    for f in failures:
        print("   •", f)
    raise AssertionError(f"Treebank chunking guard failed ({len(failures)} issue(s)) — do not deploy.")
print("✓ All treebank chapters consistent across source, monolith, and shards.")

── Treebank chunking guard ───────────────────────────────────────────

[tlg0012.tlg001]
  daphne_perstb-grc1     source=416 chapters  monolith=416  shard=416  (parts: tlg0012.tlg001.part1.db, tlg0012.tlg001.part2.db)
      ✓ source ↔ monolith ↔ shard agree (['1-21', '1-26', '1-30']…['899-909'])

[tlg0012.tlg002]
  daphne_perstb-grc1     source=276 chapters  monolith=276  shard=276  (parts: tlg0012.tlg002.part1.db)
      ✓ source ↔ monolith ↔ shard agree (['1-34', '1-35', '1-36']…['795-847'])

[tlg0086.tlg034]
  kassel-tb-grc1         source= 26 chapters  monolith= 26  shard=n/a  (parts: tlg0086.tlg034.part1.db)
      ✗ no shard parts have rows for this version
  bishrmatta-tb-ara1     source=  1 chapters  monolith=  1  shard=n/a  (parts: tlg0086.tlg034.part1.db)
      ✗ no shard parts have rows for this version

[tlg0011.tlg001]
  daphne-tb-grc1         source= 70 chapters  monolith= 70  shard=64  (parts: tlg0011.tlg001.part1.db)
      ✗ in monolith but NOT in shard: ['102', '506', '5

AssertionError: Treebank chunking guard failed (13 issue(s)) — do not deploy.

In [ ]:
# NOTE: this also removes the lexicon_meta/lexicon_scope/lexicon_entries/
# lexicon_aliases/lexicon_citations tables added by Cell 1c, since they live
# in this same monolith file. That's fine -- Cell 2c has already exported
# them to site/data/lexica/*.db + site/lexica.json by this point, which is
# self-contained same as every other shard. Re-run Cell 1c + Cell 2c (in
# that order, before this cell) if you need the monolith's lexicon tables
# again. Make sure Cell 2b and Cell 1c/2c above have ALL run before this
# cell -- they need the monolith too, and running this first will wipe it
# out from under them (this bit someone already once; don't repeat it).
# ── Cleanup: drop the temp build monolith when you are done ───────────────
# Safe to run any time AFTER Cell 2 (shards written) and Cell 3 (index.html
# built). The deployed site/ (shards + catalog.json + index.html) is fully
# self-contained and does NOT need the monolith. Re-run Cell 1 to rebuild it.
from pathlib import Path
import shutil
BUILD_DIR = Path("/tmp/persvers_build")
if BUILD_DIR.exists():
    n = sum(1 for _ in BUILD_DIR.rglob("*"))
    shutil.rmtree(BUILD_DIR)
    print(f"Removed {BUILD_DIR} ({n} item(s)).")
else:
    print(f"Nothing to clean - {BUILD_DIR} does not exist.")